<a href="https://colab.research.google.com/github/johnnyxr96/Competitor-Regression-Dashboard/blob/main/V1_1_Fleet_Daily_Dashboard.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# D01 · Fleet daily dashboard

The daily dashboard (Daily_Dashboard_2.1.xlsm) on Colab, in two parts with the pricing model run in Excel in between.

**Each morning**
1. **Settings:** type today's swaps (`SWAP_2Y`, `SWAP_5Y`, and `SWAP_1Y` for the 1-year products) and choose the day to run in `RUN_FOR`. With `Ask` (the default) the notebook lists the last five working days with their applications and how many have no balance yet, and asks which one to use. Yesterday's balances fill in around midday, so in the morning pick the day before.
2. **Part 1:** run the cells down to the end of Part 1. It pulls DIPs, applications and completions from BigQuery and downloads **`LLPM_Inputs_<date>.csv`**: one row per pricing code in the LLPM's input layout (the old Prodinput and ProdInput - PricingSwaps tabs, in that order).
3. **In Excel:** paste the CSV rows into the LLPM's input data, put today's swaps into the manual swap curve, and run the model as normal. Save the workbook.
4. **Part 2:** run the rest. Upload the LLPM workbook (or its Outputs tab saved as CSV) when asked. It builds **`Fleet_Daily_Dashboard_<date>.html`** and updates the **history file**.

**Sharing the dashboard (`SHARE_FOLDER`).** Set it to a Cloud Storage folder your team can use, e.g. `'gs://<bucket>/fleet-daily-dashboard'`. Each run then saves there:
- `Fleet_Daily_Dashboard_latest.html`, a link that stays the same every day and opens in the browser for anyone with read access to the folder (`https://storage.cloud.google.com/<bucket>/fleet-daily-dashboard/Fleet_Daily_Dashboard_latest.html`), plus a dated copy;
- the history, current pricing and product code files, so they are picked up automatically next time (no uploads apart from the model results);
- the day's LLPM input CSV and outputs workbook, which you can also download from the bucket in the Cloud console if Colab's downloads are blocked.

Without a shared folder the files are only in this Colab session: download them from the file browser on the left.

If the Colab session times out between the parts, re-run Settings and Setup (choose the same day as in Part 1), then go straight to Part 2: it pulls the tapes again.

**Input files** (upload buttons; Cancel = use the copy already in `INPUT_DIR`)
- `Fleet_Current_Pricing.xlsx`: the Current Pricing tab (Part 1). A Daily Dashboard workbook works too.
- The LLPM workbook after it has run, or its Outputs tab as CSV (Part 2).
- `Fleet_Daily_Dashboard_History.xlsx`: the running history (Part 2). It replaces the Month of LLPM Outputs, Historic Return Data and Swap Rates tabs. To start it, upload the starter file, or a Daily Dashboard workbook (its history tabs are read).
- `Fleet_Product_Codes.csv` (optional): product code → pricing code, as in FleetProductMaster. New product codes not in it are coded from their description and listed.

**Data handling:** the tapes are loan-level and stay in memory; nothing loan-level is displayed or saved. The CSV and every other output hold totals and averages per pricing code.

## Settings

| Setting | What it does |
|---|---|
| `RUN_FOR` | Which application date to report. `Ask` = show the last five working days (applications, £, how many still have no balance) and ask; `Previous working day`; `2 working days ago`; `Date below` = the date in `REPORTING_DATE`. |
| `REPORTING_DATE` | `'YYYY-MM-DD'`, used when `RUN_FOR = 'Date below'`. |
| `SWAP_1Y`, `SWAP_2Y`, `SWAP_5Y` | Today's swap rates in %, e.g. `4.74` (what the Refresh DD macro asked for, plus 1-year). 2- and 5-year are needed; if blank the notebook asks. |
| `BBR` | Bank Rate in % (the Control Panel's C5): used to turn tracker rates into margins in the input file. |
| `MATCH_EXCEL` | `True` keeps the Excel workbook's own definitions where they differ from the labels (see the Checks & notes tab of the dashboard). |
| `TARGETS` | Completions target for the year, application-to-completion conversion, daily volume target, RoE target, working days in the year. |
| `BACK_DAYS` | Earlier working days the dashboard can show, rebuilt from the history. |
| `SHARE_FOLDER` | Cloud Storage folder for the shared dashboard link and the files kept between runs, e.g. `'gs://<bucket>/fleet-daily-dashboard'`. Blank = files stay in this session only. |
| `INPUT_DIR`, `UPLOAD` | Folder for the input files. `UPLOAD = 'ask'` shows an upload button for each; `'auto'` uses the copies in `INPUT_DIR`. |
| `DOWNLOAD` | `'buttons'` (default) = download buttons under the cell: click to save the file. `'colab'` = Colab's own download (can stall in Colab Enterprise). `''` = none; the files are also in the file browser on the left (right-click → Download). |
| `BILLING_PROJECT` | As in the other notebooks. |

In [ ]:
import os
import datetime
import importlib
import numpy as np
import pandas as pd
from IPython.display import display, HTML

os.environ["GOOGLE_APPLICATION_CREDENTIALS"] = "/home/jupyter/.config/gcloud/application_default_credentials.json"
pd.set_option('display.width', 220)
pd.set_option('display.max_columns', 30)

RUN_FOR = 'Ask'       # @param ["Ask", "Previous working day", "2 working days ago", "Date below"]
REPORTING_DATE = ''   # @param {type:"string"}   'YYYY-MM-DD', used when RUN_FOR = 'Date below'
SWAP_1Y = ''          # e.g. 4.53
SWAP_2Y = ''          # e.g. 4.74
SWAP_5Y = ''          # e.g. 4.80
BBR = 3.75
MATCH_EXCEL = True
TARGETS = {'completions_fy': 1.0e9, 'conversion': 0.5, 'daily_volume': 8.0e6, 'roe': 0.25, 'fy_working_days': 253}
BACK_DAYS = 30
SHARE_FOLDER = ''      # @param {type:"string"}   e.g. 'gs://<bucket>/fleet-daily-dashboard'
INPUT_DIR = ''
UPLOAD = 'ask'
BILLING_PROJECT = ''
DOWNLOAD = 'buttons'   # @param ["buttons", "colab", ""]

## Setup: shared SQL and dashboard code

In [ ]:
# @title
%%writefile fleet_sql.py
"""fleet_sql: the shared BigQuery definitions every Fleet notebook uses (build F1).

One place for: which loans are Fleet, the latest application record and status, the origination record, the product,
rate, margin, fees and ERCs on the first daily position on or after the application and the completion date, and the
pricing-code label. Notebooks build their queries from these pieces, so every report counts the same cases the same way.

Queries run through run(): STRING parameters (@start, @asat ...) as in the earlier notebooks, results as a DataFrame.
In a notebook, run() sends each query through the %%bigquery cell magic, exactly as the earlier notebooks do, so it signs in
the same way (runtime credentials if there are any, otherwise the Google sign-in link) and runs as the same account.
Nothing here returns loan-level rows on its own; each notebook decides what it aggregates.
"""
import contextlib
import io
import json
import os
import re
import time
import numpy as np
import pandas as pd

VERSION = '1.4'
BILLING = 'ds-notebooks-prod-vhpr'           # project the jobs run (and are billed) in
DATA = 'warehouse-analytics-prod-ommc'       # project the warehouse tables live in
RUNNER = None                                # tests can set a function(sql, params, label) -> DataFrame
CREDENTIALS = None                           # optional explicit credentials; None = the same sign-in as the %%bigquery cells
ACCOUNT = None                               # the account sign_in() found
INPUT_DIR = ''                               # folder holding the input files (CoF swaps, pricing inputs, competitor files); '' = this folder
UPLOAD = 'ask'                               # 'ask' = upload button for each outside file every run; 'auto' = use the file already in INPUT_DIR, ask only if there is none
LAST_SQL = {}                                # label -> SQL of the last run, for printing or pasting into BigQuery

T = {
    'mortgage': 'dimension.credit_agreement_mortgage',
    'status': 'dimension.credit_agreement_mortgage_status_details',
    'details': 'dimension.credit_agreement_mortgage_details',
    'product': 'dimension.mortgage_product_details',
    'application': 'fact.credit_agreement_mortgage_application',
    'origination': 'fact.credit_agreement_mortgage_origination',
    'position': 'fact.credit_agreement_mortgage_daily_position',
    'valuation': 'fact.credit_agreement_mortgage_valuation',
}


def t(name):
    """Full table name in backticks, e.g. t('position')."""
    return '`%s.%s`' % (DATA, T[name])


def clean_date(expr):
    """Placeholder dates (0001-01-01, 1800-..., 9999-12-31) become NULL."""
    return "IF(%s BETWEEN DATE '1990-01-01' AND DATE '2100-12-31', %s, NULL)" % (expr, expr)


# columns each shared query reads, per table (used by check_access)
USED = {
    'mortgage': ['key', 'source_key', 'origination_date', 'redemption_date', 'borrower_type', 'property_type', 'tenure', 'repayment_method', 'purpose',
                 'uk_region', 'property_postcode_outward_code', 'is_house_in_multiple_occupancy_property', 'is_multiple_unit_block_property',
                 'year_of_birth_primary_borrower', 'employment_status_primary_borrower', 'is_first_time_buyer_primary_borrower', 'year_built'],
    'status': ['key', 'application_status', 'account_status'],
    'details': ['key', 'current_interest_rate_type', 'current_epc_rating'],
    'product': ['key', 'product_code', 'product_description', 'product_description_term_length', 'product_description_rate_type',
                'product_fee_type', 'product_fee_amount', 'is_product_switch', 'is_loyalty_product'],
    'application': ['credit_agreement_mortgage_key', 'application_date', 'decision_in_principle_date', 'offer_date', 'declined_date', 'cancelled_date',
                    'expected_completion_date', 'credit_agreement_mortgage_status_details_key'],
    'origination': ['credit_agreement_mortgage_key', 'date', 'origination_balance', 'original_annual_interest_rate', 'original_term_months',
                    'number_of_buy_to_let_properties', 'servicer_provided_origination_ltv', 'original_servicer_provided_valuation',
                    'completion_fees_added_to_loan_balance_by_borrower', 'cashback', 'original_stressed_interest_coverage_ratio',
                    'gross_annual_rental_income', 'income_primary_borrower', 'original_bureau_score_primary_borrower'],
    'position': ['credit_agreement_mortgage_key', 'date', 'current_annual_interest_rate', 'current_interest_margin_rate', 'reversion_margin_rate',
                 'first_early_repayment_charge', 'second_early_repayment_charge', 'third_early_repayment_charge', 'fourth_early_repayment_charge',
                 'fifth_early_repayment_charge', 'current_principal_balance', 'current_total_balance', 'reversion_date', 'months_in_arrears',
                 'arrears_balance', 'servicer_provided_current_ltv', 'current_debt_service_coverage_ratio', 'mortgage_product_details_key',
                 'credit_agreement_mortgage_status_details_key', 'credit_agreement_mortgage_details_key'],
    'valuation': ['credit_agreement_mortgage_key', 'date'],
}


MODE = 'auto'        # 'auto' = through %%bigquery in a notebook, else the Python client; 'magic' or 'client' to force one


def _magic():
    """The IPython shell with the %%bigquery cell magic loaded, or None outside a notebook."""
    try:
        from IPython import get_ipython
        ip = get_ipython()
    except Exception:
        return None
    if ip is None:
        return None
    cells = ip.magics_manager.magics.get('cell', {})
    if 'bigquery' not in cells:
        for ext in ('bigquery_magics', 'google.cloud.bigquery'):
            try:
                ip.run_line_magic('load_ext', ext)
                break
            except Exception:
                pass
    return ip if 'bigquery' in ip.magics_manager.magics.get('cell', {}) else None


def _via_magic(ip, sql, params=None, dry_run=False, project=None):
    """Run one query through %%bigquery. Returns (result, error text). result is a DataFrame (or a QueryJob for a dry run)."""
    var = '_fs_result'
    ip.user_ns.pop(var, None)
    line = '%s --project %s --verbose' % (var, project or BILLING)
    if dry_run:
        line += ' --dry_run'
    if params:
        line += ' --params ' + json.dumps(params)
    err = io.StringIO()
    with contextlib.redirect_stderr(err):
        ip.run_cell_magic('bigquery', line, sql)
    return ip.user_ns.pop(var, None), err.getvalue()


def _magic_context():
    for mod in ('bigquery_magics', 'google.cloud.bigquery.magics'):
        try:
            return __import__(mod, fromlist=['context']).context
        except Exception:
            pass
    return None


def client():
    """BigQuery Python client on the same credentials as the %%bigquery cells (signing in the same way if needed)."""
    from google.cloud import bigquery
    if CREDENTIALS is not None:
        return bigquery.Client(project=BILLING, credentials=CREDENTIALS)
    ctx = _magic_context()
    if ctx is not None:
        return bigquery.Client(project=BILLING, credentials=ctx.credentials)
    return bigquery.Client(project=BILLING)


def _email(creds):
    sa = getattr(creds, 'service_account_email', None)
    if sa and sa != 'default':
        return sa
    try:
        import requests
        r = requests.get('https://oauth2.googleapis.com/tokeninfo', params={'access_token': creds.token}, timeout=10).json()
        return r.get('email')
    except Exception:
        return None


def sign_in(new_login=False):
    """Sign in exactly as the %%bigquery cells do and say which account it is.
    new_login=True forgets the sign-in remembered on this runtime (the pydata credentials file) and asks again:
    use it if the account shown is wrong or queries are refused in a session that has been signed in before."""
    global ACCOUNT
    ctx = _magic_context()
    if new_login:
        path = os.path.join(os.path.expanduser('~'), '.config', 'pydata', 'pydata_google_credentials.json')
        if os.path.exists(path):
            os.remove(path)
            print('Forgot the sign-in remembered on this runtime (%s).' % path)
        if ctx is not None:
            ctx._credentials = None
    p = os.environ.get('GOOGLE_APPLICATION_CREDENTIALS')
    if p and not os.path.exists(p):
        print('Note: GOOGLE_APPLICATION_CREDENTIALS points to %s, which is not on this runtime, so you will be asked to sign in.' % p)
    if ctx is None:
        print('The %%bigquery magic is not available here.')
        return None
    from google.auth.transport.requests import Request
    creds = ctx.credentials
    try:
        creds.refresh(Request())
    except Exception:
        pass
    ACCOUNT = _email(creds)
    if not ACCOUNT:
        try:
            ACCOUNT = whoami()
        except Exception:
            ACCOUNT = None
    print('Signed in as:', ACCOUNT or '(could not read the account name)')
    return ACCOUNT


def explain_error(e):
    """Turn a BigQuery access error into plain words: which identity, which table, which columns, what to ask for."""
    msg = str(e)
    if 'policy tag' in msg or 'Access Denied' in msg or '403' in msg:
        cols = sorted(set(re.findall(r'%s\.([a-z_]+\.[a-z_]+)\.([a-z_0-9]+)' % re.escape(DATA), msg)))
        tabs = sorted(set(t_ for t_, _ in cols))
        tag = re.search(r'policy tag "([^"]+)"', msg)
        print('\nBigQuery refused access (403). This is a permission, not a fault in the notebook.')
        print('  Running as:', ACCOUNT or '(unknown: run fs.sign_in() to see)')
        if 'jobs.create' in msg:
            print('  This account cannot run queries in %s at all.' % BILLING)
            print('  Try a fresh sign-in first: Runtime > Restart session (or run fs.sign_in(new_login=True)), run all again and sign in with your Starling account.')
            print('  If a notebook that works for you (e.g. the cohort pack) fails in the same session, ask the data platform team for BigQuery Job User on %s.' % BILLING)
            return True
        if tag: print('  Protected by policy tag:', tag.group(1))
        for t_ in tabs:
            print('  Table %s, columns: %s' % (t_, ', '.join(c for tt, c in cols if tt == t_)))
        print('  Fix: run the access check notebook (Z00) to see every blocked table, then ask the data platform team for '
              'Fine-Grained Reader on that policy tag for this account, or confirm which account the notebook should run as.')
        return True
    return False


def whoami():
    """The account BigQuery sees for this notebook (runs SELECT SESSION_USER())."""
    return run('SELECT SESSION_USER() AS u', label='who am I')['u'].iloc[0]


def check_access(tables=None):
    """Dry-run a query on the columns each shared query reads, table by table, the same way the %%bigquery cells run.
    Costs nothing. Returns a DataFrame."""
    rows, ip = [], (_magic() if MODE in ('auto', 'magic') else None)
    for name in (tables or USED):
        sql = 'SELECT %s FROM %s LIMIT 1' % (', '.join(USED[name]), t(name))
        if ip is not None:
            job, m = _via_magic(ip, sql, dry_run=True)
            ok = job is not None and not m.strip()
            gb = (getattr(job, 'total_bytes_processed', 0) or 0) / 1e9 if ok else 0
        else:
            from google.cloud import bigquery
            try:
                job = client().query(sql, job_config=bigquery.QueryJobConfig(dry_run=True, use_query_cache=False)); ok, m, gb = True, '', (job.total_bytes_processed or 0) / 1e9
            except Exception as e:
                ok, m = False, str(e)
        if ok:
            rows.append({'table': T[name], 'access': 'OK', 'blocked columns': '', 'detail': '%.1f GB if read in full' % gb})
        else:
            cols = sorted(set(c for _, c in re.findall(r'%s\.([a-z_]+\.[a-z_]+)\.([a-z_0-9]+)' % re.escape(DATA), m)))
            tag = re.search(r'policy tag "([^"]+)"', m)
            kind = 'NO QUERY RIGHTS' if 'jobs.create' in m else ('BLOCKED' if ('Access Denied' in m or '403' in m) else 'ERROR')
            rows.append({'table': T[name], 'access': kind, 'blocked columns': ', '.join(cols),
                         'detail': tag.group(1) if tag else ('cannot run queries in %s' % BILLING if kind == 'NO QUERY RIGHTS' else m.strip()[:160])})
    return pd.DataFrame(rows)


def run(sql, params=None, label='query', show_sql=False):
    """Run a query in BigQuery and return a DataFrame. params are passed as STRING query parameters."""
    params = {k: ('' if v is None else str(v)) for k, v in (params or {}).items()}
    LAST_SQL[label] = sql
    if show_sql:
        print('-- %s\n%s\n' % (label, sql))
    t0 = time.time()
    if RUNNER is not None:
        df = RUNNER(sql, params, label)
        print('%-28s %8s rows  %5.1fs  (test runner)' % (label, '{:,}'.format(len(df)), time.time() - t0), flush=True)
        return df
    ip = _magic() if MODE in ('auto', 'magic') else None
    if ip is not None:
        print('%-28s running ...' % label, flush=True)
        df, err = _via_magic(ip, sql, params)
        if not isinstance(df, pd.DataFrame):
            if not explain_error(Exception(err)):
                print(err.strip()[:1500])
            raise RuntimeError('BigQuery query "%s" failed (see above).' % label) from None
        print('%-28s %8s rows  %5.0fs' % (label, '{:,}'.format(len(df)), time.time() - t0), flush=True)
        return df
    from google.cloud import bigquery
    cl = client()
    cfg = bigquery.QueryJobConfig(query_parameters=[bigquery.ScalarQueryParameter(k, 'STRING', v) for k, v in params.items()])
    try:
        job = cl.query(sql, job_config=cfg)
        res = job.result()
    except Exception as e:
        if explain_error(e):
            raise RuntimeError('BigQuery access denied for "%s" (see above).' % label) from None
        raise
    df = res.to_dataframe(create_bqstorage_client=False)
    print('%-28s %8s rows  %5.0fs  %6.1f GB scanned' % (label, '{:,}'.format(len(df)), time.time() - t0, (job.total_bytes_processed or 0) / 1e9), flush=True)
    return df


# ------------------------------------------------------------------------------------------------ pricing-code label
def pricing_code_cols(p):
    """SQL columns that label a product the same way as the other notebooks: product line (BTL / LTD / HMO), term,
    LTV, then TRC tracker, ZF no fee, FF fixed £ fee, G green. p = prefix of the product columns (e.g. 'a_')."""
    d = 'UPPER(TRIM(%sdesc))' % p
    return ",\n".join([
        "REGEXP_EXTRACT(%s, r'^(BTL|LTD|HMO)') AS %sline" % (d, p),
        "CASE %sterm_raw WHEN '2_YEAR' THEN '2YR' WHEN '3_YEAR' THEN '3YR' WHEN '5_YEAR' THEN '5YR' WHEN '7_YEAR' THEN '7YR' "
        "WHEN '10_YEAR' THEN '10YR' WHEN 'LIFETIME' THEN 'LIFE' ELSE NULL END AS %sterm" % (p, p),
        "REGEXP_EXTRACT(%s, r'(?:^|\\s)(\\d\\d)%%') AS %sltv_band" % (d, p),
        "(%srate_type = 'TRACKER' OR REGEXP_CONTAINS(%s, r'\\b(TRK|TRC|TRACKER)\\b')) AS %stracker" % (p, d, p),
        "(REGEXP_CONTAINS(%s, r'\\sG$') OR ENDS_WITH(UPPER(%scode), 'G')) AS %sgreen" % (d, p, p),
    ])


def pricing_code_expr(p):
    return ("IF(%sline IS NULL, 'UNKNOWN', CONCAT(%sline, IFNULL(%sterm, 'UNK'), IFNULL(%sltv_band, ''), IF(%stracker, 'TRC', ''), "
            "CASE %sfee_type WHEN 'NO_FEE' THEN 'ZF' WHEN 'FIXED' THEN 'FF' ELSE '' END, IF(%sgreen, 'G', '')))") % ((p,) * 7)


# ------------------------------------------------------------------------------------------------ the standard case set
def cases_sql(open_years=3, snap_days=14, extra_orig='', extra_dim='', include_dip=False):
    """WITH ... cases AS (...): one row per Fleet case that applied or completed between @start and @asat, plus cases
    still open (APPLICATION / OFFERED) that applied in the last `open_years` years.

    Columns: k, dates (cleaned), status, borrower and property fields, the origination record, and for the product
    'a_' = first daily position on or after the application date (within snap_days) and 'c_' = the same for the
    completion date: rate, margin, reversion margin, ERCs 1-5, principal, product code / description / term / rate type /
    fee type / fee amount / switch / loyalty, and the pricing-code label. n_app_rows = application records for the loan.
    Callers add their own SELECT ... FROM cases. extra_dim / extra_orig add columns from the mortgage / origination
    tables (e.g. ', m.build_type'). include_dip=True also brings in cases whose DIP date falls in the window.
    """
    snap_cols = ('date, rate, margin, rev_margin, erc1, erc2, erc3, erc4, erc5, principal, pk')

    def prod(p, alias):
        return ', '.join('%s.%s AS %s%s' % (alias, src, p, dst) for src, dst in (
            ('product_code', 'code'), ('product_description', 'desc'), ('product_description_term_length', 'term_raw'),
            ('product_description_rate_type', 'rate_type'), ('product_fee_type', 'fee_type'), ('product_fee_amount', 'fee_amount'),
            ('is_product_switch', 'ps'), ('is_loyalty_product', 'loyal')))

    def snapcols(p, alias):
        return ', '.join('%s.%s AS %s%s' % (alias, c.strip(), p, c.strip()) for c in snap_cols.split(',') if c.strip() != 'pk')

    return """WITH params AS (
  SELECT DATE(@start) AS s, DATE(@asat) AS e
),
fleet AS (
  SELECT m.key AS k, %(od)s AS completion_date, %(rd)s AS redemption_date,
         m.borrower_type, m.property_type, m.tenure, m.repayment_method, m.purpose, m.uk_region, m.property_postcode_outward_code AS postcode_area,
         m.is_house_in_multiple_occupancy_property AS is_hmo, m.is_multiple_unit_block_property AS is_mub,
         m.year_of_birth_primary_borrower AS year_of_birth, m.employment_status_primary_borrower AS employment_status,
         m.is_first_time_buyer_primary_borrower AS first_time, m.year_built%(extra_dim)s
  FROM %(mortgage)s m
  WHERE m.source_key LIKE 'FLEET~%%'
),
app_all AS (
  SELECT a.credit_agreement_mortgage_key AS k, %(ad)s AS application_date, %(dipd)s AS dip_date, %(ofd)s AS offer_date, %(dd)s AS declined_date,
         %(cd)s AS cancelled_date, %(ecd)s AS expected_completion_date, UPPER(TRIM(sd.application_status)) AS status,
         COUNT(*) OVER (PARTITION BY a.credit_agreement_mortgage_key) AS n_app_rows,
         ROW_NUMBER() OVER (PARTITION BY a.credit_agreement_mortgage_key ORDER BY a.application_date DESC) AS rn
  FROM %(application)s a
  JOIN %(status)s sd ON a.credit_agreement_mortgage_status_details_key = sd.key
  WHERE a.credit_agreement_mortgage_key IN (SELECT k FROM fleet)
),
app AS (
  SELECT * EXCEPT (rn) FROM app_all WHERE rn = 1
),
scope AS (
  SELECT f.k, f.completion_date, app.application_date, app.status
  FROM fleet f LEFT JOIN app ON app.k = f.k CROSS JOIN params
  WHERE app.application_date BETWEEN params.s AND params.e
     OR f.completion_date BETWEEN params.s AND params.e
     OR (app.status IN ('APPLICATION', 'OFFERED') AND app.application_date >= DATE_SUB(params.e, INTERVAL %(open_years)d YEAR))%(dip_scope)s
),
orig AS (
  SELECT o.credit_agreement_mortgage_key AS k, o.origination_balance AS balance, o.original_annual_interest_rate AS orig_rate,
         o.original_term_months AS term_months, o.number_of_buy_to_let_properties AS btl_props,
         o.servicer_provided_origination_ltv AS ltv, o.original_servicer_provided_valuation AS valuation,
         o.completion_fees_added_to_loan_balance_by_borrower AS fees_added, o.cashback,
         o.original_stressed_interest_coverage_ratio AS icr, o.gross_annual_rental_income AS rent,
         o.income_primary_borrower AS income, o.original_bureau_score_primary_borrower AS bureau%(extra_orig)s
  FROM %(origination)s o
  WHERE o.credit_agreement_mortgage_key IN (SELECT k FROM scope)
  QUALIFY ROW_NUMBER() OVER (PARTITION BY o.credit_agreement_mortgage_key ORDER BY o.date DESC) = 1
),
dp AS (
  SELECT d.credit_agreement_mortgage_key AS k, d.date, d.current_annual_interest_rate AS rate, d.current_interest_margin_rate AS margin,
         d.reversion_margin_rate AS rev_margin, d.first_early_repayment_charge AS erc1, d.second_early_repayment_charge AS erc2,
         d.third_early_repayment_charge AS erc3, d.fourth_early_repayment_charge AS erc4, d.fifth_early_repayment_charge AS erc5,
         d.current_principal_balance AS principal, d.mortgage_product_details_key AS pk
  FROM %(position)s d CROSS JOIN params
  WHERE d.credit_agreement_mortgage_key IN (SELECT k FROM scope)
    AND d.date BETWEEN params.s AND DATE_ADD(params.e, INTERVAL %(snap_days)d DAY)
),
snap_a AS (
  SELECT scope.k, %(sc)s, dp.pk
  FROM scope JOIN dp ON dp.k = scope.k AND dp.date BETWEEN scope.application_date AND DATE_ADD(scope.application_date, INTERVAL %(snap_days)d DAY)
  QUALIFY ROW_NUMBER() OVER (PARTITION BY scope.k ORDER BY dp.date) = 1
),
snap_c AS (
  SELECT scope.k, %(sc)s, dp.pk
  FROM scope JOIN dp ON dp.k = scope.k AND dp.date BETWEEN scope.completion_date AND DATE_ADD(scope.completion_date, INTERVAL %(snap_days)d DAY)
  QUALIFY ROW_NUMBER() OVER (PARTITION BY scope.k ORDER BY dp.date) = 1
),
joined AS (
  SELECT scope.k, f.* EXCEPT (k),
         app.application_date, app.dip_date, app.offer_date, app.declined_date, app.cancelled_date, app.expected_completion_date, app.status, app.n_app_rows,
         o.* EXCEPT (k),
         %(sa)s, %(pa)s,
         %(scc)s, %(pc)s
  FROM scope
  JOIN fleet f ON f.k = scope.k
  LEFT JOIN app ON app.k = scope.k
  LEFT JOIN orig o ON o.k = scope.k
  LEFT JOIN snap_a a ON a.k = scope.k
  LEFT JOIN %(product)s pa ON pa.key = a.pk
  LEFT JOIN snap_c c ON c.k = scope.k
  LEFT JOIN %(product)s pc ON pc.key = c.pk
),
labelled AS (
  SELECT joined.*,
  %(lab_a)s,
  %(lab_c)s
  FROM joined
),
cases AS (
  SELECT labelled.*, %(code_a)s AS a_pricing_code, %(code_c)s AS c_pricing_code
  FROM labelled
)""" % dict(
        od=clean_date('m.origination_date'), rd=clean_date('m.redemption_date'), ad=clean_date('a.application_date'),
        ofd=clean_date('a.offer_date'), dd=clean_date('a.declined_date'), cd=clean_date('a.cancelled_date'),
        ecd=clean_date('a.expected_completion_date'), dipd=clean_date('a.decision_in_principle_date'), open_years=open_years, snap_days=snap_days,
        dip_scope=("\n     OR app.dip_date BETWEEN params.s AND params.e" if include_dip else ''),
        mortgage=t('mortgage'), application=t('application'), status=t('status'), origination=t('origination'),
        position=t('position'), product=t('product'), sc=', '.join('dp.%s' % c.strip() for c in snap_cols.split(',') if c.strip() != 'pk'),
        sa=snapcols('a_', 'a'), pa=prod('a_', 'pa'), scc=snapcols('c_', 'c'), pc=prod('c_', 'pc'),
        lab_a=pricing_code_cols('a_').replace('\n', '\n  '), lab_c=pricing_code_cols('c_').replace('\n', '\n  '),
        code_a=pricing_code_expr('a_'), code_c=pricing_code_expr('c_'), extra_orig=extra_orig, extra_dim=extra_dim)


# ------------------------------------------------------------------------------------------------ the live book at month ends
def book_sql(months_back=12):
    """WITH ... book AS (...): every Fleet loan live at each month end from (@asat month - months_back) to @asat, with its
    last daily position in that month (NULL columns if it has none) and the product and status on that position.
    A loan is live at a month end if it completed by then and had not redeemed by then."""
    return """WITH params AS (
  SELECT DATE(@asat) AS e
),
month_ends AS (
  SELECT LAST_DAY(m) AS me
  FROM params, UNNEST(GENERATE_DATE_ARRAY(DATE_TRUNC(DATE_SUB(params.e, INTERVAL %(mb)d MONTH), MONTH), DATE_TRUNC(params.e, MONTH), INTERVAL 1 MONTH)) AS m
),
loans AS (
  SELECT m.key AS k, %(od)s AS completion_date, %(rd)s AS redemption_date
  FROM %(mortgage)s m
  WHERE m.source_key LIKE 'FLEET~%%'
),
live AS (
  SELECT loans.k, loans.completion_date, loans.redemption_date, month_ends.me
  FROM loans CROSS JOIN month_ends
  WHERE loans.completion_date <= month_ends.me AND (loans.redemption_date IS NULL OR loans.redemption_date > month_ends.me)
),
pos AS (
  SELECT d.credit_agreement_mortgage_key AS k, LAST_DAY(d.date) AS me, d.date,
         d.current_principal_balance AS principal, d.current_total_balance AS total_balance,
         d.current_annual_interest_rate AS rate, d.current_interest_margin_rate AS margin, d.reversion_date, d.reversion_margin_rate AS rev_margin,
         d.months_in_arrears, d.arrears_balance, d.servicer_provided_current_ltv AS current_ltv, d.current_debt_service_coverage_ratio AS dscr,
         d.first_early_repayment_charge AS erc1, d.mortgage_product_details_key AS pk,
         d.credit_agreement_mortgage_status_details_key AS sk, d.credit_agreement_mortgage_details_key AS dk
  FROM %(position)s d CROSS JOIN params
  WHERE d.credit_agreement_mortgage_key IN (SELECT k FROM loans)
    AND d.date BETWEEN DATE_TRUNC(DATE_SUB(params.e, INTERVAL %(mb)d MONTH), MONTH) AND params.e
  QUALIFY ROW_NUMBER() OVER (PARTITION BY d.credit_agreement_mortgage_key, LAST_DAY(d.date) ORDER BY d.date DESC) = 1
),
book AS (
  SELECT live.k, live.me, live.completion_date, live.redemption_date, pos.* EXCEPT (k, me, pk, sk, dk),
         pr.product_code, st.account_status, det.current_interest_rate_type AS rate_type_now, det.current_epc_rating AS epc
  FROM live
  LEFT JOIN pos ON pos.k = live.k AND pos.me = live.me
  LEFT JOIN %(product)s pr ON pr.key = pos.pk
  LEFT JOIN %(status)s st ON st.key = pos.sk
  LEFT JOIN %(details)s det ON det.key = pos.dk
)""" % dict(mb=months_back, od=clean_date('m.origination_date'), rd=clean_date('m.redemption_date'), mortgage=t('mortgage'),
            position=t('position'), product=t('product'), status=t('status'), details=t('details'))


def use_project(project):
    """Run (and bill) queries in another project, e.g. the one the BigQuery console uses for you. The warehouse tables are
    still read from DATA; only where the job runs changes. %%bigquery cells are pointed at it too."""
    global BILLING
    BILLING = project
    for mod in ('bigquery_magics', 'google.cloud.bigquery.magics'):
        try:
            m = __import__(mod, fromlist=['context'])
            m.context.project = project
        except Exception:
            pass
    print('Queries will run in project', project)


def find_billing_projects(max_projects=60):
    """List the projects this account can see and test (with a free dry run) which ones it can run BigQuery queries in."""
    from google.cloud import bigquery
    from google.auth.transport.requests import AuthorizedSession
    creds = CREDENTIALS
    if creds is None:
        ctx = _magic_context()
        if ctx is not None:
            creds = ctx.credentials
        else:
            import google.auth
            creds, _ = google.auth.default(scopes=['https://www.googleapis.com/auth/cloud-platform'])
    sess = AuthorizedSession(creds)
    projects, token = [], None
    while len(projects) < max_projects:
        r = sess.get('https://cloudresourcemanager.googleapis.com/v1/projects', params={'pageToken': token} if token else {}).json()
        projects += [p_['projectId'] for p_ in r.get('projects', []) if p_.get('lifecycleState') == 'ACTIVE']
        token = r.get('nextPageToken')
        if not token: break
    if BILLING not in projects: projects.insert(0, BILLING)
    rows = []
    for p_ in projects[:max_projects]:
        try:
            bigquery.Client(project=p_, credentials=creds).query('SELECT 1', job_config=bigquery.QueryJobConfig(dry_run=True))
            rows.append({'project': p_, 'can run queries': 'Yes'})
        except Exception as e:
            rows.append({'project': p_, 'can run queries': 'No', 'reason': ('no permission' if 'jobs.create' in str(e) or '403' in str(e) else str(e)[:80])})
    out = pd.DataFrame(rows)
    return out.sort_values('can run queries', ascending=False).reset_index(drop=True)


# ------------------------------------------------------------------------------------------------ swap rates (CoF file)
SWAP_TENORS = {'1 month': 's1m', '3 month': 's3m', '6 month': 's6m', '1 year': 's1y', '18 months': 's18m', '2 year': 's2y',
               '3 year': 's3y', '5 year': 's5y', '7 year': 's7y', '10 year': 's10y'}
TERM_SWAP = {'2YR': 's2y', '3YR': 's3y', '5YR': 's5y', '7YR': 's7y', '10YR': 's10y'}     # product term -> matching swap


def find_files(patterns, folder=None):
    """Input files matching the patterns. If INPUT_DIR (or `folder`) is set, only that folder and its sub-folders;
    otherwise this folder and its sub-folders, then the parent folder itself. Newest last."""
    import glob as _glob
    root = folder or INPUT_DIR
    if root:
        searches = [(os.path.expanduser(root), True)]
    else:
        searches = [('.', True), ('..', False)]
    hits = []
    for r_, deep in searches:
        for pat in patterns:
            hits += _glob.glob(os.path.join(r_, '**', pat) if deep else os.path.join(r_, pat), recursive=deep)
    out = []
    for h in sorted(set(os.path.abspath(x) for x in hits), key=os.path.getmtime):
        if h not in out:
            out.append(h)
    return out


def find_swap_files(folder=None, patterns=('CoF_Data*.xlsx', 'CoF*.xlsx', 'Swap_rates*.csv', 'Swap*.csv')):
    """Swap files: CoF_Data*.xlsx / Swap_rates*.csv in INPUT_DIR, this folder or its parent (newest last)."""
    return find_files(patterns, folder)


def ask_for_upload(what):
    """Upload button (Colab) for an outside file. Saves what you choose into INPUT_DIR (or this folder) and returns the paths.
    Choose nothing (or cancel) to skip; returns [] then, or if this runtime has no upload button."""
    try:
        from google.colab import files
    except Exception:
        print('No upload button here: put the %s in %s and re-run.' % (what, os.path.abspath(INPUT_DIR or '.')))
        return []
    print('Upload the %s (or cancel to use the one already in %s):' % (what, os.path.abspath(INPUT_DIR or '.')), flush=True)
    try:
        up = files.upload()
    except Exception as e:
        print('Upload skipped (%s).' % str(e)[:80])
        return []
    dest = INPUT_DIR or '.'
    os.makedirs(dest, exist_ok=True)
    out = []
    for name, data in (up or {}).items():
        path = os.path.join(dest, os.path.basename(name))
        if os.path.abspath(path) != os.path.abspath(name) or not os.path.exists(path):
            with open(path, 'wb') as f_:
                f_.write(data)
        os.utime(path, None)            # newest, so it wins over older copies
        out.append(os.path.abspath(path))
        print('  saved', path)
    return out


# the outside files the notebooks use: label, file name patterns, several files allowed (combined)
INPUT_FILES = {
    'swaps': ('CoF swap rates file (CoF_Data.xlsx, or Swap_rates.csv for older history)', ('CoF_Data*.xlsx', 'CoF*.xlsx', 'Swap_rates*.csv', 'Swap*.csv'), True),
    'pricing_inputs': ('Fleet pricing inputs file (Fleet_Pricing_Inputs_vX.Y.xlsx)', ('Fleet_Pricing_Inputs_v*.xlsx',), False),
    'market_volume': ('UK Finance BTL lending by lender file (competitor_volume_YYYY.csv, table MM11)', ('competitor_volume*.csv', 'competitor_volume*.xlsx', '*MM11*.csv'), False),
    'product_returns': ('product returns file (Fleet_Products_and_Returns.csv: pricing code and RoE per product)', ('Fleet_Products_and_Returns*.csv', '*Products_and_Returns*.csv', '*Products_and_Returns*.xlsx'), False),
    'llpm': ('loan level pricing model workbook (Loan_Level_Pricing_Model_V3.0.xlsm, saved in Excel)', ('Loan_Level_Pricing_Model*.xlsm', '*Pricing_Model*.xlsm'), False),
    'current_pricing': ('current pricing file (Fleet_Current_Pricing.xlsx, or a Daily Dashboard workbook: its Current Pricing tab is read)', ('Fleet_Current_Pricing*.xlsx', 'Daily_Dashboard*.xlsm'), False),
    'llpm_outputs': ('pricing model results: the LLPM workbook after it has run, or its Outputs tab saved as CSV / xlsx', ('LLPM_Outputs*.csv', 'LLPM_Outputs*.xlsx', '*Outputs*.csv', 'Loan_Level_Pricing_Model*.xlsm'), False),
    'dd_history': ('daily dashboard history file (Fleet_Daily_Dashboard_History.xlsx), or a Daily Dashboard workbook to start one', ('Fleet_Daily_Dashboard_History*.xlsx',), False),
}


def need_file(kind, upload=None):
    """An outside file a notebook needs. UPLOAD = 'ask': show an upload button first (what you upload is saved into INPUT_DIR
    and used); cancel it to use the newest matching file already there. UPLOAD = 'auto': use the file already there, and ask
    only if there is none. Returns one path, or a list for kinds that combine several files (swaps)."""
    label, pats, multi = INPUT_FILES[kind]
    mode = upload or UPLOAD
    got = ask_for_upload(label) if mode == 'ask' else []
    found = find_files(pats)
    if multi:
        paths = [p_ for p_ in found if p_ not in got] + got           # uploads last, so they win where files overlap
    else:
        paths = got[-1:] or found[-1:]
    if not paths and mode != 'ask':
        paths = ask_for_upload(label)
    if not paths:
        raise FileNotFoundError('No %s. Upload it when asked, or put it in %s.' % (label, os.path.abspath(INPUT_DIR or '.')))
    print('Using:', ', '.join(os.path.basename(p_) for p_ in paths))
    return paths if multi else paths[0]


def load_swaps(paths=None):
    """Read the CoF swap file(s) (Treasury's CoF_Data.xlsx and/or Swap_rates.csv: a Date column and one column per tenor,
    in %). With no paths, asks for them (need_file). Later files win where dates overlap. Returns a DataFrame indexed by date with columns s1m ... s7y (in %)."""
    paths = paths or need_file('swaps')
    parts = []
    for p_ in paths:
        df = pd.read_excel(p_) if p_.lower().endswith(('.xlsx', '.xls')) else pd.read_csv(p_)
        dcol = next(c for c in df.columns if str(c).strip().lower() in ('date', 'as of', 'asof', 'value date'))
        dates = df[dcol] if pd.api.types.is_datetime64_any_dtype(df[dcol]) else pd.to_datetime(df[dcol].astype(str), dayfirst=True, errors='coerce')
        ren = {c: SWAP_TENORS[str(c).strip().lower()] for c in df.columns if str(c).strip().lower() in SWAP_TENORS}
        d = df.rename(columns=ren)[list(ren.values())].apply(pd.to_numeric, errors='coerce')
        d.index = dates
        d = d[d.index.notna()]
        if d.stack().median() < 1:       # held as decimals
            d = d * 100
        parts.append(d)
        print('Swaps: %s, %d days, %s to %s' % (os.path.basename(p_), len(d), d.index.min().date(), d.index.max().date()))
    allp = pd.concat(parts)
    allp = allp[~allp.index.duplicated(keep='last')].sort_index()
    return allp


def swap_calendar(curve, start, end, max_gap_days=5):
    """One row per calendar day from start to end with the swap to use for that day: the first quote on or after the day
    (the LLPM's rule), or the latest quote before it, but only within max_gap_days. Days inside a longer gap in the file
    (or before it starts) get no swap, so they drop out of margins instead of borrowing a quote from weeks away.
    Days after the last quote use the last quote (the notebooks warn). Prints any gaps."""
    days = pd.date_range(pd.Timestamp(start), pd.Timestamp(end))
    c = curve.sort_index()
    c = c[~c.index.duplicated(keep='last')]
    tol = pd.Timedelta(days=max_gap_days)
    nxt = c.reindex(days, method='bfill', tolerance=tol)
    prev = c.reindex(days, method='ffill', tolerance=tol)
    cal = nxt.fillna(prev)
    after = days > c.index.max()
    if after.any():
        cal.loc[after] = c.iloc[[-1] * int(after.sum())].values
    q = c.index[(c.index >= days[0] - tol) & (c.index <= days[-1])]
    gaps = [(a_, b_) for a_, b_ in zip(q[:-1], q[1:]) if (b_ - a_).days > max_gap_days]
    if c.index.min() > days[0]:
        print('Swap file starts %s: days before then have no swap and are left out of margins.' % c.index.min().date())
    for a_, b_ in gaps:
        print('WARNING: no swap quotes between %s and %s (%d days). Applications in that gap have no swap and are left out of margins. '
              'Add a file that covers it (e.g. Swap_rates.csv with the CoF file).' % (a_.date(), b_.date(), (b_ - a_).days))
    for col in TERM_SWAP.values():          # every matching-term column exists, empty if the file doesn't have that tenor
        if col not in cal.columns:
            cal[col] = np.nan
    cal = cal.dropna(how='all', subset=[c_ for c_ in cal.columns if c_ in TERM_SWAP.values()])
    cal.index.name = 'd'
    return cal


def swaps_cte(cal, name='swaps'):
    """The swap calendar as a SQL CTE (one row per day), so margins over swap are worked out inside BigQuery."""
    cols = [c for c in cal.columns]
    rows = []
    for d, r in cal.iterrows():
        vals = ', '.join(('%.5f' % v if pd.notna(v) else 'CAST(NULL AS FLOAT64)') + ' AS ' + c for c, v in zip(cols, r.values))
        rows.append("STRUCT(DATE '%s' AS d, %s)" % (d.date(), vals))
    return '%s AS (\n  SELECT s.* FROM UNNEST([\n    %s]) AS s\n)' % (name, ',\n    '.join(rows))


def swap_for_term(term_col, alias='sw'):
    """SQL: the matching-term swap for a product term column ('2YR' -> s2y ...)."""
    return 'CASE %s %s END' % (term_col, ' '.join("WHEN '%s' THEN %s.%s" % (k, alias, v) for k, v in TERM_SWAP.items()))


# ----------------------------------------------------------------------------------- other input files
def read_key_inputs(path):
    """Key and Value columns of the 'Key inputs' tab of the pricing inputs file, as a dict (blank values -> None)."""
    x = pd.read_excel(path, sheet_name='Key inputs', header=None)
    hr = next(i for i in range(min(len(x), 30)) if 'Key' in [str(v).strip() for v in x.iloc[i]])
    d = x.iloc[hr + 1:].copy()
    d.columns = [str(v).strip() for v in x.iloc[hr]]
    out = {}
    for k, v in zip(d['Key'], d['Value']):
        if pd.isna(k):
            continue
        out[str(k).strip()] = None if (v is None or (isinstance(v, float) and np.isnan(v)) or str(v).strip() == '') else v
    return out


def _bn(v):
    """'£59,780,000,000' / ' 59.78 ' / ' -   ' -> float (as written; caller sets the unit)."""
    t = str(v).replace('｣', '').replace('£', '').replace('£', '').replace(',', '').replace('%', '').strip()
    if t in ('', '-', 'nan', 'None'):
        return 0.0 if t == '-' else np.nan
    try:
        return float(t)
    except ValueError:
        return np.nan


def read_market_volume(path):
    """UK Finance table MM11 as saved in competitor_volume_YYYY.csv: BTL balances outstanding and BTL gross lending by
    lender for two years side by side, plus an optional mapping to short lender names ('Mapped Specialist Lender').
    Returns dict: lending, balances (DataFrames: lender, name, rank, value_bn, share, rank_prev, value_prev_bn, share_prev),
    year, prev_year, source. Lenders listed twice keep their first row."""
    if str(path).lower().endswith(('.xlsx', '.xls')):
        raw = pd.read_excel(path, header=None, dtype=str)
    else:
        raw = None
        for enc in ('utf-8', 'cp1252', 'latin-1'):
            try:
                raw = pd.read_csv(path, header=None, dtype=str, encoding=enc, keep_default_na=False, skip_blank_lines=False)
                break
            except UnicodeDecodeError:
                continue
    raw = raw.fillna('').astype(str)
    top = {c: ' '.join(raw.iloc[:4, c].tolist()).lower() for c in range(raw.shape[1])}
    starts = {}
    for c, txt in top.items():
        if 'outstanding' in txt and 'balances' not in starts:
            starts['balances'] = c
        elif 'gross lending' in txt and 'lending' not in starts:
            starts['lending'] = c
    if 'lending' not in starts:
        raise ValueError('Could not find the "BTL gross lending" table in %s. Expected UK Finance table MM11 laid out as in competitor_volume_2025.csv.' % os.path.basename(path))
    # short names
    names = {}
    for r in range(min(10, len(raw))):
        for c in range(1, raw.shape[1]):
            if raw.iat[r, c].strip().lower().startswith('mapped'):
                for rr in range(r + 1, len(raw)):
                    a, b = raw.iat[rr, c - 1].strip(), raw.iat[rr, c].strip()
                    if a and a not in names:
                        names[a] = b or a
    out = {'source': os.path.basename(path)}
    for kind, c in starts.items():
        hr = next(r for r in range(len(raw)) if raw.iat[r, c].strip().lower() == 'lender')
        yrs = [int(t) for t in re.findall(r'(\d{4})', ' '.join(raw.iloc[hr, c + 1:c + 7].tolist()))]
        rows = []
        for r in range(hr + 1, len(raw)):
            lender = raw.iat[r, c].strip()
            if not lender:
                if rows:
                    break
                continue
            v = [raw.iat[r, c + j] for j in range(1, 7)]
            rows.append({'lender': lender, 'name': names.get(lender, lender), 'rank': _bn(v[0]), 'value_bn': _bn(v[1]), 'share': _bn(v[2]) / 100,
                         'rank_prev': _bn(v[3]), 'value_prev_bn': _bn(v[4]), 'share_prev': _bn(v[5]) / 100})
        d = pd.DataFrame(rows).drop_duplicates('lender', keep='first').reset_index(drop=True)
        d[['value_bn', 'value_prev_bn']] = d[['value_bn', 'value_prev_bn']].fillna(0.0)
        out[kind] = d
        if yrs:
            out['year'], out['prev_year'] = max(yrs), min(yrs)
    out.setdefault('year', None); out.setdefault('prev_year', None)
    print('Market file %s: gross lending %d lenders, %s £%.1fbn (%s £%.1fbn)%s' % (
        out['source'], len(out['lending']), out['year'], out['lending']['value_bn'].sum(), out['prev_year'], out['lending']['value_prev_bn'].sum(),
        ('; balances %d lenders' % len(out['balances'])) if 'balances' in out else ''))
    return out


def read_product_returns(path):
    """Product RoE table from the returns pack: a short pricing-code column and an RoE column (as '28.80%' or 0.288).
    Returns a DataFrame indexed by pricing code with roe (fraction) and, where present, rate (%)."""
    d = pd.read_excel(path) if str(path).lower().endswith(('.xlsx', '.xls')) else None
    if d is None:
        for enc in ('utf-8', 'cp1252', 'latin-1'):
            try:
                d = pd.read_csv(path, encoding=enc); break
            except UnicodeDecodeError:
                continue
    cols = {str(c).strip().lower(): c for c in d.columns}
    code_col = next((cols[k] for k in cols if 'short' in k or k in ('pricing_code', 'pricing code', 'code')), None)
    roe_col = next((cols[k] for k in cols if k == 'roe'), None)
    if code_col is None or roe_col is None:
        raise ValueError('Need a pricing-code column (e.g. "Short Product Name") and an "RoE" column in %s' % os.path.basename(path))
    pct = lambda v: (float(str(v).replace('%', '').strip()) / (100 if '%' in str(v) or float(str(v).replace('%', '').strip()) > 1.5 else 1)) if str(v).strip() not in ('', 'nan', 'None') else np.nan
    out = pd.DataFrame({'code': d[code_col].astype(str).str.strip().str.upper(), 'roe': d[roe_col].map(pct)})
    rate_col = next((cols[k] for k in cols if 'rate' in k), None)
    if rate_col is not None:
        out['rate'] = d[rate_col].map(lambda v: pct(v) * 100 if str(v).strip() not in ('', 'nan') else np.nan)
    out = out[out['code'].str.match(r'^(BTL|LTD|HMO)') & out['roe'].notna()].drop_duplicates('code').set_index('code')
    print('Product returns: %s, %d pricing codes, RoE %.1f%% to %.1f%%' % (os.path.basename(path), len(out), 100 * out['roe'].min(), 100 * out['roe'].max()))
    return out

In [ ]:
# @title
%%writefile fleet_daily.py
"""fleet_daily: the Fleet daily dashboard (D01), rebuilt from Daily_Dashboard_2.1.xlsm.

What the workbook did, step by step, and where it now lives:
  1. SQL tapes (DIPs & Apps, Comps) ............................ apps_sql(), comps_sql()  (run through fleet_sql.run)
  2. Tape columns (pricing code, fee, term, swap, FAPSM, PS) .... enrich_apps()
  3. Prodinput / ProdInput - PricingSwaps (one row per code) .... build_grid()
  4. Run the LLPM (V3.0) and paste LLPM Output ................... fleet_llpm.run() on that grid
  5. Page 1, Page 2, Page 5 ...................................... page1(), page2(), page5()
  6. Refresh DD macro: Month of LLPM Outputs, Historic Return Data, Swap Rates ... the history workbook (update_history)
  7. Print and share ............................................. render_html() -> one self-contained HTML file

Loan-level rows stay in memory: nothing here displays or writes them. Everything returned is per day / per code totals.

MATCH_EXCEL = True keeps the workbook's own definitions where they differ from what the labels suggest, so the numbers
agree with the Excel dashboard while the two run side by side (see EXCEL_NOTES). Set it to False for the labelled meaning.
"""
import datetime as _dt
import json
import math
import os
import re

import numpy as np
import pandas as pd

VERSION = '2.1'
MATCH_EXCEL = True

EXCEL_NOTES = {
    'fapsm_2y_dod': "2-year FAPSM 'Day on Day' compares today's figure (excluding trackers) with today's figure "
                    "including trackers, not with the previous working day.",
    'flat_share': "Flat share for the pricing grid counts PropertyType = 2. The SQL tape holds text "
                  "(RESIDENTIAL_FLAT), so the share is always 0 and every row uses the non-flat forced-sale discount.",
    'ytd_weights': "Page 5 W.A. RoE and W.A. FAPSM weight each day by the cumulative year-to-date applications, "
                   "not by that day's applications, so later days count for more.",
}

# ---------------------------------------------------------------------------------------------- defaults from the workbook
# Control Panel ERC table (B10:G20): % per policy year by category
ERC_TABLE = {'2YR': [3, 2], '2YR ZF': [3, 2], '2YR LFF': [3, 2], '2YR FF': [3, 2], 'TRC': [2, 1], '5YR': [5, 4, 3, 2, 1],
             '5YR FF': [5, 5, 5, 4, 3], '5YR ZF': [5, 5, 5, 4, 3], '5YR LF': [3, 2], '5YR HFF': [5, 5, 5, 4, 3],
             '7YR': [7, 6, 5, 4, 3]}
TARGETS = {'completions_fy': 1.0e9, 'conversion': 0.5, 'daily_volume': 8.0e6, 'roe': 0.25,
           'fy_working_days': 253}       # Page 5 H86; None = count the working days in the financial year
BANK_HOLIDAYS = ['2024-01-01', '2024-03-29', '2024-04-01', '2024-05-06', '2024-05-27', '2024-08-26', '2024-12-25', '2024-12-26',
                 '2025-01-01', '2025-04-18', '2025-04-21', '2025-05-05', '2025-05-26', '2025-08-25', '2025-12-25', '2025-12-26',
                 '2026-01-01', '2026-04-03', '2026-04-06', '2026-05-04', '2026-05-25', '2026-08-31', '2026-12-25', '2026-12-28',
                 '2027-01-01', '2027-03-26', '2027-03-29', '2027-05-03', '2027-05-31', '2027-08-30', '2027-12-27', '2027-12-28']
EXCL_PS_ROWS = 19            # Page 2 rows 9-27


def bdays(start, end, holidays=None):
    """Working days from start to end inclusive (England and Wales bank holidays left out)."""
    hol = pd.to_datetime(holidays if holidays is not None else BANK_HOLIDAYS)
    return pd.bdate_range(start, end, freq='C', holidays=list(hol))


def prev_bday(d, holidays=None):
    """WORKDAY(d, -1)."""
    d = pd.Timestamp(d)
    days = bdays(d - pd.Timedelta(days=14), d - pd.Timedelta(days=1), holidays)
    return days[-1]


def fy_start(d):
    d = pd.Timestamp(d)
    return pd.Timestamp(d.year if d.month >= 4 else d.year - 1, 4, 1)


# ---------------------------------------------------------------------------------------------- 1. SQL
def apps_sql(fs):
    """DIPs & Apps tape (from fleet_daily_dips_and_apps.sql): one row per Fleet case whose DIP or application date is
    between @start and @end. No identifiers: `ord` keeps the tape's order (by source key), which the cashback lookup needs."""
    return """
WITH app AS (
  SELECT a.credit_agreement_mortgage_key AS k, a.decision_in_principle_date, a.application_date, sd.application_status
  FROM {application} a
  JOIN {status} sd ON a.credit_agreement_mortgage_status_details_key = sd.key
  JOIN {mortgage} m ON m.key = a.credit_agreement_mortgage_key
  WHERE m.source_key LIKE 'FLEET~%'
  QUALIFY ROW_NUMBER() OVER (PARTITION BY a.credit_agreement_mortgage_key ORDER BY a.application_date DESC) = 1
),
scope AS (
  SELECT app.k FROM app
  WHERE app.decision_in_principle_date BETWEEN DATE(@start) AND DATE(@end)
     OR app.application_date BETWEEN DATE(@start) AND DATE(@end)
),
now_ AS (
  SELECT dp.credit_agreement_mortgage_key AS k, dp.current_total_balance, dp.current_annual_interest_rate,
         dp.current_interest_margin_rate, dp.mortgage_product_details_key
  FROM {position} dp
  WHERE dp.credit_agreement_mortgage_key IN (SELECT k FROM scope)
    AND dp.date BETWEEN DATE_SUB(DATE(@start), INTERVAL 400 DAY) AND DATE(@end)
  QUALIFY ROW_NUMBER() OVER (PARTITION BY dp.credit_agreement_mortgage_key ORDER BY dp.date DESC) = 1
),
orig AS (
  SELECT o.credit_agreement_mortgage_key AS k, o.origination_balance, o.original_annual_interest_rate,
         o.number_of_buy_to_let_properties, o.cashback
  FROM {origination} o
  WHERE o.credit_agreement_mortgage_key IN (SELECT k FROM scope)
  QUALIFY ROW_NUMBER() OVER (PARTITION BY o.credit_agreement_mortgage_key ORDER BY o.date DESC) = 1
),
d AS (
  SELECT m.source_key, m.property_type, m.tenure, app.application_status, app.decision_in_principle_date, app.application_date,
         orig.origination_balance, orig.original_annual_interest_rate, orig.number_of_buy_to_let_properties, orig.cashback,
         now_.current_total_balance, now_.current_annual_interest_rate, now_.current_interest_margin_rate,
         pd.product_code, pd.product_description
  FROM scope
  JOIN {mortgage} m ON m.key = scope.k
  JOIN app ON app.k = scope.k
  LEFT JOIN orig ON orig.k = scope.k
  LEFT JOIN now_ ON now_.k = scope.k
  LEFT JOIN {product} pd ON pd.key = now_.mortgage_product_details_key
)
SELECT
  ROW_NUMBER() OVER (ORDER BY CAST(source_key AS STRING)) AS ord,
  ROUND(SAFE_CAST(origination_balance AS FLOAT64), 2) AS OriginalBalance,
  ROUND(CASE WHEN UPPER(TRIM(CAST(application_status AS STRING))) = 'COMPLETION' AND current_total_balance IS NOT NULL
             THEN SAFE_CAST(current_total_balance AS FLOAT64) ELSE SAFE_CAST(origination_balance AS FLOAT64) END, 2) AS CurrentBalance,
  ROUND(COALESCE(SAFE_CAST(current_annual_interest_rate AS FLOAT64), SAFE_CAST(original_annual_interest_rate AS FLOAT64)) * 100, 2) AS CurrentInterestRate,
  CASE WHEN UPPER(TRIM(CAST(property_type AS STRING))) IN ('', 'UNKNOWN', 'NONE', 'NULL') THEN NULL ELSE TRIM(CAST(property_type AS STRING)) END AS PropertyType,
  CASE WHEN SAFE_CAST(number_of_buy_to_let_properties AS FLOAT64) >= 0 THEN SAFE_CAST(number_of_buy_to_let_properties AS FLOAT64) END AS NumberofBuytoLetProperties,
  IFNULL(CASE WHEN UPPER(TRIM(CAST(product_code AS STRING))) IN ('', 'UNKNOWN', 'NONE', 'NULL') THEN NULL ELSE TRIM(CAST(product_code AS STRING)) END, 'ND') AS ProductCode,
  IFNULL(CASE WHEN UPPER(TRIM(CAST(product_description AS STRING))) IN ('', 'UNKNOWN', 'NONE', 'NULL') THEN NULL ELSE TRIM(CAST(product_description AS STRING)) END, 'ND') AS ProductDescription,
  IFNULL(CASE WHEN UPPER(TRIM(CAST(application_status AS STRING))) IN ('', 'UNKNOWN', 'NONE', 'NULL') THEN NULL ELSE UPPER(TRIM(CAST(application_status AS STRING))) END, 'ND') AS LoanStatus,
  {dip} AS DIPDate,
  {appd} AS ApplicationDate,
  CASE WHEN UPPER(TRIM(CAST(tenure AS STRING))) IN ('', 'UNKNOWN', 'NONE', 'NULL') THEN NULL ELSE INITCAP(REPLACE(TRIM(CAST(tenure AS STRING)), '_', ' ')) END AS Tenure,
  CASE WHEN SAFE_CAST(cashback AS FLOAT64) = -1 THEN NULL ELSE SAFE_CAST(cashback AS FLOAT64) END AS Cashback
FROM d
""".format(application=fs.t('application'), status=fs.t('status'), mortgage=fs.t('mortgage'), position=fs.t('position'),
           origination=fs.t('origination'), product=fs.t('product'),
           dip=fs.clean_date('SAFE_CAST(decision_in_principle_date AS DATE)'), appd=fs.clean_date('SAFE_CAST(application_date AS DATE)'))


def comps_sql(fs):
    """Completions tape (from fleet_daily_completions.sql): one row per Fleet loan completed between @start and @end,
    retention included, with the product on its latest daily position."""
    return """
WITH scope AS (
  SELECT m.key AS k, m.source_key, {od} AS odate
  FROM {mortgage} m
  WHERE m.source_key LIKE 'FLEET~%'
    AND {od} BETWEEN DATE(@start) AND DATE(@end)
),
now_ AS (
  SELECT dp.credit_agreement_mortgage_key AS k, dp.mortgage_product_details_key
  FROM {position} dp
  WHERE dp.credit_agreement_mortgage_key IN (SELECT k FROM scope)
    AND dp.date BETWEEN DATE(@start) AND DATE(@end)
  QUALIFY ROW_NUMBER() OVER (PARTITION BY dp.credit_agreement_mortgage_key ORDER BY dp.date DESC) = 1
),
orig AS (
  SELECT o.credit_agreement_mortgage_key AS k, o.origination_balance
  FROM {origination} o
  WHERE o.credit_agreement_mortgage_key IN (SELECT k FROM scope)
  QUALIFY ROW_NUMBER() OVER (PARTITION BY o.credit_agreement_mortgage_key ORDER BY o.date DESC) = 1
)
SELECT scope.odate AS LoanOriginationDate,
  ROUND(SAFE_CAST(orig.origination_balance AS FLOAT64), 2) AS OriginalBalance,
  IFNULL(CASE WHEN UPPER(TRIM(CAST(pd.product_code AS STRING))) IN ('', 'UNKNOWN', 'NONE', 'NULL') THEN NULL ELSE TRIM(CAST(pd.product_code AS STRING)) END, 'ND') AS ProductCode,
  IFNULL(CASE WHEN UPPER(TRIM(CAST(pd.product_description AS STRING))) IN ('', 'UNKNOWN', 'NONE', 'NULL') THEN NULL ELSE TRIM(CAST(pd.product_description AS STRING)) END, 'ND') AS ProductDescription
FROM scope
LEFT JOIN orig ON orig.k = scope.k
LEFT JOIN now_ ON now_.k = scope.k
LEFT JOIN {product} pd ON pd.key = now_.mortgage_product_details_key
WHERE scope.odate IS NOT NULL
""".format(mortgage=fs.t('mortgage'), position=fs.t('position'), origination=fs.t('origination'), product=fs.t('product'),
           od=fs.clean_date('SAFE_CAST(m.origination_date AS DATE)'))


# ---------------------------------------------------------------------------------------------- 2. tape columns
_DESC = re.compile(r'^\s*(BTL|LTD|HMO)\s+(\d)YR\s+(FIX|TRC)\s+(\d+)%', re.I)
_LIFE = re.compile(r'^\s*(BTL|LTD|HMO)\s+LIFE\b', re.I)


def fee_from_description(desc):
    """Fee as the product master holds it: a fraction for % fees (0.03), pounds for fixed fees (1499), 0 for none."""
    s = str(desc or '')
    body = re.sub(r'\b\d{2}/\d{2}\b', ' ', s)
    body = re.sub(r'BBR\s*\+\s*\d+(\.\d+)?%', ' ', body, flags=re.I)
    pcts = re.findall(r'(?<![\d.])(\d+(?:\.\d+)?)%', body)
    skip = 2 if re.match(r'^\s*\w+\s+\dYR\s+FIX', s, re.I) else 1        # LTV, then the rate on fixed products
    if len(pcts) > skip:
        return float(pcts[skip]) / 100
    gbp = re.findall(r'(?<![\d.%])\b(\d{3,4})\b(?![%/])', body)
    return float(gbp[-1]) if gbp else 0.0


def pricing_code(desc, product_code=None):
    """The pricing code the dashboard groups by, from the product description (the workbook looked it up in
    FleetProductMaster). Scheme + term + LTV, then TRC (tracker), ZF (no fee), FF (fixed £ fee), HPF (fee of 4.5% or more),
    G (EPC A-C). Product switches -> <scheme><term>YRLOYAL. Lifetime trackers -> <scheme>LIFE. None if unreadable."""
    s = str(desc or '').strip()
    if not s or s.upper() == 'ND':
        return None
    u = s.upper()
    m = _LIFE.match(u)
    if m:
        pc = str(product_code or '').upper()[:3]
        return (pc if pc in ('BTL', 'LTD', 'HMO') else m.group(1)) + 'LIFE'
    m = _DESC.match(u)
    if not m:
        return None
    scheme, term, kind, ltv = m.groups()
    pc = str(product_code or '').upper()[:3]
    if pc in ('BTL', 'LTD', 'HMO'):                  # the product code's scheme wins (the workbook's Cohort column)
        scheme = pc
    toks = u.split()
    if 'PS' in toks:                                  # product switches: <scheme><term>YRLOYAL, trackers LOYALTRC
        return '%s%sYRLOYAL%s' % (scheme, term, 'TRC' if kind == 'TRC' else '')
    code = '%s%sYR%s' % (scheme, term, ltv)
    fee = fee_from_description(s)
    if kind == 'TRC':                                 # trackers: TRC, or FFTRC with a fixed £ fee
        return code + ('FFTRC' if fee > 1 else 'TRC')
    if fee == 0:
        return code + ('G' if 'G' in toks else '') + 'ZF'
    if fee > 1:
        return code + ('G' if 'G' in toks else '') + 'FF'
    if fee >= 0.045:
        return code + 'HPF'
    return code + ('G' if 'G' in toks else '')


def read_product_codes(path):
    """Fleet_Product_Codes.csv: ProductCode -> PricingCode (and Fee), as the FleetProductMaster held them. Codes found here
    win over the description rule; new product codes fall back to the rule (and are listed in the checks)."""
    t = pd.read_csv(path, dtype=str)
    t = t[t['ProductCode'].notna() & t['PricingCode'].notna()]
    return dict(zip(t['ProductCode'].str.strip(), t['PricingCode'].str.strip()))


def product_codes_from_workbook(path):
    """The ProductCode -> PricingCode / Fee pairs in a Daily Dashboard workbook's DIPs&Apps tab (product level only)."""
    import openpyxl
    wb = openpyxl.load_workbook(path, data_only=True, read_only=True)
    rows = list(wb['DIPs&Apps'].iter_rows(values_only=True))
    wb.close()
    h = [str(x) for x in rows[0]]
    t = pd.DataFrame([r[:len(h)] for r in rows[1:] if r and r[0] is not None], columns=h)
    t = t[~t['PricingCode'].astype(str).isin(['ND', '#N/A', '#ERROR!', 'None', 'nan'])]
    out = t.groupby('ProductCode').agg(PricingCode=('PricingCode', 'first'), Fee=('Fee', 'first'),
                                       ProductDescription=('ProductDescription', 'first')).reset_index()
    return out


def swap_on(dates, curve, col):
    """INDEX(Swap Rates, MATCH(date, dates)): the last quote on or before each date (in %)."""
    c = curve[col].dropna().sort_index()
    idx = np.searchsorted(c.index.values.astype('datetime64[ns]'), pd.to_datetime(dates).values.astype('datetime64[ns]'), side='right') - 1
    out = np.where(idx >= 0, c.values[np.clip(idx, 0, len(c) - 1)], np.nan)
    return pd.Series(out, index=getattr(dates, 'index', None))


def enrich_apps(apps, swaps, bbr, code_map=None):
    """The DIPs & Apps formula columns: PricingCode, Fee, Fixed Term, Swap, FAPSM, Loyalty, Cohort, Tenor, Fee Type,
    Product Switch. swaps: DataFrame indexed by date with columns '2Y', '5Y' in %. code_map: optional
    {ProductCode: PricingCode} (e.g. from the product master) that wins over the description rule."""
    a = apps.copy()
    a['ApplicationDate'] = pd.to_datetime(a['ApplicationDate'], errors='coerce')
    a['DIPDate'] = pd.to_datetime(a['DIPDate'], errors='coerce')
    for c in ('OriginalBalance', 'CurrentBalance', 'CurrentInterestRate', 'NumberofBuytoLetProperties'):
        a[c] = pd.to_numeric(a[c], errors='coerce')
    desc = a['ProductDescription'].astype(str)
    code = pd.Series([pricing_code(d_, p_) for d_, p_ in zip(desc, a['ProductCode'])], index=a.index)
    a['CodeSource'] = 'rule'
    if code_map:
        mapped = a['ProductCode'].map(code_map)
        a.loc[mapped.notna(), 'CodeSource'] = 'list'
        code = mapped.where(mapped.notna(), code)
    a['PricingCode'] = np.where(a['ProductCode'].astype(str) == 'ND', 'ND', code.fillna('UNMAPPED'))
    a['Fee'] = np.where(a['ProductCode'].astype(str) == 'ND', np.nan, desc.map(fee_from_description))
    term = desc.str[4:5]
    a['Fixed Term'] = pd.to_numeric(term, errors='coerce')                         # NaN = 'ND' (LIFE, unreadable)
    sw2 = swap_on(a['ApplicationDate'], swaps, '2Y') / 100
    sw5 = swap_on(a['ApplicationDate'], swaps, '5Y') / 100
    sw = np.where(a['Fixed Term'] == 2, sw2, sw5)
    a['Swap'] = np.where(a['Fixed Term'].isna() | a['ApplicationDate'].isna(), bbr, sw)
    fee = a['Fee'].astype(float)
    fee_pct = np.where(fee > 1, fee / a['OriginalBalance'], fee)
    yrs = a['Fixed Term'].fillna(2.75)
    fapsm = a['CurrentInterestRate'] / 100 + fee_pct / yrs - a['Swap']
    a['FAPSM'] = np.where(a['ApplicationDate'].isna() | (a['ProductCode'].astype(str) == 'ND'), np.nan, fapsm)
    a['Loyalty'] = a['PricingCode'].astype(str).str.contains('LOYAL')
    a['Cohort'] = a['ProductCode'].astype(str).str[:3]
    a['Tenor'] = np.where(desc.str.upper().str.contains('LIFE'), 'LIFE', desc.str[4:7])
    pclc = a['PricingCode'].astype(str).str[3:]
    a['Fee Type'] = np.where(pclc.str.contains('FF'), 'FF', np.where(pclc.str.contains('ZF'), 'ZF', 'Standard'))
    a['Product Switch'] = np.where(desc.str.upper().str.contains('PS'), 'Y', 'N')
    a['Scheme'] = a['PricingCode'].astype(str).str[:3]
    return a


def enrich_comps(comps, code_map=None):
    c = comps.copy()
    c['LoanOriginationDate'] = pd.to_datetime(c['LoanOriginationDate'], errors='coerce')
    c['OriginalBalance'] = pd.to_numeric(c['OriginalBalance'], errors='coerce')
    code = pd.Series([pricing_code(d_, p_) for d_, p_ in zip(c['ProductDescription'].astype(str), c['ProductCode'])], index=c.index)
    if code_map:
        mapped = c['ProductCode'].map(code_map)
        code = mapped.where(mapped.notna(), code)
    c['PricingCode'] = code.fillna('UNMAPPED')
    c['Product Switch'] = np.where(c['ProductDescription'].astype(str).str.upper().str.contains('PS'), 'Y', 'N')
    return c


# ---------------------------------------------------------------------------------------------- pricing input file
def _f(v, default=None):
    try:
        if v is None or (isinstance(v, str) and v.strip() in ('', 'n/a', 'N/A')):
            return default
        x = float(v)
        return default if math.isnan(x) else x
    except (TypeError, ValueError):
        return default


def read_current_pricing(path):
    """The 'Current Pricing' sheet (the Daily Dashboard's own tab, or Fleet_Current_Pricing.xlsx, same layout).
    Returns dict: products (B:M), loyalty (Q:Z), returns (AB:AO, with 'proposed_roe' = AQ4), defaults (AS:AY), erc table."""
    import openpyxl
    wb = openpyxl.load_workbook(path, data_only=True, read_only=True)
    ws = wb['Current Pricing']
    rows = [list(r) for r in ws.iter_rows(min_row=1, max_row=200, max_col=52, values_only=True)]
    col = lambda L: sum((ord(ch) - 64) * 26 ** i for i, ch in enumerate(reversed(L))) - 1
    get = lambda r, L: rows[r][col(L)] if r < len(rows) and col(L) < len(rows[r]) else None
    prods, loyal, rets, dflt = [], [], [], []
    for r in range(2, len(rows)):
        a_s = get(r, 'AS')                                   # defaults start on row 3 (header on row 2)
        if a_s not in (None, 0, '0', '') and str(a_s).strip() != 'ProductCode':
            dflt.append({'code': str(a_s).strip(), 'Original_Balance': _f(get(r, 'AT')), 'Current_Balance': _f(get(r, 'AU')),
                         'Property_Type': _f(get(r, 'AV')), 'Perc_Portfolio': _f(get(r, 'AW')), 'Cashback': _f(get(r, 'AX')),
                         'Tenure': _f(get(r, 'AY'))})
        if r < 3:
            continue
        b = get(r, 'B')
        if b not in (None, 0, '0', '', 0.0):
            prods.append({'code': str(b).strip(), 'name': get(r, 'A'), 'new_product': get(r, 'C'), 'reprice': get(r, 'D'),
                          'rate_current': _f(get(r, 'E')), 'rate': _f(get(r, 'F'), 0.0), 'fee_current': _f(get(r, 'H')),
                          'fee': _f(get(r, 'I'), 0.0), 'revert_current': _f(get(r, 'K')), 'revert': _f(get(r, 'L'), 0.0)})
        q = get(r, 'Q')
        if q not in (None, 0, '0', ''):
            loyal.append({'code': str(q).strip(), 'base': get(r, 'P'), 'rate_current': _f(get(r, 'R')), 'rate': _f(get(r, 'S'), 0.0),
                          'fee_current': _f(get(r, 'U')), 'fee': _f(get(r, 'V'), 0.0), 'revert': _f(get(r, 'Y'), 0.0)})
        ab = get(r, 'AB')
        if ab not in (None, 0, '0', '') and r >= 3 and str(ab).strip() != 'Product Code':
            rets.append({'code': str(ab).strip(), 'type': get(r, 'AD'), 'mix_current': _f(get(r, 'AE'), 0.0),
                         'mix_proposed': _f(get(r, 'AF'), 0.0), 'swap': _f(get(r, 'AM')), 'net_return': _f(get(r, 'AN')),
                         'roe': _f(get(r, 'AO'))})
    proposed = _f(get(3, 'AQ'))
    erc = dict(ERC_TABLE)
    if 'ERCs' in wb.sheetnames:
        for r in wb['ERCs'].iter_rows(min_row=2, values_only=True):
            if r and r[0]:
                erc[str(r[0]).strip()] = [int(_f(x, 0)) for x in r[1:6] if _f(x) is not None]
    wb.close()
    rets_df = pd.DataFrame(rets)
    if proposed is None and len(rets_df):
        w = rets_df['mix_proposed'].fillna(0)
        proposed = float((w * rets_df['roe'].fillna(0)).sum() / w.sum()) if w.sum() else None
    swaps = {}
    for t_ in ('2YR', '5YR'):
        hit = rets_df[rets_df['type'] == t_]['swap'].dropna() if len(rets_df) else []
        swaps[t_] = float(hit.iloc[0]) if len(hit) else None
    return {'products': pd.DataFrame(prods), 'loyalty': pd.DataFrame(loyal), 'returns': rets_df, 'defaults': pd.DataFrame(dflt),
            'proposed_roe': proposed, 'pricing_swaps': swaps, 'erc': erc, 'source': os.path.basename(str(path))}


def code_list(cp):
    """Prodinput A column (without the -10 rows): current product codes in sheet order, then loyalty codes."""
    return list(cp['products']['code']) + list(cp['loyalty']['code'] if len(cp['loyalty']) else [])


# ---------------------------------------------------------------------------------------------- 3. the pseudo-loan grid
def _ptype(code):
    return 'LIFE' if 'LIFE' in code else code[3:6]


def _erc(code, erc_table):
    p = _ptype(code)
    if code.endswith('F'):
        cat = p + ' ' + code[-2:]
    elif code.endswith('TRC'):
        cat = 'TRC'
    else:
        cat = p
    v = list(erc_table.get(cat, []))
    return (v + [0] * 5)[:5]


def _ltv(code):
    try:
        return float(code[:8][-2:])
    except ValueError:
        return 75.0


def build_grid(cp, apps, rd, bbr, match_excel=None):
    """Prodinput (App rows, then the -10bp rows) and ProdInput - PricingSwaps, in the LLPM's ModellingData layout.
    apps: enriched tape. rd: reporting date. Returns a DataFrame with the engine's columns plus Code and Run."""
    match_excel = MATCH_EXCEL if match_excel is None else match_excel
    rd = pd.Timestamp(rd)
    codes = code_list(cp)
    prod = cp['products'].set_index('code')
    loy = cp['loyalty'].set_index('code') if len(cp['loyalty']) else pd.DataFrame()
    dflt = cp['defaults'].set_index('code') if len(cp['defaults']) else pd.DataFrame()
    today = apps[apps['ApplicationDate'] == rd]
    ob_w = today['OriginalBalance'].fillna(0)
    if match_excel:
        flat = pd.to_numeric(today['PropertyType'], errors='coerce') == 2
    else:
        flat = today['PropertyType'].astype(str).str.upper().str.contains('FLAT|MAISONETTE')
    first_cb = apps.sort_values('ord').drop_duplicates('PricingCode').set_index('PricingCode')['Cashback'] if 'ord' in apps else pd.Series(dtype=float)

    def rate_fee(code):
        if code in prod.index:
            return prod.at[code, 'rate'], prod.at[code, 'fee'], prod.at[code, 'revert']
        if len(loy) and code in loy.index:
            return loy.at[code, 'rate'], loy.at[code, 'fee'], loy.at[code, 'revert']
        return np.nan, 0.0, 0.03

    rows = []
    for run in ('App', 'App-10', 'PricingSwaps'):
        for code in codes:
            ident = code + ('-10' if run == 'App-10' else '')
            rate, fee, _rev = rate_fee(code)
            rate = rate - (0.001 if run == 'App-10' else 0.0)
            idx = 'BBR' if ('TRC' in code or 'LIFE' in code) else 'FIXED'
            ltv = _ltv(code)
            if run == 'PricingSwaps':
                d = dflt.loc[code] if (len(dflt) and code in dflt.index) else None
                ob = d['Original_Balance'] if d is not None and d['Original_Balance'] is not None else 300000.0
                cbal = ob
                if match_excel:                              # the PricingSwaps sheet takes the flat share from the tape too
                    m = today['PricingCode'] == code
                    tot = ob_w[m].sum()
                    ptype_share = float(ob_w[m][flat[m]].sum() / tot) if tot else 0.0
                else:
                    ptype_share = d['Property_Type'] if d is not None and d['Property_Type'] is not None else 0.0
                perc = d['Perc_Portfolio'] if d is not None and d['Perc_Portfolio'] is not None else 0.4
                cashback = d['Cashback'] if d is not None and d['Cashback'] is not None else 0.0
                tenure = d['Tenure'] if d is not None and d['Tenure'] is not None else 0.7
            else:
                m = today['PricingCode'] == code
                w = ob_w[m]
                ob = float(today.loc[m, 'OriginalBalance'].mean()) if m.any() else 0.0
                cbal = float(today.loc[m, 'CurrentBalance'].mean()) if m.any() else 0.0
                tot = w.sum()
                share = lambda mask: float(w[mask[m]].sum() / tot) if tot else 0.0
                ptype_share = share(flat)
                perc = share(today['NumberofBuytoLetProperties'] > 3)
                tenure = share(today['Tenure'] == 'Freehold')
                cb_ = first_cb.get(code, np.nan)
                cashback = 0.0 if (cb_ is None or pd.isna(cb_)) else float(cb_)
            fee_in = fee / ob if ('FF' in code and ob) else (0.0 if 'FF' in code else fee)
            e = _erc(code, cp.get('erc', ERC_TABLE))
            rows.append({'Identifier': ident, 'Occupancy_Type': 'BTL', 'Original_Balance': ob, 'Current_Balance': cbal,
                         'Loan_Status': 'APPLICATION', 'Application_Date': rd, 'Product_Scheme': code[:3], 'Product_Type': _ptype(code),
                         'Current_Rate': rate, 'Initial_Margin': rate if idx == 'FIXED' else rate - bbr, 'Initial_Index': idx,
                         'Revert_Margin': 0.03, 'Revert_Index': 'BBR', 'Repayment_Method': 'Interest Only', 'Loan_Purpose': 1,
                         'Loan_Term': 180, 'ERC_Year1': e[0], 'ERC_Year2': e[1], 'ERC_Year3': e[2], 'ERC_Year4': e[3], 'ERC_Year5': e[4],
                         'Current_LTV': ltv, 'Property_Type': ptype_share, 'Original_Valuation': ob / (ltv / 100) if ltv else 0.0,
                         'Valuation_Date': rd, 'Capitalised_Fees': fee_in, 'Perc_Portfolio': perc, 'Cashback': cashback,
                         'Tenure': tenure, 'Code': code, 'Run': 'PricingSwaps' if run == 'PricingSwaps' else 'App'})
    g = pd.DataFrame(rows)
    return g


def run_grid(grid, A, llpm):
    """Run the pricing engine on the grid. Returns the engine output with Code and Run attached."""
    cols = [c for c in llpm.GRID_COLUMNS]
    L = grid.reindex(columns=cols)
    out = llpm.run(L, A)
    out['Code'] = grid['Code'].values
    out['Run'] = np.where(grid['Identifier'].str.endswith('-10'), 'App-10', grid['Run'].values)
    return out


# ---------------------------------------------------------------------------------------------- 4. pages
def _sum(s):
    return float(np.nansum(s)) if len(s) else 0.0


def _wavg(v, w):
    v, w = np.asarray(v, float), np.asarray(w, float)
    m = np.isfinite(v) & np.isfinite(w)
    return float((v[m] * w[m]).sum() / w[m].sum()) if w[m].sum() else np.nan


def fapsm(apps, day, term=None, excl_trc=False):
    d = apps[(apps['ApplicationDate'] == pd.Timestamp(day)) & apps['FAPSM'].notna()]
    if term is None:
        d = d[d['Fixed Term'].isin([2, 5])]
    else:
        d = d[d['Fixed Term'] == term]
    if excl_trc:
        d = d[~d['PricingCode'].astype(str).str.endswith('TRC')]
    return _wavg(d['FAPSM'], d['OriginalBalance'])


def page2(cp, apps, out, rd, history_codes=None, match_excel=None):
    """Live products table. One row per code: pricing RoE, pricing RoE at today's swaps, volume, return today and at -10bp,
    annual contribution, 7-day and month volumes with their volume-weighted returns (from the history of daily runs)."""
    match_excel = MATCH_EXCEL if match_excel is None else match_excel
    rd = pd.Timestamp(rd)
    codes = code_list(cp)
    rets = cp['returns'].drop_duplicates('code').set_index('code') if len(cp['returns']) else pd.DataFrame()
    o = out.set_index(['Code', 'Run'])
    get = lambda c, r, k: float(o.at[(c, r), k]) if (c, r) in o.index else 0.0
    mstart = rd - pd.Timedelta(days=30)   # rolling 30-day window: applications after this date up to the reporting date
    hist = history_codes if history_codes is not None else pd.DataFrame(columns=['Date', 'Code', 'Run', 'RoE', 'NetReturn'])
    hist = hist[hist['Run'] == 'App'] if len(hist) else hist
    rows = []
    for i, c in enumerate(codes):
        m = apps['PricingCode'] == c
        vol = _sum(apps.loc[m & (apps['ApplicationDate'] == rd), 'OriginalBalance'])
        roe, nr = get(c, 'App', 'RoE'), get(c, 'App', 'NetReturn')
        r = {'code': c, 'loyalty': 'LOYAL' in c, 'pos': i,
             'pricing_roe': float(rets.at[c, 'roe']) if (len(rets) and c in rets.index and rets.at[c, 'roe'] is not None) else 0.0,
             'pricing_roe_today_swaps': get(c, 'PricingSwaps', 'RoE'), 'volume': vol, 'roe': roe, 'roe_10': get(c, 'App-10', 'RoE'),
             'net_return': nr, 'contribution': vol * nr, 'nim': get(c, 'App', 'NIM%')}
        for key, start in (('wk', rd - pd.Timedelta(days=7)), ('mth', mstart)):
            sel = apps[m & (apps['ApplicationDate'] > start) & (apps['ApplicationDate'] <= rd)]
            byday = sel.groupby('ApplicationDate')['OriginalBalance'].sum()
            r[key + '_volume'] = float(byday.sum())
            h = hist[(hist['Code'] == c)].drop_duplicates('Date', keep='last').set_index('Date') if len(hist) else pd.DataFrame()
            num = den = cnum = 0.0
            for d_, v_ in byday.items():
                if d_ == rd:
                    rr, nn = roe, nr
                elif len(h) and d_ in h.index and _f(h.at[d_, 'Balance'], 0) > 0:
                    # a saved run with no applications for the code that day (cases keyed later) has no return to use
                    rr, nn = float(h.at[d_, 'RoE']), float(h.at[d_, 'NetReturn'])
                else:
                    continue
                num += v_ * rr; den += v_; cnum += v_ * nn
            r[key + '_roe'] = num / den if den else np.nan
            r[key + '_covered'] = den / r[key + '_volume'] if r[key + '_volume'] else np.nan
            r[key + '_contribution'] = cnum
        rows.append(r)
    t = pd.DataFrame(rows)
    base = ~t['loyalty']                                      # all codes except the loyalty (product switch) ones
    def avg(mask):
        s = t[mask]
        v = s['volume'].sum()
        return {'volume': v, 'pricing_roe': _wavg(s['pricing_roe'], s['volume']),
                'pricing_roe_today_swaps': _wavg(s['pricing_roe_today_swaps'], s['volume']),
                'roe': _wavg(s['roe'], s['volume']), 'roe_10': _wavg(s['roe_10'], s['volume']),
                'contribution': s['contribution'].sum(), 'wk_volume': s['wk_volume'].sum(), 'wk_roe': _wavg(s['wk_roe'], s['wk_volume']),
                'mth_volume': s['mth_volume'].sum(), 'mth_roe': _wavg(s['mth_roe'], s['mth_volume'])}
    return t, {'excl_ps': avg(base), 'total': avg(pd.Series(True, index=t.index)), 'excl_ps_codes': list(t.loc[base, 'code'])}


def page1(cp, apps, comps, rd, p2avg, swaps, match_excel=None, holidays=None):
    """Headline tiles: volumes and counts with day-on-day, daily returns, FAPSM, swaps."""
    match_excel = MATCH_EXCEL if match_excel is None else match_excel
    rd = pd.Timestamp(rd)
    pv = prev_bday(rd, holidays)
    def vol(df, dcol, day, ps=None):
        m = df[dcol] == day
        if ps is not None:
            m &= df['Product Switch'] == ps
        return _sum(df.loc[m, 'OriginalBalance']), int(m.sum())
    a_now, n_now = vol(apps, 'ApplicationDate', rd, 'N'); a_pv, n_pv = vol(apps, 'ApplicationDate', pv, 'N')
    t_now, tn_now = vol(apps, 'ApplicationDate', rd); t_pv, tn_pv = vol(apps, 'ApplicationDate', pv)
    c_now, cn_now = vol(comps, 'LoanOriginationDate', rd); c_pv, cn_pv = vol(comps, 'LoanOriginationDate', pv)
    ex = p2avg['excl_ps']
    proposed = cp.get('proposed_roe') or np.nan
    f2 = fapsm(apps, rd, 2, excl_trc=True)
    f5 = fapsm(apps, rd, 5)
    fa = fapsm(apps, rd, None)
    f2_dod = f2 - (fapsm(apps, rd, 2) if match_excel else fapsm(apps, pv, 2, excl_trc=True))
    sw = lambda d, col: (float(swap_on(pd.Series([d]), swaps, col).iloc[0]) / 100) if (col in swaps.columns and swaps[col].notna().any()) else np.nan
    return {
        'date': rd, 'prev': pv,
        'apps': {'value': a_now, 'dod': a_now - a_pv, 'count': n_now, 'count_dod': n_now - n_pv},
        'apps_ps': {'value': t_now, 'dod': t_now - t_pv, 'count': tn_now, 'count_dod': tn_now - tn_pv},
        'comps': {'value': c_now, 'dod': c_now - c_pv, 'count': cn_now, 'count_dod': cn_now - cn_pv},
        'returns': {'roe': ex['roe'], 'vs_pricing': ex['roe'] - ex['pricing_roe'], 'vs_proposed': ex['roe'] - proposed},
        'returns_10': {'delta': ex['roe_10'] - ex['roe'], 'vs_pricing': ex['roe_10'] - ex['pricing_roe'], 'vs_proposed': ex['roe_10'] - proposed},
        'fapsm': {'2Y': f2, '2Y_dod': f2_dod, '5Y': f5, '5Y_dod': f5 - fapsm(apps, pv, 5), 'all': fa, 'all_dod': fa - fapsm(apps, pv, None)},
        'swaps': {'1Y': sw(rd, '1Y'), '2Y': sw(rd, '2Y'), '5Y': sw(rd, '5Y'), '1Y_prev': sw(pv, '1Y'), '2Y_prev': sw(pv, '2Y'), '5Y_prev': sw(pv, '5Y')},
        'proposed_roe': proposed, 'pricing_roe': ex['pricing_roe'],
    }


def page5(apps, comps, rd, daily_hist, match_excel=None, targets=None, holidays=None):
    """Year to date: cumulative applications (all, PS included) and completions against target, W.A. RoE and FAPSM."""
    match_excel = MATCH_EXCEL if match_excel is None else match_excel
    tg = dict(TARGETS, **(targets or {}))
    rd = pd.Timestamp(rd)
    f0 = fy_start(rd)
    fy_days = bdays(f0, f0 + pd.DateOffset(years=1) - pd.Timedelta(days=1), holidays)
    days = fy_days[fy_days <= rd]
    nd = int(tg.get('fy_working_days') or len(fy_days))
    a = apps[(apps['ApplicationDate'] >= f0) & (apps['ApplicationDate'] <= rd)]
    c = comps[(comps['LoanOriginationDate'] >= f0) & (comps['LoanOriginationDate'] <= rd)]
    app_day = a.groupby('ApplicationDate')['OriginalBalance'].sum()
    comp_day = c.groupby('LoanOriginationDate')['OriginalBalance'].sum()
    cum_a = np.array([app_day[app_day.index <= d].sum() for d in days])
    cum_c = np.array([comp_day[comp_day.index <= d].sum() for d in days])
    apps_target = tg['completions_fy'] / tg['conversion']
    t_a = np.arange(1, len(days) + 1) * apps_target / nd
    t_c = np.arange(1, len(days) + 1) * tg['completions_fy'] / nd
    dh = daily_hist.set_index('Date') if len(daily_hist) else pd.DataFrame()
    roe = np.array([float(dh.at[d, 'RoE']) if (len(dh) and d in dh.index and pd.notna(dh.at[d, 'RoE'])) else np.nan for d in days])
    f2 = np.array([fapsm(apps, d, 2) for d in days])
    f5 = np.array([fapsm(apps, d, 5) for d in days])
    vol_ex = np.array([_sum(a.loc[(a['ApplicationDate'] == d) & (a['Product Switch'] == 'N'), 'OriginalBalance']) for d in days])
    w = cum_a if match_excel else vol_ex
    def cumw(x):
        out = []
        for i in range(len(days)):
            out.append(_wavg(x[:i + 1], w[:i + 1]))
        return np.array(out)
    wa_roe, wa_f2, wa_f5 = cumw(roe), cumw(f2), cumw(f5)
    ytd = float(cum_a[-1]) if len(cum_a) else 0.0
    wa_now = float(wa_roe[-1]) if len(wa_roe) else np.nan
    todo = apps_target - ytd
    need = ((apps_target * tg['roe']) - ytd * wa_now) / todo if todo > 0 and np.isfinite(wa_now) else np.nan
    return {'days': [d.strftime('%Y-%m-%d') for d in days], 'apps_ytd': cum_a.tolist(), 'apps_target': t_a.tolist(),
            'comps_ytd': cum_c.tolist(), 'comps_target': t_c.tolist(), 'roe': roe.tolist(), 'wa_roe': wa_roe.tolist(),
            'fapsm_2y': f2.tolist(), 'fapsm_5y': f5.tolist(), 'wa_fapsm_2y': wa_f2.tolist(), 'wa_fapsm_5y': wa_f5.tolist(),
            'app_ytd': ytd, 'app_wa_roe': wa_now, 'apps_target_fy': apps_target, 'todo': todo, 'need_roe': need,
            'comps_ytd_total': float(cum_c[-1]) if len(cum_c) else 0.0, 'fy_start': f0.strftime('%Y-%m-%d'), 'fy_days': nd,
            'target_roe': tg['roe']}


# ---------------------------------------------------------------------------------------------- 5. history
HIST_DAILY = ['Date', 'RoE', 'RoE_10', 'PricingRoE', 'PricingRoE_TodaySwaps', 'Volume', 'Count', 'Apps_incl_PS', 'Comps',
              'FAPSM_2Y', 'FAPSM_5Y', 'FAPSM_All', 'Swap_1Y', 'Swap_2Y', 'Swap_5Y', 'Source']
HIST_CODES = ['Date', 'Code', 'Run', 'RoE', 'NetReturn', 'NIM', 'Balance', 'Rate', 'Volume']


def empty_history():
    return {'daily': pd.DataFrame(columns=HIST_DAILY), 'codes': pd.DataFrame(columns=HIST_CODES),
            'swaps': pd.DataFrame(columns=['Date', '1Y', '2Y', '5Y'])}


def read_history(path):
    """The history workbook (replaces Month of LLPM Outputs, Historic Return Data and Swap Rates)."""
    x = pd.read_excel(path, sheet_name=None)
    h = empty_history()
    for k, sheet in (('daily', 'Daily'), ('codes', 'Codes'), ('swaps', 'Swaps')):
        if sheet in x:
            df = x[sheet]
            df['Date'] = pd.to_datetime(df['Date'], errors='coerce')
            h[k] = df[df['Date'].notna()].reset_index(drop=True)
    return h


def seed_history_from_workbook(path):
    """Build the history from an existing Daily Dashboard workbook: Historic Return Data (RoE, volume), Swap Rates,
    and Month of LLPM Outputs (per code, per day). The App / PricingSwaps tag in the workbook can be out of line with its
    rows, so the run is worked out from the row order instead: each day's first block of codes is App, the -10 rows are
    App-10, and a code's second plain row that day is PricingSwaps. Totals per code only, no loan rows."""
    import openpyxl
    wb = openpyxl.load_workbook(path, data_only=True, read_only=True)
    h = empty_history()
    rows = [r[:3] for r in wb['Historic Return Data'].iter_rows(min_row=2, values_only=True) if r and r[0]]
    d = pd.DataFrame(rows, columns=['Date', 'RoE', 'Volume'])
    d['Date'] = pd.to_datetime(d['Date'], errors='coerce')
    d['Source'] = 'Excel dashboard'
    h['daily'] = d.reindex(columns=HIST_DAILY)
    sr = [r[:5] for r in wb['Swap Rates'].iter_rows(min_row=2, values_only=True) if r and r[0]]
    s2 = pd.DataFrame([(r[0], r[1]) for r in sr if r[0] is not None], columns=['Date', '2Y'])
    s5 = pd.DataFrame([(r[3], r[4]) for r in sr if len(r) > 4 and r[3] is not None], columns=['Date', '5Y'])
    for s_ in (s2, s5):
        s_['Date'] = pd.to_datetime(s_['Date'], errors='coerce')
    sw = pd.merge(s2.dropna(subset=['Date']).drop_duplicates('Date', keep='last'), s5.dropna(subset=['Date']).drop_duplicates('Date', keep='last'),
                  on='Date', how='outer').sort_values('Date')
    h['swaps'] = sw.reset_index(drop=True)
    ws = wb['Month of LLPM Outputs']
    it = ws.iter_rows(values_only=True)
    hdr = [str(x) for x in next(it)]
    ix = {k: hdr.index(k) for k in ('Row', 'AccountID', 'ApplicationDate', 'RoE', 'NetReturn', 'NIM%', 'CurrentBalance', 'CurrentRate')}
    recs, block, last_row = [], 0, None
    for r in it:
        if not r or r[ix['AccountID']] in (None, ''):
            continue
        dte = pd.Timestamp(r[ix['ApplicationDate']]) if r[ix['ApplicationDate']] else None
        if dte is None:
            continue
        rown = _f(r[ix['Row']], 0)
        if last_row is None or rown <= last_row:            # Row restarts at 1 with each pasted run
            block += 1
        last_row = rown
        recs.append([block, dte, str(r[ix['AccountID']]), r[ix['RoE']], r[ix['NetReturn']], r[ix['NIM%']], r[ix['CurrentBalance']], r[ix['CurrentRate']]])
    wb.close()
    c = pd.DataFrame(recs, columns=['block', 'Date', 'AccountID', 'RoE', 'NetReturn', 'NIM', 'Balance', 'Rate'])
    keep = c.groupby('Date')['block'].transform('max') == c['block']        # a day pasted twice: keep the last run
    c = c[keep].copy()
    c['n'] = c.groupby(['Date', 'AccountID']).cumcount()
    c['Code'] = c['AccountID'].str.replace('-10$', '', regex=True)
    c['Run'] = np.where(c['AccountID'].str.endswith('-10'), 'App-10', np.where(c['n'] == 0, 'App', 'PricingSwaps'))
    c['Volume'] = np.nan
    h['codes'] = c[HIST_CODES].reset_index(drop=True)
    # the last run is still on the LLPM Output tab (the macro copies it across the next morning): add it if missing
    try:
        out, _ = read_llpm_outputs(path)
        d0 = out['ApplicationDate'].dropna()
        if len(d0) and d0.iloc[0].normalize() not in set(h['codes']['Date']):
            o = out.rename(columns={'NIM%': 'NIM', 'CurrentBalance': 'Balance', 'CurrentRate': 'Rate'})
            o['Date'] = d0.iloc[0].normalize()
            o['Volume'] = np.nan
            h['codes'] = pd.concat([h['codes'], o.reindex(columns=HIST_CODES)], ignore_index=True)
    except Exception:
        pass
    return h


def update_history(h, rd, p1, p2rows, p2avg, out, apps, comps):
    """Add (or replace) the reporting date's rows."""
    rd = pd.Timestamp(rd)
    ex = p2avg['excl_ps']
    day = {'Date': rd, 'RoE': ex['roe'], 'RoE_10': ex['roe_10'], 'PricingRoE': ex['pricing_roe'],
           'PricingRoE_TodaySwaps': ex['pricing_roe_today_swaps'], 'Volume': p1['apps']['value'], 'Count': p1['apps']['count'],
           'Apps_incl_PS': p1['apps_ps']['value'], 'Comps': p1['comps']['value'], 'FAPSM_2Y': p1['fapsm']['2Y'],
           'FAPSM_5Y': p1['fapsm']['5Y'], 'FAPSM_All': p1['fapsm']['all'], 'Swap_1Y': p1['swaps']['1Y'] * 100, 'Swap_2Y': p1['swaps']['2Y'] * 100,
           'Swap_5Y': p1['swaps']['5Y'] * 100, 'Source': 'Colab D01 %s' % VERSION}
    dd = h['daily'][pd.to_datetime(h['daily']['Date']) != rd]
    h['daily'] = pd.concat([dd, pd.DataFrame([day])], ignore_index=True).sort_values('Date').reset_index(drop=True)
    vol = p2rows.set_index('code')['volume']
    cr = out[['Code', 'Run', 'RoE', 'NetReturn', 'NIM%', 'CurrentBalance', 'CurrentRate']].rename(columns={'NIM%': 'NIM', 'CurrentBalance': 'Balance', 'CurrentRate': 'Rate'})
    cr.insert(0, 'Date', rd)
    cr['Volume'] = np.where(cr['Run'] == 'App', cr['Code'].map(vol).fillna(0.0), np.nan)
    hc = h['codes'][pd.to_datetime(h['codes']['Date']) != rd]
    h['codes'] = pd.concat([hc, cr], ignore_index=True).sort_values(['Date', 'Run', 'Code']).reset_index(drop=True)
    h = add_swaps(h, rd, {12: p1['swaps']['1Y'] * 100, 24: p1['swaps']['2Y'] * 100, 60: p1['swaps']['5Y'] * 100})
    return h


def save_history(h, path):
    """Write the history workbook (totals only: per day and per code)."""
    with pd.ExcelWriter(path, engine='openpyxl') as xw:
        pd.DataFrame({'Fleet daily dashboard history': [
            'Kept by the D01 notebook. Upload it each run; download the updated copy afterwards.',
            'Daily: one row per reporting date (headline return excluding PS, volumes, FAPSM, swaps).',
            'Codes: one row per date, pricing code and run (App = today\'s applications, App-10 = 10bp lower rate, '
            'PricingSwaps = the pricing loan at today\'s swaps), from the pricing engine.',
            'Swaps: 2-year and 5-year swap rates (%) by date.',
            'Totals per day and per code only; no loan-level data.']}).to_excel(xw, sheet_name='Read me', index=False)
        for k, sheet in (('daily', 'Daily'), ('codes', 'Codes'), ('swaps', 'Swaps')):
            df = h[k].copy()
            df['Date'] = pd.to_datetime(df['Date']).dt.date
            df.to_excel(xw, sheet_name=sheet, index=False)
    return path


def swaps_frame(h):
    s = h['swaps'].copy()
    s['Date'] = pd.to_datetime(s['Date'])
    for c in ('1Y', '2Y', '5Y'):
        if c not in s.columns:
            s[c] = np.nan
    return s.set_index('Date').sort_index()[['1Y', '2Y', '5Y']].apply(pd.to_numeric, errors='coerce')


def add_swaps(h, rd, swaps_pct):
    """Put the day's swaps (in %, keyed by tenor in months: 12, 24, 60) into the history's Swaps sheet."""
    rd = pd.Timestamp(rd)
    row = {'Date': rd}
    for t_, col in ((12, '1Y'), (24, '2Y'), (60, '5Y')):
        v = swaps_pct.get(t_)
        row[col] = float(v) if v not in (None, '') and pd.notna(v) else np.nan
    sw = h['swaps'][pd.to_datetime(h['swaps']['Date']) != rd]
    h['swaps'] = pd.concat([sw, pd.DataFrame([row])], ignore_index=True).sort_values('Date').reset_index(drop=True)
    return h


# ---------------------------------------------------------------------------------------------- 6. commentary
def commentary(p1, p2rows):
    """The Control Panel's example commentary, filled in."""
    a = p1['apps']
    s = p1['swaps']
    d2 = (s['2Y'] - s['2Y_prev']) * 1e4
    d5 = (s['5Y'] - s['5Y_prev']) * 1e4
    upd = lambda x: ('up %.0f bps' % x) if x > 0.5 else (('down %.0f bps' % -x) if x < -0.5 else 'unchanged')
    top = p2rows.sort_values('volume', ascending=False).iloc[0] if len(p2rows) and p2rows['volume'].max() > 0 else None
    lines = ['Morning all. DD for %s against previous working day (%s):' % (p1['date'].strftime('%d/%m/%Y'), p1['prev'].strftime('%d/%m/%Y')),
             'Apps %s by £%.1fm to £%.1fm.' % ('up' if a['dod'] >= 0 else 'down', abs(a['dod']) / 1e6, a['value'] / 1e6),
             '2-year swap %s and 5-year swap %s, returns at %.1f%%.' % (upd(d2), upd(d5), p1['returns']['roe'] * 100)]
    if top is not None:
        lines.append('%s the most popular product with volumes of £%.1fm generating returns of %.1f%%.' % (top['code'], top['volume'] / 1e6, top['roe'] * 100))
    return '\n'.join(lines)


# ---------------------------------------------------------------------------------------------- 7. checks
def checks(apps, comps, rd, cp, out, swaps=None, output_warnings=None):
    rd = pd.Timestamp(rd)
    t = apps[apps['ApplicationDate'] == rd]
    unm = t[t['PricingCode'].isin(['UNMAPPED'])]
    known = set(code_list(cp))
    off = t[~t['PricingCode'].isin(list(known) + ['ND', 'UNMAPPED'])]
    notes = {}
    for code, f in zip(out['Code'], out['Flags'].astype(str) if 'Flags' in out.columns else [''] * len(out)):
        for msg in [m_.strip() for m_ in f.split(';') if m_.strip()]:
            notes.setdefault(re.sub(r'\s*\(.*$', '', msg).replace(' - check Current_Rate / Initial_Margin', ''), set()).add(code)
    prev_ok = swaps is not None and prev_bday(rd) in set(swaps.index)
    o = out.set_index(['Code', 'Run'])
    vol = t.groupby('PricingCode')['OriginalBalance'].sum()
    zero_app = [c for c in code_list(cp) if vol.get(c, 0) > 0 and (c, 'App') in o.index and not (abs(float(o.at[(c, 'App'), 'RoE'])) > 0)]
    zero_ps = [c for c in code_list(cp) if (c, 'PricingSwaps') in o.index and not (abs(float(o.at[(c, 'PricingSwaps'), 'RoE'])) > 0)]
    res = [{'check': 'Applications today with a description the code rule cannot read', 'value': int(len(unm)),
            'detail': '£{:,.0f}'.format(_sum(unm['OriginalBalance'])), 'ok': len(unm) == 0},
           {'check': 'Applications today on codes not in Current Pricing (in totals, not in the product table)', 'value': int(len(off)),
            'detail': ', '.join('%s £%.0fk' % (k, v / 1e3) for k, v in off.groupby('PricingCode')['OriginalBalance'].sum().items()) or '-',
            'ok': True},
           {'check': 'Applications today with no product (ND)', 'value': int((t['PricingCode'] == 'ND').sum()), 'detail': '', 'ok': True},
           {'check': 'Codes in Current Pricing with a rate of 0', 'value': int((cp['products']['rate'] == 0).sum()),
            'detail': ', '.join(cp['products'].loc[cp['products']['rate'] == 0, 'code']) or '-', 'ok': int((cp['products']['rate'] == 0).sum()) == 0},
           {'check': 'Model output matches today\'s inputs', 'value': 'yes' if not output_warnings else 'check',
            'detail': ' '.join(output_warnings or []) or 'same codes, order, date and balances as the input file', 'ok': not output_warnings},
           {'check': 'Codes with applications today but a 0% return from the model', 'value': len(zero_app),
            'detail': ', '.join(zero_app) or '-', 'ok': len(zero_app) == 0},
           {'check': 'Pricing loans with a 0% return at today\'s swaps (the model found no RoE)', 'value': len(zero_ps),
            'detail': ', '.join(zero_ps) or '-', 'ok': len(zero_ps) == 0}]
    if notes:
        res.append({'check': 'Pricing engine notes (codes)', 'value': len(notes), 'detail': ' | '.join('%s: %s' % (k, ', '.join(sorted(v))) for k, v in notes.items()), 'ok': True})
    res += [
           {'check': 'Swap for the previous working day in the history', 'value': 'yes' if prev_ok else 'no',
            'detail': '' if prev_ok else 'the day-on-day swap movement uses the latest earlier date in the history', 'ok': prev_ok},
           {'check': 'Latest application date in the tape', 'value': str(apps['ApplicationDate'].max().date()) if apps['ApplicationDate'].notna().any() else '-',
            'detail': 'reporting date %s' % rd.date(), 'ok': apps['ApplicationDate'].max() >= rd},
           {'check': 'Latest completion date in the tape', 'value': str(comps['LoanOriginationDate'].max().date()) if len(comps) else '-',
            'detail': '', 'ok': True}]
    return pd.DataFrame(res)


# ---------------------------------------------------------------------------------------------- 8. other dates, from history
def out_from_history(hist_codes, day):
    """Engine-output-like rows (Code, Run, RoE, NetReturn, NIM%) for a past date, from the history."""
    h = hist_codes[pd.to_datetime(hist_codes['Date']) == pd.Timestamp(day)]
    return h.rename(columns={'NIM': 'NIM%'})[['Code', 'Run', 'RoE', 'NetReturn', 'NIM%']].drop_duplicates(['Code', 'Run'], keep='last')


def _clean(x):
    """JSON-safe numbers."""
    if isinstance(x, dict):
        return {k: _clean(v) for k, v in x.items()}
    if isinstance(x, (list, tuple)):
        return [_clean(v) for v in x]
    if isinstance(x, (pd.Timestamp, _dt.date, _dt.datetime)):
        return pd.Timestamp(x).strftime('%Y-%m-%d')
    if isinstance(x, (np.integer,)):
        return int(x)
    if isinstance(x, (np.floating, float)):
        return None if not np.isfinite(x) else round(float(x), 8)
    if isinstance(x, (np.bool_,)):
        return bool(x)
    return x


def day_view(cp, apps, comps, day, out, hist, swaps, match_excel=None, holidays=None):
    p2, avg = page2(cp, apps, out, day, hist['codes'], match_excel)
    p1 = page1(cp, apps, comps, day, avg, swaps, match_excel, holidays)
    rows = p2.drop(columns=['pos']).to_dict('records')
    return {'p1': p1, 'p2': {'rows': rows, 'avg': avg}, 'commentary': commentary(p1, p2)}, p1, p2, avg


def dashboard_data(cp, apps, comps, rd, out, hist, swaps, p5, checks_df, meta, back_days=30, match_excel=None, holidays=None):
    """Everything the HTML page shows, as one JSON-ready dict. Today comes from today's engine run; earlier working days
    (up to back_days) are rebuilt from the history of daily runs and the tapes, so the page can step back through them."""
    rd = pd.Timestamp(rd)
    days = {}
    today, p1, p2, avg = day_view(cp, apps, comps, rd, out, hist, swaps, match_excel, holidays)
    days[rd.strftime('%Y-%m-%d')] = today
    have = set(pd.to_datetime(hist['codes'].loc[hist['codes']['Run'] == 'App', 'Date']).dt.normalize()) if len(hist['codes']) else set()
    lo = apps['ApplicationDate'].min()
    for d in reversed(bdays(rd - pd.Timedelta(days=int(back_days * 1.6)), rd - pd.Timedelta(days=1), holidays)[-back_days:]):
        if d not in have or (pd.notna(lo) and d - pd.Timedelta(days=35) < lo):
            continue
        v, *_ = day_view(cp, apps, comps, d, out_from_history(hist['codes'], d), hist, swaps, match_excel, holidays)
        v['from_history'] = True
        days[d.strftime('%Y-%m-%d')] = v
    dh = hist['daily'].copy()
    dh['Date'] = pd.to_datetime(dh['Date'])
    dh = dh.sort_values('Date')
    trend = {'dates': [d.strftime('%Y-%m-%d') for d in dh['Date']], 'roe': dh['RoE'].tolist(), 'volume': dh['Volume'].tolist()}
    # daily volumes by scheme for the last 10 working days (Page 1 grid)
    last = bdays(rd - pd.Timedelta(days=20), rd, holidays)[-10:]
    grid = []
    for d in last:
        t = apps[apps['ApplicationDate'] == d]
        rec = {'date': d.strftime('%Y-%m-%d')}
        for s_ in ('BTL', 'LTD', 'HMO'):
            rec[s_] = _sum(t.loc[(t['Scheme'] == s_) & ~t['Loyalty'], 'OriginalBalance'])
        rec['LOYALTY'] = _sum(t.loc[t['Loyalty'], 'OriginalBalance'])
        rec['Other'] = _sum(t['OriginalBalance']) - rec['BTL'] - rec['LTD'] - rec['HMO'] - rec['LOYALTY']
        grid.append(rec)
    data = {'meta': meta, 'rd': rd.strftime('%Y-%m-%d'), 'dates': sorted(days.keys(), reverse=True), 'days': days,
            'trend': trend, 'ytd': p5, 'grid': grid, 'checks': checks_df.to_dict('records'),
            'targets': {k: v for k, v in TARGETS.items()}}
    return _clean(data)


# ---------------------------------------------------------------------------------------------- 9. swaps for the run
def todays_swaps(rd, typed=None, hist=None, cof=None):
    """Today's swap rates (in %) by tenor in months: typed values first, then a CoF / swap file row for the date,
    then the history's row for the date. Returns ({tenor: rate}, {tenor: where it came from})."""
    rd = pd.Timestamp(rd)
    got, src = {}, {}
    for t_, v in (typed or {}).items():
        if v not in (None, ''):
            got[int(t_)], src[int(t_)] = float(v), 'typed in settings'
    if cof is not None and len(cof):
        c = cof[cof.index == rd]
        for t_, col in ((12, 's1y'), (24, 's2y'), (36, 's3y'), (60, 's5y'), (84, 's7y'), (120, 's10y')):
            if t_ not in got and len(c) and col in c.columns and pd.notna(c[col].iloc[-1]):
                got[t_], src[t_] = float(c[col].iloc[-1]), 'swap file'
    if hist is not None and len(hist['swaps']):
        h = swaps_frame(hist)
        if rd in h.index:
            for t_, col in ((24, '2Y'), (60, '5Y')):
                if t_ not in got and pd.notna(h.at[rd, col]):
                    got[t_], src[t_] = float(h.at[rd, col]), 'history file'
    return got, src


def apply_swaps(A, swaps_pct):
    """Put today's swaps into the LLPM's manual swap curve (3. Rates N35:N43) and switch the overwrite on. Tenors not
    given keep the workbook's saved value; 48, 84 and 180 months are the averages of their neighbours, as in the workbook."""
    ten = list(A['swap_tenors'])
    curve = list(A['swap_curve_manual'] if A.get('swap_overwrite') else A['swap_curve_calc'])
    for t_, v in swaps_pct.items():
        if int(t_) in ten:
            curve[ten.index(int(t_))] = float(v) / 100
    for mid, (a_, b_) in {48: (36, 60), 84: (60, 120), 180: (120, 360)}.items():
        if mid in ten and a_ in ten and b_ in ten and mid not in swaps_pct:
            curve[ten.index(mid)] = (curve[ten.index(a_)] + curve[ten.index(b_)]) / 2
    A['swap_curve_manual'] = curve
    A['swap_overwrite'] = True
    return dict(zip(ten, curve))


def write_current_pricing_template(src, path):
    """Copy the Current Pricing tab (values) and the ERC table out of a Daily Dashboard workbook into
    Fleet_Current_Pricing.xlsx, so the pricing grid can be kept on its own."""
    import openpyxl
    from openpyxl.styles import Font, PatternFill
    wb = openpyxl.load_workbook(src, data_only=True, read_only=True)
    ws = wb['Current Pricing']
    out = openpyxl.Workbook()
    o = out.active
    o.title = 'Current Pricing'
    for r in ws.iter_rows(min_row=1, max_row=70, max_col=51):
        for c in r:
            if getattr(c, 'value', None) is not None and not isinstance(c, openpyxl.cell.read_only.EmptyCell):
                o.cell(row=c.row, column=c.column, value=c.value)
    cp_ = wb['Control Panel']
    e = out.create_sheet('ERCs')
    e.append(['Category', 'Year 1', 'Year 2', 'Year 3', 'Year 4', 'Year 5'])
    for r in cp_.iter_rows(min_row=10, max_row=20, min_col=2, max_col=7, values_only=True):
        if r[0]:
            e.append(list(r))
    rm = out.create_sheet('Read me', 0)
    for line in ['Fleet current pricing: the input the D01 daily dashboard notebook reads.',
                 'Same layout as the Current Pricing tab of Daily_Dashboard_2.1.xlsm (values only). Update it when pricing changes:',
                 '  B:M  current product codes with current and new rate, fee and revert margin (the notebook uses New: F, I, L);',
                 '  Q:Z  loyalty (product switch) codes;',
                 '  AB:AO pricing exercise results: code, type, mixes, swaps used at pricing (AM) and RoE (AO); AQ4 = proposed-mix RoE;',
                 '  AS:AY default loan for each code (balance, flat share, % portfolio landlords, cashback, % freehold) for the',
                 '        "pricing RoE at today\'s swaps" run.',
                 'ERCs: early repayment charges (%%) by category, from the Control Panel. Source: %s.' % os.path.basename(str(src))]:
        rm.append([line])
    rm['A1'].font = Font(bold=True, size=13)
    out.save(path)
    wb.close()
    return path


# ---------------------------------------------------------------------------------------------- 10. two-phase workflow
# Phase 1: the pricing grid as a CSV to paste into the LLPM's input data (the Prodinput / ProdInput - PricingSwaps tabs).
# Phase 2: read the LLPM Outputs tab back in, once the model has been run in Excel.
PRODINPUT_COLUMNS = ['Identifier', 'BorrowerType', 'Occupancy_Type', 'Lien', 'Original_Balance', 'Current_Balance', 'Arrears_Balance',
                     'Months_in_arrears', 'Loan_Status', 'Application_Date', 'Offer_Date', 'Expected_Comp_Date', 'Completion_Date',
                     'Product_Code', 'Product_Scheme', 'Product_Type', 'Current_Rate', 'Initial_Margin', 'Initial_Index', 'Revert_Margin',
                     'Revert_Index', 'Geographic_Region', 'Repayment_Method', 'Loan_Purpose', 'Loan_Term', 'ERC_Year1', 'ERC_Year2',
                     'ERC_Year3', 'ERC_Year4', 'ERC_Year5', 'Current_LTV', 'Property_Type', 'Original_Valuation', 'Valuation_Date',
                     'Capitalised_Fees', 'CustomerValuation_Fee', 'StarlingValuation_Fee', 'LandRegistry_Fees', 'Intermediary_Fees',
                     'MortgageOriginator_Fees', 'CapitalisedApplication_Fee', 'CapitalisedTT_Fee', 'BureauScoreValue', 'Perc_Portfolio',
                     'Cashback ', 'Tenure']


def llpm_input_table(grid):
    """The grid in the Prodinput column layout (A:AT), plus the workbook's two helper columns: AU = pricing code,
    AV = App / PricingSwaps. Rows in the workbook's order: App codes, the -10bp rows, then the PricingSwaps rows."""
    g = grid.rename(columns={'Cashback': 'Cashback '})
    t = pd.DataFrame({c: (g[c] if c in g.columns else None) for c in PRODINPUT_COLUMNS})
    for c in ('Application_Date', 'Valuation_Date'):
        t[c] = pd.to_datetime(t[c]).dt.strftime('%d/%m/%Y')
    t['Pricing_Code'] = grid['Code'].values
    t['Run'] = grid['Run'].values
    return t


def write_llpm_input_csv(grid, path):
    llpm_input_table(grid).to_csv(path, index=False)
    return path


def _match_cols(hdr):
    return {str(h).strip(): i for i, h in enumerate(hdr) if h is not None and str(h).strip()}


def read_llpm_outputs(path, expected=None, rd=None):
    """The LLPM Outputs tab after the model has run. Takes the model workbook itself (.xlsm / .xlsx, the 'Outputs' tab, or a
    'LLPM Output' tab), or a CSV / xlsx saved from that tab. Finds the header row (the one holding AccountID and RoE), reads
    rows until the first blank AccountID, and works out each row's run from its place: a '-10' row is App-10, a code's first
    plain row is App and its second is PricingSwaps (the workbook's own tag column can be out of line with its rows).
    expected: the phase 1 grid, to check the model ran on today's inputs. Returns (outputs, list of warnings)."""
    import openpyxl
    ext = os.path.splitext(str(path))[1].lower()
    if ext == '.csv':
        raw = pd.read_csv(path, header=None, dtype=object).values.tolist()
    else:
        wb = openpyxl.load_workbook(path, data_only=True, read_only=True)
        name = next((n for n in ('Outputs', 'LLPM Output', 'Output') if n in wb.sheetnames), wb.sheetnames[0])
        raw = [list(r) for r in wb[name].iter_rows(max_row=3000, max_col=70, values_only=True)]
        wb.close()
    hrow = next((i for i, r in enumerate(raw[:30]) if r and 'AccountID' in [str(x).strip() for x in r if x is not None]
                 and 'RoE' in [str(x).strip() for x in r if x is not None]), None)
    if hrow is None:
        raise ValueError('No header row with AccountID and RoE in %s: upload the LLPM workbook or its Outputs tab.' % os.path.basename(str(path)))
    ix = _match_cols(raw[hrow])
    rows = []
    for r in raw[hrow + 1:]:
        acc = r[ix['AccountID']] if ix['AccountID'] < len(r) else None
        if acc is None or str(acc).strip() in ('', 'nan', 'None'):
            break
        rows.append(r)
    keep = [k for k in ('Row', 'AccountID', 'PricingCode', 'ApplicationDate', 'CurrentRate', 'CurrentBalance', 'CarryValue', 'RoE',
                        'NetFees', 'SWAPCost', 'GrossEIR', 'PostSwapEIR', 'CPR%', 'COF%', 'NIM%', 'NetReturn', 'ProductFee', 'WAL') if k in ix]
    out = pd.DataFrame([[r[ix[k]] if ix[k] < len(r) else None for k in keep] for r in rows], columns=keep)
    for k in keep:
        if k not in ('AccountID', 'PricingCode', 'ApplicationDate'):
            out[k] = pd.to_numeric(out[k], errors='coerce')
    out['AccountID'] = out['AccountID'].astype(str).str.strip()
    def _d(v):
        if v is None or (isinstance(v, float) and np.isnan(v)):
            return pd.NaT
        if isinstance(v, (int, float, np.integer, np.floating)) or (isinstance(v, str) and re.fullmatch(r'\d{5}(\.0+)?', v.strip())):
            return pd.Timestamp('1899-12-30') + pd.Timedelta(days=float(v))       # Excel serial
        s_ = str(v).strip()
        return pd.to_datetime(s_, errors='coerce', dayfirst=not re.match(r'\d{4}-', s_))
    out['ApplicationDate'] = pd.Series([_d(v) for v in out['ApplicationDate']], index=out.index)
    out['Code'] = out['AccountID'].str.replace('-10$', '', regex=True)
    n = out.groupby('AccountID').cumcount()
    out['Run'] = np.where(out['AccountID'].str.endswith('-10'), 'App-10', np.where(n == 0, 'App', 'PricingSwaps'))
    out['Flags'] = ''
    warn = []
    if rd is not None:
        bad = out['ApplicationDate'].notna() & (out['ApplicationDate'].dt.normalize() != pd.Timestamp(rd))
        if bad.any():
            warn.append('%d rows have an application date other than %s (e.g. %s): the model may not have been run on today\'s inputs.'
                        % (int(bad.sum()), pd.Timestamp(rd).date(), out.loc[bad, 'ApplicationDate'].iloc[0].date()))
    if expected is not None:
        exp_ids = list(expected['Identifier'])
        got = list(out['AccountID'])
        if got[:len(exp_ids)] != exp_ids:
            missing = sorted(set(exp_ids) - set(got))
            extra = sorted(set(got) - set(exp_ids))
            warn.append('The outputs are not the %d rows of today\'s input file in the same order%s%s.'
                        % (len(exp_ids), (' (missing: %s)' % ', '.join(missing[:8])) if missing else '',
                           (' (not in today\'s inputs: %s)' % ', '.join(extra[:8])) if extra else ''))
        elif len(got) > len(exp_ids):
            warn.append('The outputs have %d rows after the %d expected; they are left out.' % (len(got) - len(exp_ids), len(exp_ids)))
            out = out.iloc[:len(exp_ids)].copy()
        if 'CurrentBalance' in out.columns and got[:len(exp_ids)] == exp_ids:
            e = expected.reset_index(drop=True)
            fee = pd.to_numeric(e['Capitalised_Fees'], errors='coerce').fillna(0)
            cb = pd.to_numeric(e['Current_Balance'], errors='coerce').fillna(0)
            gla = cb + np.where(fee < 1, fee * cb, fee)
            off = np.abs(out['CurrentBalance'].values[:len(e)] - gla) > np.maximum(1.0, 1e-6 * gla)
            if off.any():
                warn.append('%d rows have a different balance from today\'s inputs (e.g. %s: model %.0f, inputs %.0f): check the latest '
                            'CSV was pasted in before the model was run.' % (int(off.sum()), e['Identifier'][np.argmax(off)],
                                                                              out['CurrentBalance'].values[np.argmax(off)], gla[np.argmax(off)]))
    return out, warn


# ---------------------------------------------------------------------------------------------- 11. which day to run
def availability_sql(fs):
    """Applications per application date between @start and @end, and how many have no balance yet (the warehouse
    fills balances in during the day, so the latest day can show cases with blank balances)."""
    return """
WITH app AS (
  SELECT a.credit_agreement_mortgage_key AS k, {appd} AS d
  FROM {application} a
  JOIN {mortgage} m ON m.key = a.credit_agreement_mortgage_key
  WHERE m.source_key LIKE 'FLEET~%'
  QUALIFY ROW_NUMBER() OVER (PARTITION BY a.credit_agreement_mortgage_key ORDER BY a.application_date DESC) = 1
),
orig AS (
  SELECT o.credit_agreement_mortgage_key AS k, SAFE_CAST(o.origination_balance AS FLOAT64) AS ob
  FROM {origination} o
  WHERE o.credit_agreement_mortgage_key IN (SELECT k FROM app WHERE d BETWEEN DATE(@start) AND DATE(@end))
  QUALIFY ROW_NUMBER() OVER (PARTITION BY o.credit_agreement_mortgage_key ORDER BY o.date DESC) = 1
)
SELECT app.d AS ApplicationDate, COUNT(*) AS cases,
       SUM(CASE WHEN orig.ob IS NULL OR orig.ob = 0 THEN 1 ELSE 0 END) AS no_balance,
       SUM(IFNULL(orig.ob, 0)) AS balance
FROM app LEFT JOIN orig ON orig.k = app.k
WHERE app.d BETWEEN DATE(@start) AND DATE(@end)
GROUP BY app.d
ORDER BY app.d
""".format(application=fs.t('application'), mortgage=fs.t('mortgage'), origination=fs.t('origination'),
           appd=fs.clean_date('SAFE_CAST(a.application_date AS DATE)'))


def day_choices(avail, today, n=5, holidays=None):
    """The last n working days before today with their cases, £ and share without a balance. 'ready' = cases with
    balances (no more than 10% blank). Returns a DataFrame, latest first."""
    today = pd.Timestamp(today).normalize()
    days = bdays(today - pd.Timedelta(days=n * 2 + 10), today - pd.Timedelta(days=1), holidays)[-n:][::-1]
    a = avail.copy()
    a['ApplicationDate'] = pd.to_datetime(a['ApplicationDate'])
    a = a.set_index('ApplicationDate')
    rows = []
    for i, d in enumerate(days, 1):
        cases = int(a.at[d, 'cases']) if d in a.index else 0
        nb = int(a.at[d, 'no_balance']) if d in a.index else 0
        bal = float(a.at[d, 'balance']) if d in a.index else 0.0
        rows.append({'#': i, 'Date': d, 'Day': d.strftime('%a %d %b'), 'Applications': cases, '£m': round(bal / 1e6, 2),
                     'No balance yet': nb, 'Ready': cases > 0 and nb <= 0.1 * cases})
    return pd.DataFrame(rows)


# ---------------------------------------------------------------------------------------------- 12. downloads
_MIME = {'.csv': 'text/csv', '.html': 'text/html', '.png': 'image/png', '.pdf': 'application/pdf',
         '.xlsx': 'application/vnd.openxmlformats-officedocument.spreadsheetml.sheet'}


def download_links(paths, title='Download'):
    """Download buttons for files the notebook wrote, as links that carry the file inside the page. Unlike
    google.colab.files.download (which can stall in Colab Enterprise), clicking one saves the file straight away.
    Returns an HTML string for IPython.display.HTML."""
    import base64
    btn = ('display:inline-block;margin:4px 8px 4px 0;padding:8px 14px;border-radius:8px;background:#321E37;color:#fff;'
           'font:600 13px sans-serif;text-decoration:none')
    parts = ['<div style="margin:6px 0"><div style="font:600 13px sans-serif;margin-bottom:4px">%s</div>' % title]
    for p in paths:
        name = os.path.basename(p)
        with open(p, 'rb') as f:
            b64 = base64.b64encode(f.read()).decode()
        mime = _MIME.get(os.path.splitext(name)[1].lower(), 'application/octet-stream')
        parts.append('<a download="%s" href="data:%s;base64,%s" style="%s">⬇ %s (%s KB)</a>'
                     % (name, mime, b64, btn, name, '{:,.0f}'.format(os.path.getsize(p) / 1024)))
    parts.append('</div>')
    return ''.join(parts)


# ---------------------------------------------------------------------------------------------- 13. shared folder (Cloud Storage)
# A bucket folder in the bank's Google Cloud keeps the history and inputs between runs and hosts the dashboard, so it can
# be shared as a link (https://storage.cloud.google.com/<bucket>/<path>) that opens for anyone with read access.
def _gcs_split(uri):
    u = str(uri).strip()
    if not u.startswith('gs://'):
        raise ValueError("SHARE_FOLDER should look like 'gs://bucket-name/folder'")
    b, _, p = u[5:].partition('/')
    return b, p.strip('/')


def _gcs_client(project=None):
    from google.cloud import storage
    return storage.Client(project=project) if project else storage.Client()


def gcs_put(local, folder, name=None, project=None, content_type=None):
    """Copy a local file into the shared folder. Returns the browser link."""
    b, p = _gcs_split(folder)
    name = name or os.path.basename(local)
    key = (p + '/' if p else '') + name
    blob = _gcs_client(project).bucket(b).blob(key)
    ct = content_type or _MIME.get(os.path.splitext(name)[1].lower(), 'application/octet-stream')
    if ct == 'text/html':
        ct = 'text/html; charset=utf-8'
        blob.cache_control = 'no-cache'                  # so the link always shows the latest run
    blob.upload_from_filename(local, content_type=ct)
    return 'https://storage.cloud.google.com/%s/%s' % (b, key)


def gcs_get(folder, name, local, project=None):
    """Copy a file from the shared folder to `local` if it is there. Returns True if it was."""
    b, p = _gcs_split(folder)
    key = (p + '/' if p else '') + name
    blob = _gcs_client(project).bucket(b).blob(key)
    if not blob.exists():
        return False
    os.makedirs(os.path.dirname(os.path.abspath(local)) or '.', exist_ok=True)
    blob.download_to_filename(local)
    return True

In [ ]:
# @title
%%writefile fleet_daily_html.py
"""fleet_daily_html: the D01 dashboard as one self-contained HTML file (no internet needed to open it)."""
import json

TEMPLATE = r"""<!doctype html>
<html lang="en">
<head>
<meta charset="utf-8">
<meta name="viewport" content="width=device-width, initial-scale=1">
<title>Fleet Daily Dashboard</title>
<style>
:root{
  --bg:#F8F5F0; --card:#FFFFFF; --ink:#321E37; --ink2:#5E5062; --muted:#8E8590; --grid:#E8E4E9; --axis:#D6D2D7;
  --brand:#321E37; --brand-ink:#FFFFFF; --accent:#50FFEB;
  --s1:#5A6FE0; --s2:#1E9E8F; --s3:#D9731E; --s4:#8E5AA8; --target:#8E8590;
  --up:#1E7B4F; --down:#B4441C; --hi:#DCE3FF; --lo:#FCE3CF; --row:#FBF9F6; --line:#ECE7EE;
}
@media (prefers-color-scheme: dark){
  :root:not([data-theme="light"]){
    --bg:#1A1A19; --card:#242226; --ink:#F3EEF4; --ink2:#C9C0CB; --muted:#9A909C; --grid:#3A363C; --axis:#4A454C;
    --brand:#2B1A30; --brand-ink:#F3EEF4; --s1:#7487F0; --s2:#279E90; --s3:#C9762F; --s4:#AD78CC; --target:#9A909C;
    --up:#5CC48F; --down:#F08A5D; --hi:#263056; --lo:#4A2E18; --row:#2A282C; --line:#38343A;
  }
}
:root[data-theme="dark"]{
  --bg:#1A1A19; --card:#242226; --ink:#F3EEF4; --ink2:#C9C0CB; --muted:#9A909C; --grid:#3A363C; --axis:#4A454C;
  --brand:#2B1A30; --brand-ink:#F3EEF4; --s1:#7487F0; --s2:#279E90; --s3:#C9762F; --s4:#AD78CC; --target:#9A909C;
  --up:#5CC48F; --down:#F08A5D; --hi:#263056; --lo:#4A2E18; --row:#2A282C; --line:#38343A;
}
*{box-sizing:border-box}
body{margin:0;background:var(--bg);color:var(--ink);font:14px/1.45 "Inter Tight","Inter","Segoe UI",system-ui,-apple-system,sans-serif}
.num,td.n,.tile .v{font-variant-numeric:tabular-nums}
header{background:var(--brand);color:var(--brand-ink);padding:18px 24px 0}
header .top{display:flex;flex-wrap:wrap;gap:12px 24px;align-items:center;justify-content:space-between}
header h1{font-size:22px;margin:0;font-weight:700;letter-spacing:-.01em}
header h1 span{color:var(--accent);font-weight:600}
header .sub{font-size:12.5px;opacity:.8;margin-top:2px}
.ctl{display:flex;gap:10px;align-items:center;flex-wrap:wrap}
.ctl label{font-size:12px;opacity:.85}
select,button{font:inherit;font-size:13px;border-radius:8px;border:1px solid rgba(255,255,255,.35);background:rgba(255,255,255,.08);color:inherit;padding:6px 10px}
select option{color:#321E37}
button{cursor:pointer}
nav{display:flex;gap:4px;margin-top:14px;overflow-x:auto}
nav button{border:0;border-radius:10px 10px 0 0;background:transparent;padding:10px 16px;font-weight:600;opacity:.75}
nav button.on{background:var(--bg);color:var(--ink);opacity:1}
main{padding:20px 24px 40px;max-width:1400px;margin:0 auto}
section{display:none}
section.on{display:block}
h2{font-size:15px;margin:22px 0 10px;font-weight:700}
h2:first-child{margin-top:4px}
.grid{display:grid;gap:14px}
.g4{grid-template-columns:repeat(auto-fit,minmax(230px,1fr))}
.g2{grid-template-columns:repeat(2,minmax(0,1fr))}
.card{background:var(--card);border-radius:14px;padding:14px 16px;box-shadow:0 1px 0 var(--line)}
.tile .k{font-size:12px;color:var(--ink2);font-weight:600;text-transform:none}
.tile .v{font-size:28px;font-weight:700;margin:2px 0 4px;letter-spacing:-.01em}
.tile .d{font-size:12.5px;color:var(--ink2);display:flex;gap:4px 12px;flex-wrap:wrap}
.pair{white-space:nowrap}
.chg{font-weight:600;white-space:nowrap}
.chg.up{color:var(--up)} .chg.down{color:var(--down)} .chg.flat{color:var(--muted)}
.mini{display:grid;grid-template-columns:auto 1fr 1fr;gap:4px 14px;font-size:13px;margin-top:4px}
.mini .h{color:var(--muted);font-size:11.5px}
.mini .r{text-align:right}
.note{font-size:12px;color:var(--muted)}
.chart{position:relative}
.chart svg{display:block;width:100%;height:auto;overflow:visible}
.chart .title{font-weight:700;font-size:13.5px;margin-bottom:2px}
.legend{display:flex;gap:14px;flex-wrap:wrap;font-size:12px;color:var(--ink2);margin:2px 0 6px}
.legend i{display:inline-block;width:14px;height:0;border-top:2px solid;vertical-align:middle;margin-right:5px}
.legend i.bar{height:10px;border:0;border-radius:2px}
.legend i.dash{border-top-style:dashed}
.tip{position:absolute;pointer-events:none;background:var(--card);border:1px solid var(--axis);border-radius:8px;padding:7px 9px;font-size:12px;box-shadow:0 4px 14px rgba(0,0,0,.12);display:none;min-width:150px;z-index:5}
.tip .t{color:var(--muted);margin-bottom:3px}
.tip .row{display:flex;justify-content:space-between;gap:12px}
.tip .row b{font-variant-numeric:tabular-nums}
.tip .row span i{display:inline-block;width:10px;border-top:2px solid;vertical-align:middle;margin-right:5px}
.seg{display:inline-flex;border:1px solid var(--axis);border-radius:8px;overflow:hidden}
.seg button{border:0;border-radius:0;background:transparent;color:var(--ink2);padding:5px 10px;font-size:12.5px}
.seg button.on{background:var(--ink);color:var(--card)}
.filters{display:flex;gap:14px;align-items:center;flex-wrap:wrap;margin:6px 0 4px}
table{border-collapse:collapse;width:100%;font-size:13px}
th{font-size:11.5px;color:var(--ink2);text-align:right;font-weight:600;padding:8px 6px;border-bottom:1px solid var(--axis);cursor:pointer;user-select:none;white-space:nowrap;vertical-align:bottom}
th:first-child,td:first-child{text-align:left}
td{padding:6px 6px;border-bottom:1px solid var(--line);text-align:right;white-space:nowrap}
tr.avg td{font-weight:700;border-top:1px solid var(--axis);background:var(--row)}
tr.loy td:first-child{color:var(--ink2)}
td.hi{background:var(--hi)} td.lo{background:var(--lo)}
td.z{color:var(--muted)}
th .grp{display:block;color:var(--muted);font-weight:500}
.tablewrap{overflow-x:auto}
.copy{float:right;border:1px solid var(--axis);background:transparent;color:var(--ink);border-radius:8px;padding:4px 10px;font-size:12px}
pre.comm{white-space:pre-wrap;font:inherit;margin:6px 0 0}
.badge{display:inline-block;font-size:11px;font-weight:700;border-radius:20px;padding:2px 8px;background:var(--lo);color:var(--ink)}
.ok{color:var(--up);font-weight:700} .bad{color:var(--down);font-weight:700}
details summary{cursor:pointer;color:var(--ink2);font-size:12.5px;margin-top:6px}
.kv{display:grid;grid-template-columns:minmax(160px,auto) 1fr;gap:4px 16px;font-size:13px}
.kv div:nth-child(odd){color:var(--ink2)}
@media (max-width:640px){main{padding:14px 16px 30px} header{padding:14px 16px 0} .g2{grid-template-columns:1fr} .tile .v{font-size:24px}}
</style>
</head>
<body>
<header>
  <div class="top">
    <div><h1>Fleet Daily Dashboard <span id="hdate"></span></h1><div class="sub" id="hsub"></div></div>
    <div class="ctl">
      <label for="dsel">Reporting date</label><select id="dsel"></select>
      <button id="theme" title="Switch light / dark">◐</button>
    </div>
  </div>
  <nav id="tabs">
    <button data-t="ov" class="on">Overview</button><button data-t="pr">Products</button><button data-t="ytd">Year to date</button><button data-t="nt">Checks &amp; notes</button>
  </nav>
</header>
<main>
<section id="ov" class="on">
  <div id="histnote" class="note"></div>
  <h2>Volume</h2>
  <div class="grid g4" id="t_vol"></div>
  <h2>Daily returns <span class="note" id="ret_note"></span></h2>
  <div class="grid g4" id="t_ret"></div>
  <h2>Margins and swaps</h2>
  <div class="grid g4" id="t_mar"></div>
  <h2>Trends</h2>
  <div class="filters"><div class="seg" id="rng"><button data-w="20">4 weeks</button><button data-w="60" class="on">12 weeks</button><button data-w="130">26 weeks</button><button data-w="fy">Financial year</button></div></div>
  <div class="grid g2">
    <div class="card chart" id="c_vol"></div>
    <div class="card chart" id="c_roe"></div>
  </div>
  <h2>Applications by scheme, last 10 working days</h2>
  <div class="card tablewrap" id="t_grid"></div>
  <h2>Commentary draft</h2>
  <div class="card"><button class="copy" id="copyc">Copy</button><pre class="comm" id="comm"></pre></div>
</section>
<section id="pr">
  <div class="filters">
    <div class="seg" id="pfilt"><button data-f="all" class="on">All codes</button><button data-f="live">With volume in last 30 days</button><button data-f="nl">Excluding loyalty</button></div>
    <span class="note">Click a column to sort. Shading: return above (blue) or below (orange) the <span id="tg1"></span> target.</span>
  </div>
  <div class="card tablewrap" id="ptab"></div>
  <div class="note" id="pnote" style="margin-top:8px"></div>
</section>
<section id="ytd">
  <div class="grid g4" id="t_ytd"></div>
  <div class="grid g2" style="margin-top:14px">
    <div class="card chart" id="c_ya"></div>
    <div class="card chart" id="c_yr"></div>
    <div class="card chart" id="c_yc"></div>
    <div class="card chart" id="c_yf"></div>
  </div>
  <div class="note" id="ynote" style="margin-top:8px"></div>
</section>
<section id="nt">
  <h2>Checks</h2>
  <div class="card tablewrap" id="chk"></div>
  <h2>How this run was made</h2>
  <div class="card kv" id="meta"></div>
  <h2>Where the definitions follow the Excel dashboard</h2>
  <div class="card" id="xl"></div>
</section>
</main>
<script>
const DATA = __DATA__;
const $ = id => document.getElementById(id);
const el = (tag, cls, txt) => { const e = document.createElement(tag); if (cls) e.className = cls; if (txt != null) e.textContent = txt; return e; };
const isNum = v => typeof v === 'number' && isFinite(v);
const fm = (v, d=1) => isNum(v) ? '£' + (v/1e6).toLocaleString('en-GB', {minimumFractionDigits: d, maximumFractionDigits: d}) + 'm' : '–';
const fk = v => isNum(v) ? '£' + Math.round(v).toLocaleString('en-GB') : '–';
const fp = (v, d=1) => isNum(v) ? (v*100).toFixed(d) + '%' : '–';
const fpts = (v, d=1) => isNum(v) ? (v>=0?'+':'−') + Math.abs(v*100).toFixed(d) + ' pts' : '–';
const fbp = v => isNum(v) ? (v>=0?'+':'−') + Math.abs(v*1e4).toFixed(0) + ' bp' : '–';
const fdate = s => { const d = new Date(s + 'T00:00:00'); return d.toLocaleDateString('en-GB', {day:'numeric', month:'short', year:'numeric'}); };
const fdshort = s => { const d = new Date(s + 'T00:00:00'); return d.toLocaleDateString('en-GB', {day:'numeric', month:'short'}); };
function chg(v, txt, good='up') {
  const s = el('span', 'chg ' + (!isNum(v) || Math.abs(v) < 1e-12 ? 'flat' : ((v > 0) === (good === 'up') ? 'up' : 'down')));
  s.textContent = (isNum(v) && v > 1e-12 ? '▲ ' : (isNum(v) && v < -1e-12 ? '▼ ' : '')) + txt;
  return s;
}
let DAY = DATA.rd;
const css = n => getComputedStyle(document.documentElement).getPropertyValue(n).trim();

/* ---------- header, tabs, theme ---------- */
function header() {
  $('hdate').textContent = fdate(DAY);
  const v = DATA.days[DAY];
  $('hsub').textContent = 'Applications on ' + fdate(DAY) + ' against ' + fdate(v.p1.prev) + ' · built ' + DATA.meta.generated + ' · ' + DATA.meta.engine;
}
DATA.dates.forEach(d => { const o = el('option', null, fdate(d) + (DATA.days[d].from_history ? ' (from history)' : '')); o.value = d; $('dsel').appendChild(o); });
$('dsel').onchange = e => { DAY = e.target.value; renderDay(); };
document.querySelectorAll('#tabs button').forEach(b => b.onclick = () => {
  document.querySelectorAll('#tabs button').forEach(x => x.classList.toggle('on', x === b));
  document.querySelectorAll('main section').forEach(s => s.classList.toggle('on', s.id === b.dataset.t));
  if (b.dataset.t === 'ytd') renderYTD();
  if (b.dataset.t === 'ov') renderTrends();
});
$('theme').onclick = () => {
  const r = document.documentElement, dark = r.dataset.theme ? r.dataset.theme === 'dark' : matchMedia('(prefers-color-scheme: dark)').matches;
  r.dataset.theme = dark ? 'light' : 'dark'; renderAll();
};

/* ---------- tiles ---------- */
function tile(k, v, rows) {
  const c = el('div', 'card tile'); c.appendChild(el('div', 'k', k)); c.appendChild(el('div', 'v', v));
  const d = el('div', 'd'); rows.forEach(r => d.appendChild(r)); c.appendChild(d); return c;
}
const lab = t => el('span', null, t);
const pair = (t, node) => { const s = el('span', 'pair'); s.appendChild(document.createTextNode(t + ' ')); s.appendChild(node); return s; };
function renderTiles() {
  const p = DATA.days[DAY].p1;
  const tv = $('t_vol'); tv.replaceChildren();
  [['Applications (excluding product switches)', p.apps], ['Applications including product switches', p.apps_ps], ['Completions', p.comps]].forEach(([k, x]) => {
    tv.appendChild(tile(k, fm(x.value), [chg(x.dod, fm(Math.abs(x.dod)) + ' on day'), lab(x.count + ' cases'), chg(x.count_dod, Math.abs(x.count_dod) + ' on day')]));
  });
  const tr = $('t_ret'); tr.replaceChildren();
  $('ret_note').textContent = '(excluding product switches, weighted by today’s applications)';
  tr.appendChild(tile('Return on equity today', fp(p.returns.roe), [pair('vs pricing RoEs', chg(p.returns.vs_pricing, fpts(p.returns.vs_pricing))), pair('vs proposed pricing', chg(p.returns.vs_proposed, fpts(p.returns.vs_proposed)))]));
  tr.appendChild(tile('With 10bp off the margin', fp(isNum(p.returns.roe) && isNum(p.returns_10.delta) ? p.returns.roe + p.returns_10.delta : null), [pair('change', chg(p.returns_10.delta, fpts(p.returns_10.delta))), pair('vs pricing', chg(p.returns_10.vs_pricing, fpts(p.returns_10.vs_pricing))), pair('vs proposed', chg(p.returns_10.vs_proposed, fpts(p.returns_10.vs_proposed)))]));
  tr.appendChild(tile('Pricing RoE at today’s mix', fp(p.pricing_roe), [lab('proposed pricing ' + fp(p.proposed_roe)), lab('target ' + fp(DATA.targets.roe, 0))]));
  const tm = $('t_mar'); tm.replaceChildren();
  const f = p.fapsm;
  tm.appendChild(tile('Fee-adjusted margin over swap, 2-year', fp(f['2Y'], 2), [chg(f['2Y_dod'], fpts(f['2Y_dod'], 2) + ' on day'), lab('trackers excluded')]));
  tm.appendChild(tile('Fee-adjusted margin over swap, 5-year', fp(f['5Y'], 2), [chg(f['5Y_dod'], fpts(f['5Y_dod'], 2) + ' on day')]));
  tm.appendChild(tile('Fee-adjusted margin over swap, all', fp(f.all, 2), [chg(f.all_dod, fpts(f.all_dod, 2) + ' on day'), lab('2- and 5-year, trackers included')]));
  const s = p.swaps, c = el('div', 'card tile'); c.appendChild(el('div', 'k', 'Swaps'));
  const ten = (isNum(s['1Y']) ? ['1Y'] : []).concat(['2Y', '5Y']);
  const g = el('div', 'mini'); g.style.gridTemplateColumns = 'auto' + ' 1fr'.repeat(ten.length);
  [''].concat(ten.map(t => t[0] + '-year')).forEach(h => g.appendChild(el('div', 'h' + (h ? ' r' : ''), h)));
  [['Today', ''], ['Previous day', '_prev']].forEach(([k, sfx]) => { g.appendChild(el('div', null, k)); ten.forEach(t => g.appendChild(el('div', 'r num', fp(s[t + sfx], 2)))); });
  g.appendChild(el('div', null, 'Movement'));
  ten.forEach(t => { const m = isNum(s[t]) && isNum(s[t + '_prev']) ? s[t] - s[t + '_prev'] : null; const d = el('div', 'r'); d.appendChild(chg(m, fbp(m), 'down')); g.appendChild(d); });
  c.appendChild(g); tm.appendChild(c);
  $('histnote').textContent = DATA.days[DAY].from_history ? 'This date is rebuilt from the saved history of daily runs and today’s tapes; volumes reflect the tapes as they are now.' : '';
}

/* ---------- SVG chart with crosshair tooltip ---------- */
function chart(host, cfg) {
  host.replaceChildren();
  host.appendChild(el('div', 'title', cfg.title));
  const lg = el('div', 'legend');
  cfg.series.forEach(s => { const sp = el('span'); const i = el('i', s.kind === 'bar' ? 'bar' : (s.dash ? 'dash' : '')); if (s.kind === 'bar') i.style.background = s.color; else i.style.borderColor = s.color; sp.appendChild(i); sp.appendChild(document.createTextNode(s.name)); lg.appendChild(sp); });
  host.appendChild(lg);
  const W = 640, H = 260, m = {l: 54, r: 14, t: 10, b: 30};
  const ns = 'http://www.w3.org/2000/svg';
  const svg = document.createElementNS(ns, 'svg'); svg.setAttribute('viewBox', `0 0 ${W} ${H}`); svg.setAttribute('role', 'img'); svg.setAttribute('aria-label', cfg.title);
  const x = cfg.x, n = x.length;
  let vals = []; cfg.series.forEach(s => s.values.forEach(v => { if (isNum(v)) vals.push(v); }));
  let lo = cfg.ymin != null ? cfg.ymin : Math.min(0, ...vals), hi = Math.max(...vals, lo + 1e-9);
  if (cfg.ymin == null && cfg.unit === '%') { lo = Math.min(...vals); const pad = (hi - lo) * 0.15 || 0.01; lo -= pad; hi += pad; }
  const step = nice((hi - lo) / 4); lo = Math.floor(lo / step) * step; hi = Math.ceil(hi / step) * step;
  const X = i => m.l + (n <= 1 ? (W - m.l - m.r) / 2 : i * (W - m.l - m.r) / (n - 1)) ;
  const bw = Math.max(2, Math.min(18, (W - m.l - m.r) / Math.max(n, 1) * 0.62));
  const Xb = i => m.l + bw / 2 + 2 + i * ((W - m.l - m.r - bw - 4) / Math.max(n - 1, 1));
  const hasBar = cfg.series.some(s => s.kind === 'bar');
  const XX = hasBar ? Xb : X;
  const Y = v => m.t + (hi - v) / (hi - lo) * (H - m.t - m.b);
  const fmt = cfg.fmt;
  for (let v = lo; v <= hi + step / 2; v += step) {
    const g = document.createElementNS(ns, 'line'); g.setAttribute('x1', m.l); g.setAttribute('x2', W - m.r); g.setAttribute('y1', Y(v)); g.setAttribute('y2', Y(v)); g.setAttribute('stroke', css('--grid')); svg.appendChild(g);
    const t = document.createElementNS(ns, 'text'); t.setAttribute('x', m.l - 6); t.setAttribute('y', Y(v) + 4); t.setAttribute('text-anchor', 'end'); t.setAttribute('font-size', '11'); t.setAttribute('fill', css('--muted')); t.textContent = fmt(v, step); svg.appendChild(t);
  }
  const ticks = Math.min(6, n); for (let k = 0; k < ticks; k++) {
    const i = Math.round(k * (n - 1) / Math.max(ticks - 1, 1)); const t = document.createElementNS(ns, 'text');
    t.setAttribute('x', XX(i)); t.setAttribute('y', H - 8); t.setAttribute('text-anchor', k === 0 ? 'start' : (k === ticks - 1 ? 'end' : 'middle')); t.setAttribute('font-size', '11'); t.setAttribute('fill', css('--muted')); t.textContent = fdshort(x[i]); svg.appendChild(t);
  }
  cfg.series.forEach(s => {
    if (s.kind === 'bar') {
      s.values.forEach((v, i) => { if (!isNum(v)) return; const r = document.createElementNS(ns, 'rect'); const y0 = Y(Math.max(lo, 0)), y1 = Y(v);
        r.setAttribute('x', XX(i) - bw / 2); r.setAttribute('width', bw); r.setAttribute('y', Math.min(y0, y1)); r.setAttribute('height', Math.max(1, Math.abs(y0 - y1))); r.setAttribute('rx', Math.min(3, bw / 3)); r.setAttribute('fill', s.color); svg.appendChild(r); });
    } else {
      let d = ''; s.values.forEach((v, i) => { if (!isNum(v)) return; d += (d && isNum(s.values[i - 1]) ? 'L' : 'M') + XX(i).toFixed(1) + ',' + Y(v).toFixed(1); });
      const p = document.createElementNS(ns, 'path'); p.setAttribute('d', d); p.setAttribute('fill', 'none'); p.setAttribute('stroke', s.color); p.setAttribute('stroke-width', s.width || 2); if (s.dash) p.setAttribute('stroke-dasharray', '5 4'); p.setAttribute('stroke-linejoin', 'round'); svg.appendChild(p);
      const li = s.values.map((v, i) => isNum(v) ? i : -1).filter(i => i >= 0).pop();
      if (li != null && li >= 0 && !s.dash) { const c = document.createElementNS(ns, 'circle'); c.setAttribute('cx', XX(li)); c.setAttribute('cy', Y(s.values[li])); c.setAttribute('r', 4); c.setAttribute('fill', s.color); c.setAttribute('stroke', css('--card')); c.setAttribute('stroke-width', 2); svg.appendChild(c); }
    }
  });
  const hair = document.createElementNS(ns, 'line'); hair.setAttribute('y1', m.t); hair.setAttribute('y2', H - m.b); hair.setAttribute('stroke', css('--axis')); hair.style.display = 'none'; svg.appendChild(hair);
  host.appendChild(svg);
  const tip = el('div', 'tip'); host.appendChild(tip);
  svg.addEventListener('pointermove', ev => {
    const b = svg.getBoundingClientRect(); const px = (ev.clientX - b.left) / b.width * W;
    let i = 0, best = 1e9; for (let k = 0; k < n; k++) { const dd = Math.abs(XX(k) - px); if (dd < best) { best = dd; i = k; } }
    hair.setAttribute('x1', XX(i)); hair.setAttribute('x2', XX(i)); hair.style.display = '';
    tip.replaceChildren(el('div', 't', fdate(x[i])));
    cfg.series.forEach(s => { const r = el('div', 'row'); const nm = el('span'); const k = el('i'); k.style.borderColor = s.color; if (s.dash) k.style.borderTopStyle = 'dashed'; nm.appendChild(k); nm.appendChild(document.createTextNode(s.name)); r.appendChild(nm); r.appendChild(el('b', null, isNum(s.values[i]) ? cfg.tipfmt(s.values[i]) : '–')); tip.appendChild(r); });
    tip.style.display = 'block'; const hb = host.getBoundingClientRect();
    let left = ev.clientX - hb.left + 14; if (left > hb.width - 190) left = ev.clientX - hb.left - 190; tip.style.left = left + 'px'; tip.style.top = (ev.clientY - hb.top - 10) + 'px';
  });
  svg.addEventListener('pointerleave', () => { tip.style.display = 'none'; hair.style.display = 'none'; });
  if (cfg.note) host.appendChild(el('div', 'note', cfg.note));
  const det = el('details'); det.appendChild(el('summary', null, 'Show the numbers'));
  const tb = el('table'); const hr = el('tr'); hr.appendChild(el('th', null, 'Date')); cfg.series.forEach(s => hr.appendChild(el('th', null, s.name))); tb.appendChild(hr);
  for (let i = n - 1; i >= 0; i--) { const r = el('tr'); r.appendChild(el('td', null, fdate(x[i]))); cfg.series.forEach(s => r.appendChild(el('td', 'n', isNum(s.values[i]) ? cfg.tipfmt(s.values[i]) : '–'))); tb.appendChild(r); }
  const wrap = el('div', 'tablewrap'); wrap.style.maxHeight = '260px'; wrap.style.overflowY = 'auto'; wrap.appendChild(tb); det.appendChild(wrap); host.appendChild(det);
}
function nice(x) { const p = Math.pow(10, Math.floor(Math.log10(x || 1))); const f = x / p; return (f <= 1 ? 1 : f <= 2 ? 2 : f <= 2.5 ? 2.5 : f <= 5 ? 5 : 10) * p; }
function trendline(v) { const pts = v.map((y, i) => [i, y]).filter(p => isNum(p[1])); const k = pts.length; if (k < 2) return v.map(() => null);
  const mx = pts.reduce((a, p) => a + p[0], 0) / k, my = pts.reduce((a, p) => a + p[1], 0) / k; let sxy = 0, sxx = 0; pts.forEach(p => { sxy += (p[0] - mx) * (p[1] - my); sxx += (p[0] - mx) ** 2; });
  const b = sxx ? sxy / sxx : 0; return v.map((_, i) => my + b * (i - mx)); }

/* ---------- trends ---------- */
let WIN = '60';
document.querySelectorAll('#rng button').forEach(b => b.onclick = () => { document.querySelectorAll('#rng button').forEach(x => x.classList.toggle('on', x === b)); WIN = b.dataset.w; renderTrends(); });
function renderTrends() {
  const T = DATA.trend; let idx = T.dates.map((d, i) => i).filter(i => T.dates[i] <= DAY);
  if (WIN === 'fy') { const fy = DATA.ytd.fy_start; idx = idx.filter(i => T.dates[i] >= fy); } else idx = idx.slice(-parseInt(WIN));
  const x = idx.map(i => T.dates[i]), vol = idx.map(i => T.volume[i]), roe = idx.map(i => T.roe[i]);
  chart($('c_vol'), {title: 'Daily applications, excluding product switches', x, unit: '£', ymin: 0,
    series: [{name: 'Applications', kind: 'bar', values: vol, color: css('--s1')}, {name: 'Target ' + fm(DATA.targets.daily_volume, 0) + ' a day', values: vol.map(() => DATA.targets.daily_volume), color: css('--target'), dash: true, width: 1.5}, {name: 'Linear trend', values: trendline(vol), color: css('--s3'), dash: true, width: 1.5}],
    fmt: v => '£' + (v / 1e6).toFixed(v % 1e6 ? 1 : 0) + 'm', tipfmt: v => fm(v, 2)});
  chart($('c_roe'), {title: 'Daily return on equity, excluding product switches', x, unit: '%',
    series: [{name: 'RoE', values: roe, color: css('--s2')}, {name: 'Target ' + fp(DATA.targets.roe, 0), values: roe.map(() => DATA.targets.roe), color: css('--target'), dash: true, width: 1.5}, {name: 'Linear trend', values: trendline(roe), color: css('--s3'), dash: true, width: 1.5}],
    fmt: (v, s) => (v * 100).toFixed(s < 0.01 ? 1 : 0) + '%', tipfmt: v => fp(v, 1)});
}

/* ---------- scheme grid ---------- */
function renderGrid() {
  const t = el('table'); const h = el('tr'); ['Date', 'BTL', 'LTD', 'HMO', 'Loyalty', 'Other', 'Total'].forEach(k => h.appendChild(el('th', null, k))); t.appendChild(h);
  DATA.grid.filter(r => r.date <= DAY).reverse().forEach(r => { const tr = el('tr'); tr.appendChild(el('td', null, fdate(r.date))); let tot = 0;
    ['BTL', 'LTD', 'HMO', 'LOYALTY', 'Other'].forEach(k => { tot += r[k] || 0; tr.appendChild(el('td', 'n' + (r[k] ? '' : ' z'), r[k] ? fm(r[k], 2) : '–')); });
    tr.appendChild(el('td', 'n', fm(tot, 2))); t.appendChild(tr); });
  $('t_grid').replaceChildren(t);
}

/* ---------- products table ---------- */
const PCOLS = [
  ['code', 'Pricing code', null, 's'],
  ['pricing_roe', 'Pricing RoE', 'At pricing', 'p'], ['pricing_roe_today_swaps', 'At today’s swaps', 'At pricing', 'p'],
  ['volume', 'Volume', 'Today', 'm'], ['roe', 'Return', 'Today', 'p'], ['roe_10', 'Return −10bp', 'Today', 'p'], ['contribution', 'Contribution', 'Today', 'k'],
  ['wk_volume', 'Volume', 'Last 7 days', 'm'], ['wk_roe', 'W.A. return', 'Last 7 days', 'p'],
  ['mth_volume', 'Volume', 'Last 30 days', 'm'], ['mth_roe', 'W.A. return', 'Last 30 days', 'p'], ['mth_contribution', 'Contribution', 'Last 30 days', 'k']];
let PSORT = null, PDIR = -1, PF = 'all';
document.querySelectorAll('#pfilt button').forEach(b => b.onclick = () => { document.querySelectorAll('#pfilt button').forEach(x => x.classList.toggle('on', x === b)); PF = b.dataset.f; renderProducts(); });
function cellfmt(k, v) { return k === 'p' ? (isNum(v) && v !== 0 ? fp(v, 1) : '–') : k === 'm' ? (v ? fk(v) : '–') : k === 'k' ? (v ? fk(v) : '–') : String(v); }
function renderProducts() {
  const v = DATA.days[DAY].p2; let rows = v.rows.slice();
  if (PF === 'live') rows = rows.filter(r => r.mth_volume > 0); if (PF === 'nl') rows = rows.filter(r => !r.loyalty);
  if (PSORT) rows.sort((a, b) => { const x = a[PSORT], y = b[PSORT]; return (typeof x === 'string' ? x.localeCompare(y) : ((isNum(x) ? x : -1e18) - (isNum(y) ? y : -1e18))) * PDIR; });
  const t = el('table'); const h = el('tr');
  PCOLS.forEach(([k, lbl, grp]) => { const th = el('th'); if (grp) th.appendChild(el('span', 'grp', grp)); th.appendChild(document.createTextNode(lbl + (PSORT === k ? (PDIR > 0 ? ' ▲' : ' ▼') : ''))); th.onclick = () => { PDIR = PSORT === k ? -PDIR : -1; PSORT = k; renderProducts(); }; h.appendChild(th); });
  t.appendChild(h);
  const tg = DATA.targets.roe;
  rows.forEach(r => { const tr = el('tr', r.loyalty ? 'loy' : null);
    PCOLS.forEach(([k, , , f]) => { const val = r[k]; const td = el('td', f === 's' ? null : 'n', cellfmt(f, val));
      if (f === 'p' && ['roe', 'roe_10', 'wk_roe', 'mth_roe', 'pricing_roe_today_swaps'].includes(k) && isNum(val) && val !== 0) td.classList.add(val >= tg ? 'hi' : 'lo');
      if ((f !== 's') && (!isNum(val) || val === 0)) td.classList.add('z');
      if (k === 'wk_roe' || k === 'mth_roe') { const cov = r[k.replace('roe', 'covered')]; if (isNum(cov) && cov < 0.999) td.title = Math.round(cov * 100) + '% of the volume has a saved daily return'; }
      tr.appendChild(td); });
    t.appendChild(tr); });
  [['Average excluding product switches', v.avg.excl_ps], ['Average, all codes', v.avg.total]].forEach(([nm, a]) => { const tr = el('tr', 'avg');
    PCOLS.forEach(([k, , , f]) => { let val = k === 'code' ? nm : a[k]; tr.appendChild(el('td', f === 's' ? null : 'n', k === 'code' ? nm : (val == null ? '' : cellfmt(f, val)))); });
    t.appendChild(tr); });
  $('ptab').replaceChildren(t);
  $('tg1').textContent = fp(tg, 0);
  $('pnote').textContent = 'Contribution = volume × net return a year. Returns are £-weighted by today’s applications. "At today’s swaps" re-runs each code’s pricing loan with today’s swaps. Weekly and month-to-date returns weight each day’s saved return by that day’s applications. '
    + 'The "excluding product switches" average covers every code except the loyalty codes (' + v.avg.excl_ps_codes.length + ' codes).';
}

/* ---------- year to date ---------- */
function renderYTD() {
  const y = DATA.ytd, n = y.days.length; const t = $('t_ytd'); t.replaceChildren();
  t.appendChild(tile('Applications, year to date', fm(y.app_ytd), [lab('target to date ' + fm(y.apps_target[n - 1])), chg(y.app_ytd - y.apps_target[n - 1], fm(Math.abs(y.app_ytd - y.apps_target[n - 1])) + ' vs target')]));
  t.appendChild(tile('Weighted RoE, year to date', fp(y.app_wa_roe, 1), [lab('target ' + fp(y.target_roe, 0))]));
  t.appendChild(tile('Applications still needed this year', fm(y.todo), [lab('at ' + fp(y.need_roe, 2) + ' RoE to reach ' + fp(y.target_roe, 0) + ' for the year')]));
  t.appendChild(tile('Completions, year to date', fm(y.comps_ytd_total), [lab('target to date ' + fm(y.comps_target[n - 1])), chg(y.comps_ytd_total - y.comps_target[n - 1], fm(Math.abs(y.comps_ytd_total - y.comps_target[n - 1])) + ' vs target')]));
  const gm = v => '£' + (v / 1e6).toFixed(0) + 'm';
  chart($('c_ya'), {title: 'Applications, cumulative', x: y.days, ymin: 0, series: [{name: 'Applications', values: y.apps_ytd, color: css('--s1')}, {name: 'Target', values: y.apps_target, color: css('--target'), dash: true, width: 1.5}], fmt: gm, tipfmt: v => fm(v, 1)});
  chart($('c_yr'), {title: 'Weighted RoE, year to date', x: y.days, unit: '%', series: [{name: 'Weighted RoE', values: y.wa_roe, color: css('--s2')}, {name: 'Daily RoE', values: y.roe, color: css('--s4'), width: 1}, {name: 'Target', values: y.days.map(() => y.target_roe), color: css('--target'), dash: true, width: 1.5}], fmt: (v, s) => (v * 100).toFixed(s < 0.01 ? 1 : 0) + '%', tipfmt: v => fp(v, 1)});
  chart($('c_yc'), {title: 'Completions, cumulative', x: y.days, ymin: 0, series: [{name: 'Completions', values: y.comps_ytd, color: css('--s1')}, {name: 'Target', values: y.comps_target, color: css('--target'), dash: true, width: 1.5}], fmt: gm, tipfmt: v => fm(v, 1)});
  chart($('c_yf'), {title: 'Weighted fee-adjusted margin over swap, year to date', x: y.days, unit: '%', series: [{name: '2-year', values: y.wa_fapsm_2y, color: css('--s2')}, {name: '5-year', values: y.wa_fapsm_5y, color: css('--s4')}], fmt: (v, s) => (v * 100).toFixed(2) + '%', tipfmt: v => fp(v, 2)});
  $('ynote').textContent = 'Financial year from ' + fdate(y.fy_start) + '. Targets: ' + fm(DATA.targets.completions_fy, 0) + ' completions, applications at ' + fp(DATA.targets.conversion, 0) + ' conversion, spread evenly over ' + y.fy_days + ' working days. Applications include product switches. '
    + (DATA.meta.match_excel ? 'Weighted RoE and margins weight each day by cumulative year-to-date applications, as the Excel Page 5 does.' : 'Weighted RoE and margins weight each day by that day’s applications.');
}

/* ---------- notes ---------- */
function renderNotes() {
  const t = el('table'); const h = el('tr'); ['Check', 'Result', 'Detail'].forEach(k => h.appendChild(el('th', null, k))); t.appendChild(h);
  DATA.checks.forEach(c => { const tr = el('tr'); tr.appendChild(el('td', null, c.check)); const r = el('td', 'n'); const s = el('span', c.ok ? 'ok' : 'bad', (c.ok ? '✓ ' : '! ') + c.value); r.appendChild(s); tr.appendChild(r); const d = el('td', null, c.detail); d.style.whiteSpace = 'normal'; d.style.textAlign = 'left'; tr.appendChild(d); t.appendChild(tr); });
  $('chk').replaceChildren(t);
  const mt = $('meta'); mt.replaceChildren(); Object.entries(DATA.meta.inputs).forEach(([k, v]) => { mt.appendChild(el('div', null, k)); mt.appendChild(el('div', null, String(v))); });
  const x = $('xl'); x.replaceChildren(el('div', 'note', DATA.meta.match_excel ? 'MATCH_EXCEL is on: these follow the Excel workbook so the two agree while they run side by side.' : 'MATCH_EXCEL is off: these use the labelled meaning, so they differ from the Excel workbook.'));
  const ul = el('ul'); Object.values(DATA.meta.excel_notes).forEach(s => ul.appendChild(el('li', null, s))); x.appendChild(ul);
}

function renderDay() { header(); renderTiles(); renderTrends(); renderGrid(); renderProducts(); $('comm').textContent = DATA.days[DAY].commentary; }
function renderAll() { renderDay(); if ($('ytd').classList.contains('on')) renderYTD(); renderNotes(); }
$('copyc').onclick = () => { navigator.clipboard && navigator.clipboard.writeText($('comm').textContent); $('copyc').textContent = 'Copied'; setTimeout(() => $('copyc').textContent = 'Copy', 1500); };
renderAll();
</script>
</body>
</html>
"""


def render_html(data):
    """The dashboard page for one run. `data` comes from fleet_daily.dashboard_data()."""
    js = json.dumps(data, separators=(',', ':')).replace('</', '<\\/')
    return TEMPLATE.replace('__DATA__', js)

In [ ]:
# @title
%%writefile fleet_daily_xlsx.py
"""fleet_daily_xlsx: the D01 dashboard as an Excel workbook.

Input tabs hold the day's data as values (totals per day and per pricing code only, no loan rows): settings, the pricing
model's results, current pricing, applications and completions by day, margins by day, and the history. The Dashboard,
Products and Year to date tabs are formulas over those inputs (SUMIFS / INDEX-MATCH / SUMPRODUCT, no Excel 365-only
functions), with charts, so the workbook recalculates if an input is changed. Excel recalculates everything on opening.
"""
import datetime as _dt
import os

import numpy as np
import pandas as pd

PURPLE, TEAL, OFFWHITE, INK2, MUTED, GRID = '321E37', '50FFEB', 'F8F5F0', '5E5062', '8E8590', 'E8E4E9'
S1, S2, S3, S4 = '5A6FE0', '1E9E8F', 'D9731E', '8E5AA8'
GBP_M = '£#,##0.00,,"m";-£#,##0.00,,"m";"-"'
GBP_M1 = '£#,##0.0,,"m";-£#,##0.0,,"m";"-"'
GBP = '£#,##0;-£#,##0;"-"'
PCT1, PCT2 = '0.0%;-0.0%;"-"', '0.00%;-0.00%;"-"'
PTS = '+0.0%;-0.0%;0.0%'
PTS2 = '+0.00%;-0.00%;0.00%'
BP = '+0" bp";-0" bp";"0 bp"'
DATE = 'dd mmm yyyy'


def _styles():
    from openpyxl.styles import Alignment, Border, Font, PatternFill, Side
    return {
        'title': Font(name='Arial', size=16, bold=True, color='FFFFFF'),
        'band': PatternFill('solid', fgColor=PURPLE),
        'h2': Font(name='Arial', size=12, bold=True, color=PURPLE),
        'hdr': Font(name='Arial', size=9, bold=True, color=INK2),
        'hdrfill': PatternFill('solid', fgColor=OFFWHITE),
        'body': Font(name='Arial', size=10, color='000000'),
        'input': Font(name='Arial', size=10, color='0000FF'),
        'link': Font(name='Arial', size=10, color='008000'),
        'big': Font(name='Arial', size=18, bold=True, color=PURPLE),
        'bold': Font(name='Arial', size=10, bold=True),
        'note': Font(name='Arial', size=9, italic=True, color=MUTED),
        'subtitle': Font(name='Arial', size=9, italic=True, color='DDDDDD'),
        'line': Border(bottom=Side(style='thin', color=GRID)),
        'top': Border(top=Side(style='thin', color=MUTED)),
        'wrap': Alignment(wrap_text=True, vertical='top'),
        'right': Alignment(horizontal='right'),
        'left': Alignment(horizontal='left'),
        'wrap_bottom': Alignment(wrap_text=True, vertical='bottom'),
        'wrap_right': Alignment(wrap_text=True, vertical='bottom', horizontal='right'),
        'center': Alignment(horizontal='center', vertical='center'),
    }


def _title(ws, st, text, sub, width=14):
    ws.sheet_view.showGridLines = False
    for c in range(1, width + 1):
        ws.cell(1, c).fill = st['band']
        ws.cell(2, c).fill = st['band']
    ws['A1'] = text
    ws['A1'].font = st['title']
    ws['A2'] = sub
    ws['A2'].font = st['subtitle']


def _txt(size=800, color=INK2, rot=None, bold=False):
    """Text properties for chart axes: small, muted, horizontal (rot=0) so Excel doesn't stand date labels on end."""
    from openpyxl.chart.text import RichText
    from openpyxl.drawing.text import CharacterProperties, Paragraph, ParagraphProperties, RichTextProperties
    cp = CharacterProperties(sz=size, b=bold, solidFill=color)
    bp = RichTextProperties(rot=rot, vert='horz') if rot is not None else RichTextProperties()
    return RichText(bodyPr=bp, p=[Paragraph(pPr=ParagraphProperties(defRPr=cp), endParaRPr=cp)])


def _ctitle(text, size=1050):
    """Chart title above the plot (not laid over it), in the page's ink colour."""
    from openpyxl.chart.text import RichText, Text
    from openpyxl.chart.title import Title
    from openpyxl.drawing.text import CharacterProperties, Paragraph, ParagraphProperties, RegularTextRun
    cp = CharacterProperties(sz=size, b=True, solidFill=PURPLE)
    return Title(tx=Text(rich=RichText(p=[Paragraph(pPr=ParagraphProperties(defRPr=cp), r=[RegularTextRun(rPr=cp, t=text)])])), overlay=False)


def _tidy_chart(ch, n_points, legend=None, labels=8):
    """Square corners, small horizontal axis labels (about `labels` of them), legend only when asked for and never over the plot."""
    ch.roundedCorners = False
    ch.x_axis.auto = False                       # text axis: one label per point, so the skip below applies to dates too
    skip = max(1, -(-n_points // labels))
    ch.x_axis.tickLblSkip, ch.x_axis.tickMarkSkip = skip, skip
    ch.x_axis.txPr, ch.y_axis.txPr = _txt(rot=0), _txt()
    ch.x_axis.majorTickMark = ch.y_axis.majorTickMark = 'none'
    if legend:
        ch.legend.position, ch.legend.overlay = legend, False
        ch.legend.txPr = _txt()
    else:
        ch.legend = None


def _print_page(ws, area, footer_left):
    """One landscape A4 page: the print area scaled to fit, centred, with a small footer."""
    from openpyxl.worksheet.page import PageMargins
    from openpyxl.worksheet.properties import PageSetupProperties
    ws.print_area = area
    ws.page_setup.orientation, ws.page_setup.paperSize = 'landscape', ws.PAPERSIZE_A4
    ws.page_setup.fitToWidth, ws.page_setup.fitToHeight = 1, 1
    ws.sheet_properties.pageSetUpPr = PageSetupProperties(fitToPage=True)
    ws.print_options.horizontalCentered = True
    ws.page_margins = PageMargins(left=0.35, right=0.35, top=0.4, bottom=0.5, header=0.2, footer=0.25)
    ws.oddFooter.left.text, ws.oddFooter.left.size = footer_left, 8
    ws.oddFooter.right.text, ws.oddFooter.right.size = 'Totals only, no loan-level data', 8


def _print_wide(ws):
    """Input tabs: landscape, all columns on one page width, as many pages down as needed."""
    from openpyxl.worksheet.properties import PageSetupProperties
    ws.page_setup.orientation, ws.page_setup.paperSize = 'landscape', ws.PAPERSIZE_A4
    ws.page_setup.fitToWidth, ws.page_setup.fitToHeight = 1, 0
    ws.sheet_properties.pageSetUpPr = PageSetupProperties(fitToPage=True)
    ws.print_title_rows = None


def _table(ws, st, top, cols, rows, fmts=None, font='input', widths=None):
    """Header row at `top`, then rows of values (or formulas). Returns the first and last data row."""
    for j, c in enumerate(cols, 1):
        cell = ws.cell(top, j, c)
        cell.font, cell.fill = st['hdr'], st['hdrfill']
    for i, r in enumerate(rows, top + 1):
        for j, v in enumerate(r, 1):
            if isinstance(v, float) and not np.isfinite(v):
                v = None
            if isinstance(v, (np.floating,)):
                v = float(v)
            if isinstance(v, (np.integer,)):
                v = int(v)
            if isinstance(v, pd.Timestamp):
                v = v.to_pydatetime()
            cell = ws.cell(i, j, v)
            cell.font = st[font] if not (isinstance(v, str) and v.startswith('=')) else st['body']
            if fmts and fmts[j - 1]:
                cell.number_format = fmts[j - 1]
    if widths:
        from openpyxl.utils import get_column_letter
        for j, w in enumerate(widths, 1):
            ws.column_dimensions[get_column_letter(j)].width = w
    return top + 1, top + len(rows)


def build(path, rd, prev, cp, apps, comps, out, hist, swaps, checks_df, meta, targets, commentary='', proposed=None):
    """Write the Excel dashboard for reporting date `rd`. apps / comps: enriched tapes (aggregated here, never written)."""
    import openpyxl
    from openpyxl.chart import BarChart, LineChart, Reference
    from openpyxl.utils import get_column_letter as L
    from openpyxl.workbook.properties import CalcProperties
    import fleet_daily as D

    st = _styles()
    rd, prev = pd.Timestamp(rd), pd.Timestamp(prev)
    f0 = D.fy_start(rd)
    mstart = rd - pd.Timedelta(days=30)   # rolling 30-day window: applications after this date up to the reporting date
    codes = D.code_list(cp)
    wb = openpyxl.Workbook()
    wb.calculation = CalcProperties(fullCalcOnLoad=True)

    # ======================================================================= input tabs
    # --- Settings
    s = wb.active
    s.title = 'Settings'
    _title(s, st, 'Settings', 'Inputs for this run (blue). Change one and every page recalculates.', 4)
    sw = lambda d, col: (float(D.swap_on(pd.Series([d]), swaps, col).iloc[0]) / 100) if (col in swaps.columns and swaps[col].notna().any()) else None
    rows = [('Reporting date', rd, DATE), ('Previous working day', prev, DATE), ('7-day window starts after', rd - pd.Timedelta(days=7), DATE),
            ('30-day window starts after', mstart, DATE), ('Financial year start', f0, DATE),
            ('Working days in the year', targets['fy_working_days'] or len(D.bdays(f0, f0 + pd.DateOffset(years=1) - pd.Timedelta(days=1))), '0'),
            ('Completions target for the year', targets['completions_fy'], GBP_M1), ('Application to completion conversion', targets['conversion'], '0%'),
            ('Daily applications target', targets['daily_volume'], GBP_M1), ('Target RoE', targets['roe'], PCT1),
            ('Proposed-mix RoE (Current Pricing AQ4)', proposed if proposed is not None else cp.get('proposed_roe'), PCT2),
            ('Swap 1-year today', sw(rd, '1Y'), PCT2), ('Swap 2-year today', sw(rd, '2Y'), PCT2), ('Swap 5-year today', sw(rd, '5Y'), PCT2),
            ('Swap 1-year previous day', sw(prev, '1Y'), PCT2), ('Swap 2-year previous day', sw(prev, '2Y'), PCT2), ('Swap 5-year previous day', sw(prev, '5Y'), PCT2)]
    for i, (k, v, f) in enumerate(rows, 4):
        s.cell(i, 1, k).font = st['body']
        c = s.cell(i, 2, v.to_pydatetime() if isinstance(v, pd.Timestamp) else v)
        c.font, c.number_format = st['input'], f
    s.column_dimensions['A'].width, s.column_dimensions['B'].width = 40, 16
    S = {k: "Settings!$B$%d" % (i + 4) for i, k in enumerate(['rd', 'prev', 'wk', 'mth', 'fy', 'nd', 'comp_t', 'conv', 'day_t', 'roe_t', 'prop',
                                                             's1', 's2', 's5', 's1p', 's2p', 's5p'])}
    s['D3'] = 'Bank holidays'
    s['D3'].font = st['hdr']
    for i, h_ in enumerate(D.BANK_HOLIDAYS, 4):
        c = s.cell(i, 4, pd.Timestamp(h_).to_pydatetime())
        c.font, c.number_format = st['input'], DATE
    s.column_dimensions['D'].width = 13
    HOL = 'Settings!$D$4:$D$%d' % (3 + len(D.BANK_HOLIDAYS))
    r0 = 4 + len(rows) + 1
    s.cell(r0, 1, 'About this file').font = st['h2']
    for i, (k, v) in enumerate(meta.get('inputs', {}).items(), r0 + 1):
        s.cell(i, 1, k).font = st['body']
        s.cell(i, 2, str(v)).font = st['note']

    # --- Model Output (the LLPM Outputs tab, per code and run)
    mo = wb.create_sheet('Model Output')
    _title(mo, st, 'Pricing model results', 'Outputs tab of the LLPM run on this day\'s input file. Key = pricing code | run.', 9)
    cols = ['Key', 'Code', 'Run', 'RoE', 'NetReturn', 'NIM%', 'CurrentBalance', 'CurrentRate', 'PostSwapEIR']
    rr = [['%s|%s' % (c_, r_), c_, r_] + [o.get(k) for k in cols[3:]] for c_, r_, o in
          ((x['Code'], x['Run'], x) for x in out.to_dict('records'))]
    mo1, mo2 = _table(mo, st, 4, cols, rr, [None, None, None, PCT2, PCT2, PCT2, GBP, PCT2, PCT2], widths=[22, 16, 13, 10, 10, 10, 14, 10, 11])
    MO = lambda col: "'Model Output'!$%s$%d:$%s$%d" % (col, mo1, col, mo2)

    # --- Current Pricing
    pr = wb.create_sheet('Current Pricing')
    _title(pr, st, 'Current pricing', 'Pricing codes in dashboard order, with the RoE from the pricing exercise (Current Pricing AO).', 6)
    rets = cp['returns'].drop_duplicates('code').set_index('code') if len(cp['returns']) else pd.DataFrame()
    rr = [[c_, 'Y' if 'LOYAL' in c_ else 'N', float(rets.at[c_, 'roe']) if (len(rets) and c_ in rets.index and rets.at[c_, 'roe'] is not None) else 0.0,
           float(rets.at[c_, 'mix_proposed']) if (len(rets) and c_ in rets.index and rets.at[c_, 'mix_proposed'] is not None) else None]
          for c_ in codes]
    cp1, cp2 = _table(pr, st, 4, ['Pricing code', 'Loyalty', 'Pricing RoE', 'Proposed mix'], rr, [None, None, PCT2, PCT2], widths=[18, 9, 12, 12])

    # --- Applications by day and code (totals; RoE of that day's model run alongside)
    ap = wb.create_sheet('Applications')
    _title(ap, st, 'Applications by day and pricing code',
           'Totals from the DIPs & Apps tape (no loan rows). RoE / net return = that day\'s model run for the code, where one was saved with applications.', 14)
    lo = min(f0, mstart, rd - pd.Timedelta(days=60))
    a = apps[(apps['ApplicationDate'] >= lo) & (apps['ApplicationDate'] <= rd)].copy()
    a['PricingCode'] = a['PricingCode'].astype(str)
    g = a.groupby(['ApplicationDate', 'PricingCode', 'Product Switch'], dropna=False).agg(vol=('OriginalBalance', 'sum'), n=('OriginalBalance', 'size')).reset_index()
    g['Scheme'] = np.where(g['PricingCode'].str.contains('LOYAL'), 'LOYALTY', np.where(g['PricingCode'].str[:3].isin(['BTL', 'LTD', 'HMO']), g['PricingCode'].str[:3], 'Other'))
    g = g.sort_values(['ApplicationDate', 'PricingCode'])
    hc = hist['codes'].copy()
    hc['Date'] = pd.to_datetime(hc['Date'])
    hc = hc[(hc['Run'] == 'App') & (hc['Date'] >= lo) & (hc['Date'] <= rd)].drop_duplicates(['Date', 'Code'], keep='last')
    first = 5
    rr = []
    for i, x in enumerate(g.itertuples(index=False), first):
        key = '%s|%s' % (x.ApplicationDate.strftime('%Y-%m-%d'), x.PricingCode)
        m = "MATCH($A%d,'Code History'!$A$5:$A$%d,0)" % (i, 4 + max(1, len(hc)))
        rr.append([key, x.ApplicationDate, x.PricingCode, x._2, x.Scheme, x.vol, x.n,
                   "=IFERROR(INDEX('Code History'!$D$5:$D$%d,%s),\"\")" % (4 + max(1, len(hc)), m),
                   "=IFERROR(INDEX('Code History'!$E$5:$E$%d,%s),\"\")" % (4 + max(1, len(hc)), m),
                   "=IF(AND(H%d<>\"\",IFERROR(INDEX('Code History'!$F$5:$F$%d,%s),0)>0),1,0)" % (i, 4 + max(1, len(hc)), m),
                   '=F%d*J%d' % (i, i), '=IF(J%d=1,F%d*H%d,0)' % (i, i, i), '=IF(J%d=1,F%d*I%d,0)' % (i, i, i)])
    a1, a2 = _table(ap, st, 4, ['Key', 'Date', 'Pricing code', 'Product switch', 'Scheme', 'Volume', 'Cases', 'RoE that day', 'Net return that day',
                                'Has return', 'Volume with return', 'Volume x RoE', 'Volume x net return'],
                    rr, [None, DATE, None, None, None, GBP, '0', PCT2, PCT2, '0', GBP, GBP, GBP], widths=[24, 12, 16, 9, 9, 14, 7, 10, 10, 8, 14, 14, 14])
    a2 = max(a2, a1)
    A = lambda col: 'Applications!$%s$%d:$%s$%d' % (col, a1, col, a2)

    # --- Completions by day
    cm = wb.create_sheet('Completions')
    _title(cm, st, 'Completions by day', 'Totals from the Comps tape, retention included.', 4)
    c = comps[(comps['LoanOriginationDate'] >= lo) & (comps['LoanOriginationDate'] <= rd)]
    cg = c.groupby('LoanOriginationDate').agg(vol=('OriginalBalance', 'sum'), n=('OriginalBalance', 'size')).reset_index()
    c1, c2 = _table(cm, st, 4, ['Date', 'Volume', 'Cases'], [[d, v, n] for d, v, n in cg.itertuples(index=False)], [DATE, GBP, '0'], widths=[12, 14, 8])
    c2 = max(c2, c1)
    CM = lambda col: 'Completions!$%s$%d:$%s$%d' % (col, c1, col, c2)

    # --- Margins by day (fee-adjusted margin over swap, £-weighted sums)
    mg = wb.create_sheet('Margins')
    _title(mg, st, 'Fee-adjusted margin over swap by day', 'Sums over that day\'s applications: balance, and balance x FAPSM. Margin = second / first.', 9)
    f = a[a['FAPSM'].notna()].copy()
    f['bx'] = f['FAPSM'] * f['OriginalBalance']
    trc = f['PricingCode'].str.endswith('TRC')
    rows_m = []
    for d, x in f.groupby('ApplicationDate'):
        t2 = x['Fixed Term'] == 2
        t5 = x['Fixed Term'] == 5
        ex = t2 & ~trc[x.index]
        rows_m.append([d, x.loc[ex, 'OriginalBalance'].sum(), x.loc[ex, 'bx'].sum(), x.loc[t2, 'OriginalBalance'].sum(), x.loc[t2, 'bx'].sum(),
                       x.loc[t5, 'OriginalBalance'].sum(), x.loc[t5, 'bx'].sum(), x.loc[t2 | t5, 'OriginalBalance'].sum(), x.loc[t2 | t5, 'bx'].sum()])
    m1, m2 = _table(mg, st, 4, ['Date', '2Y excl trackers: balance', '2Y excl trackers: bal x FAPSM', '2Y: balance', '2Y: bal x FAPSM',
                                '5Y: balance', '5Y: bal x FAPSM', '2Y+5Y: balance', '2Y+5Y: bal x FAPSM'],
                    rows_m, [DATE] + [GBP] * 8, widths=[12] + [16] * 8)
    m2 = max(m2, m1)
    MG = lambda col: 'Margins!$%s$%d:$%s$%d' % (col, m1, col, m2)

    def fap(day, b, x):
        return '=IFERROR(SUMIFS(%s,%s,%s)/SUMIFS(%s,%s,%s),"")' % (MG(x), MG('A'), day, MG(b), MG('A'), day)

    # --- history
    ch = wb.create_sheet('Code History')
    _title(ch, st, 'Pricing model results by day', 'Each day\'s App run per code (from the history file). Balance 0 = no applications for the code that day.', 6)
    _table(ch, st, 4, ['Key', 'Date', 'Code', 'RoE', 'NetReturn', 'Balance'],
           [['%s|%s' % (d.strftime('%Y-%m-%d'), c_), d, c_, r_, n_, b_] for d, c_, r_, n_, b_ in hc[['Date', 'Code', 'RoE', 'NetReturn', 'Balance']].itertuples(index=False)],
           [None, DATE, None, PCT2, PCT2, GBP], widths=[24, 12, 16, 10, 10, 14])
    dh = hist['daily'].copy()
    dh['Date'] = pd.to_datetime(dh['Date'])
    dh = dh[dh['Date'] <= rd].sort_values('Date')
    dhs = wb.create_sheet('Daily History')
    _title(dhs, st, 'Headline return and volume by day', 'Return on equity (excluding product switches) and applications (excluding product switches), as reported each day.', 3)
    d1, d2 = _table(dhs, st, 4, ['Date', 'RoE', 'Volume'], [[d, r_, v] for d, r_, v in dh[['Date', 'RoE', 'Volume']].itertuples(index=False)],
                    [DATE, PCT2, GBP], widths=[12, 10, 14])
    DH = lambda col: "'Daily History'!$%s$%d:$%s$%d" % (col, d1, col, d2)

    # ======================================================================= Products (Page 2)
    p = wb.create_sheet('Products', 0)
    _title(p, st, 'Live products', 'Formulas over the input tabs. Returns weighted by applications; 7-day and 30-day figures weight each day\'s saved return.', 14)
    p['A3'] = '=TEXT(%s,"dd mmm yyyy")' % S['rd']
    p['A3'].font = st['h2']
    hdr = ['Pricing code', 'Loyalty', 'Pricing RoE', 'Pricing RoE at today\'s swaps', 'Volume today', 'Return today', 'Return -10bp', 'Annual contribution',
           'Volume last 7 days', 'W.A. return 7 days', 'Volume last 30 days', 'W.A. return 30 days', 'Contribution last 30 days']
    top = 5
    rows = []
    for i, c_ in enumerate(codes, top + 1):
        mk = lambda run, col, i=i: "IFERROR(INDEX(%s,MATCH($A%d&\"|%s\",%s,0)),0)" % (MO(col), i, run, MO('A'))
        rows.append([c_, "=INDEX('Current Pricing'!$B$%d:$B$%d,MATCH($A%d,'Current Pricing'!$A$%d:$A$%d,0))" % (cp1, cp2, i, cp1, cp2),
                     "=IFERROR(INDEX('Current Pricing'!$C$%d:$C$%d,MATCH($A%d,'Current Pricing'!$A$%d:$A$%d,0)),0)" % (cp1, cp2, i, cp1, cp2),
                     '=' + mk('PricingSwaps', 'D'),
                     '=SUMIFS(%s,%s,%s,%s,$A%d)' % (A('F'), A('B'), S['rd'], A('C'), i),
                     '=' + mk('App', 'D'), '=' + mk('App-10', 'D'), '=E%d*%s' % (i, mk('App', 'E')),
                     '=SUMIFS(%s,%s,">"&%s,%s,"<="&%s,%s,$A%d)' % (A('F'), A('B'), S['wk'], A('B'), S['rd'], A('C'), i),
                     '=IFERROR(SUMIFS(%s,%s,">"&%s,%s,"<="&%s,%s,$A%d)/SUMIFS(%s,%s,">"&%s,%s,"<="&%s,%s,$A%d),"")'
                     % (A('L'), A('B'), S['wk'], A('B'), S['rd'], A('C'), i, A('K'), A('B'), S['wk'], A('B'), S['rd'], A('C'), i),
                     '=SUMIFS(%s,%s,">"&%s,%s,"<="&%s,%s,$A%d)' % (A('F'), A('B'), S['mth'], A('B'), S['rd'], A('C'), i),
                     '=IFERROR(SUMIFS(%s,%s,">"&%s,%s,"<="&%s,%s,$A%d)/SUMIFS(%s,%s,">"&%s,%s,"<="&%s,%s,$A%d),"")'
                     % (A('L'), A('B'), S['mth'], A('B'), S['rd'], A('C'), i, A('K'), A('B'), S['mth'], A('B'), S['rd'], A('C'), i),
                     '=SUMIFS(%s,%s,">"&%s,%s,"<="&%s,%s,$A%d)' % (A('M'), A('B'), S['mth'], A('B'), S['rd'], A('C'), i)])
    p1_, p2_ = _table(p, st, top, hdr, rows, [None, None, PCT1, PCT1, GBP, PCT1, PCT1, GBP, GBP, PCT1, GBP, PCT1, GBP], font='body',
                      widths=[30, 8, 11, 13, 13, 11, 11, 13, 14, 12, 15, 13, 14])
    for cc in range(1, 14):
        p.cell(top, cc).alignment = st['wrap']
    p.row_dimensions[top].height = 30
    av1, av2 = p2_ + 1, p2_ + 2
    rng = lambda col: '$%s$%d:$%s$%d' % (col, p1_, col, p2_)
    for r_, label, cond in ((av1, 'Average excluding product switches', '"N"'), (av2, 'Average, all codes', '"*"')):
        p.cell(r_, 1, label).font = st['bold']
        flt = (',--(%s=%s)' % (rng('B'), cond)) if cond != '"*"' else ''
        den = ('SUMIF(%s,%s,%s)' % (rng('B'), cond, rng('E'))) if cond != '"*"' else 'SUM(%s)' % rng('E')
        wden = ('SUMIF(%s,%s,%s)' % (rng('B'), cond, rng('%s'))) if cond != '"*"' else 'SUM(%s)' % rng('%s')
        for col in ('C', 'D', 'F', 'G'):
            p['%s%d' % (col, r_)] = '=IFERROR(SUMPRODUCT(%s,%s%s)/%s,"")' % (rng('E'), rng(col), flt, den)
        p['E%d' % r_] = '=' + den
        p['H%d' % r_] = '=SUMPRODUCT(%s%s)' % (rng('H'), flt)
        p['I%d' % r_] = '=SUMPRODUCT(%s%s)' % (rng('I'), flt)
        p['K%d' % r_] = '=SUMPRODUCT(%s%s)' % (rng('K'), flt)
        p['M%d' % r_] = '=SUMPRODUCT(%s%s)' % (rng('M'), flt)
        for col, vcol, val, has in (('J', 'I', 'O', 'P'), ('L', 'K', 'Q', 'R')):
            p['%s%d' % (col, r_)] = '=IFERROR(SUMPRODUCT(%s,%s%s)/SUMPRODUCT(%s,%s%s),"")' % (rng(vcol), rng(val), flt, rng(vcol), rng(has), flt)
        for cc, fm in zip(range(1, 14), [None, None, PCT1, PCT1, GBP, PCT1, PCT1, GBP, GBP, PCT1, GBP, PCT1, GBP]):
            cell = p.cell(r_, cc)
            cell.font, cell.border = st['bold'], st['top']
            if fm:
                cell.number_format = fm
    for i in range(p1_, p2_ + 1):                    # helpers for the averages: the weekly / monthly return as a number, and whether there is one
        p['O%d' % i], p['P%d' % i] = '=IF(J%d="",0,J%d)' % (i, i), '=IF(J%d="",0,1)' % i
        p['Q%d' % i], p['R%d' % i] = '=IF(L%d="",0,L%d)' % (i, i), '=IF(L%d="",0,1)' % i
    for col in 'OPQR':
        p.column_dimensions[col].hidden = True
    p.freeze_panes = 'B6'
    _print_page(p, 'A1:M%d' % av2, '&8Fleet daily dashboard: live products')
    from openpyxl.formatting.rule import CellIsRule
    from openpyxl.styles import PatternFill
    for col in ('D', 'F', 'G', 'J', 'L'):
        area = '%s%d:%s%d' % (col, p1_, col, p2_)
        p.conditional_formatting.add(area, CellIsRule(operator='greaterThanOrEqual', formula=[S['roe_t']], fill=PatternFill('solid', fgColor='DCE3FF')))
        p.conditional_formatting.add(area, CellIsRule(operator='between', formula=['0.0001', S['roe_t'] + '-0.000001'], fill=PatternFill('solid', fgColor='FCE3CF')))
    PX = lambda col, r_: "Products!$%s$%d" % (col, r_)
    # note: the J / L averages weight by volume where a saved return exists

    # ======================================================================= Year to date (Page 5)
    y = wb.create_sheet('Year to date', 1)
    _title(y, st, 'Year to date', 'Cumulative applications (product switches included) and completions against target; weighted RoE and margins.', 14)
    days = [d for d in D.bdays(f0, rd)]
    yh = ['Date', 'Applications', 'Applications YTD', 'Target YTD', 'Completions', 'Completions YTD', 'Target YTD', 'Daily RoE', 'Has RoE',
          'W.A. RoE YTD', 'FAPSM 2Y', 'FAPSM 5Y', 'W.A. FAPSM 2Y', 'W.A. FAPSM 5Y', 'RoE if any', 'Has 2Y', '2Y if any', 'Has 5Y', '5Y if any']
    ytop = 40                                       # charts sit in rows 7-38, above the daily table
    yr = []
    for i, d in enumerate(days, ytop + 1):
        first_row = ytop + 1
        yr.append([d,
                   '=SUMIFS(%s,%s,">"&IF(ROW()=%d,%s-1,A%d),%s,"<="&A%d)' % (A('F'), A('B'), first_row, S['fy'], i - 1, A('B'), i),
                   '=SUM($B$%d:B%d)' % (first_row, i),
                   '=%d*(%s/%s)/%s' % (i - ytop, S['comp_t'], S['conv'], S['nd']),
                   '=SUMIFS(%s,%s,">"&IF(ROW()=%d,%s-1,A%d),%s,"<="&A%d)' % (CM('B'), CM('A'), first_row, S['fy'], i - 1, CM('A'), i),
                   '=SUM($E$%d:E%d)' % (first_row, i),
                   '=%d*%s/%s' % (i - ytop, S['comp_t'], S['nd']),
                   '=IFERROR(INDEX(%s,MATCH(A%d,%s,0)),"")' % (DH('B'), i, DH('A')),
                   '=IF(H%d="",0,1)' % i,
                   '=IFERROR(SUMPRODUCT($C$%d:C%d,$O$%d:O%d)/SUMPRODUCT($C$%d:C%d,$I$%d:I%d),"")' % ((first_row, i) * 4),
                   fap('A%d' % i, 'D', 'E'), fap('A%d' % i, 'F', 'G'),        # Page 5: 2-year with trackers, as the Excel
                   '=IFERROR(SUMPRODUCT($C$%d:C%d,$Q$%d:Q%d)/SUMPRODUCT($C$%d:C%d,$P$%d:P%d),"")' % ((first_row, i) * 4),
                   '=IFERROR(SUMPRODUCT($C$%d:C%d,$S$%d:S%d)/SUMPRODUCT($C$%d:C%d,$R$%d:R%d),"")' % ((first_row, i) * 4),
                   '=IF(I%d=1,H%d,0)' % (i, i), '=IF(K%d="",0,1)' % i, '=IF(P%d=1,K%d,0)' % (i, i), '=IF(L%d="",0,1)' % i, '=IF(R%d=1,L%d,0)' % (i, i)])
    y1, y2 = _table(y, st, ytop, yh, yr, [DATE, GBP, GBP_M1, GBP_M1, GBP, GBP_M1, GBP_M1, PCT1, '0', PCT2, PCT2, PCT2, PCT2, PCT2] + [None] * 5, font='body',
                    widths=[12, 13, 14, 14, 13, 14, 14, 10, 7, 11, 10, 10, 12, 12])
    for col in 'IOPQRS':
        y.column_dimensions[col].hidden = True
    for cc in range(1, 15):
        y.cell(ytop, cc).alignment = st['wrap']
    boxes = [('Applications YTD', '=C%d' % y2, GBP_M1), ('Target to date', '=D%d' % y2, GBP_M1),
             ('Weighted RoE YTD', '=J%d' % y2, PCT1), ('Applications still needed', '=%s/%s-C%d' % (S['comp_t'], S['conv'], y2), GBP_M1),
             ('RoE needed on them for target', '=IFERROR((%s/%s*%s-C%d*J%d)/(%s/%s-C%d),"")' % (S['comp_t'], S['conv'], S['roe_t'], y2, y2, S['comp_t'], S['conv'], y2), PCT2),
             ('Completions YTD', '=F%d' % y2, GBP_M1), ('Completions target to date', '=G%d' % y2, GBP_M1)]
    box_cols = ['A', 'C', 'E', 'G', 'J', 'L', 'N']          # column I is a hidden helper, so the boxes step over it
    for j, (k, f_, fm) in enumerate(boxes):
        col = box_cols[j]
        y['%s4' % col] = k
        y['%s4' % col].font, y['%s4' % col].alignment = st['hdr'], st['wrap']
        c_ = y['%s5' % col]
        c_.value, c_.font, c_.number_format = f_, st['big'], fm
        if col != 'N':
            nxt = L(openpyxl.utils.column_index_from_string(col) + 1)
            y.merge_cells('%s4:%s4' % (col, nxt))
            y.merge_cells('%s5:%s5' % (col, nxt))
    y.column_dimensions['N'].width = 16
    y.row_dimensions[4].height = 26
    YB = lambda j: "'Year to date'!$%s$5" % box_cols[j]

    def line(title, ws, xref, series, anchor, fmt='0%', w=16, h=7.5, ymin=None, legend=None):
        ch_ = LineChart()
        ch_.title, ch_.height, ch_.width = _ctitle(title), h, w
        ch_.y_axis.number_format = fmt
        ch_.y_axis.majorGridlines = None
        if ymin is not None:
            ch_.y_axis.scaling.min = ymin
        for ref, color, dash in series:
            ch_.add_data(ref, titles_from_data=True)
            sr = ch_.series[-1]
            sr.graphicalProperties.line.solidFill = color
            sr.graphicalProperties.line.width = 20000
            sr.smooth = False
            if dash:
                sr.graphicalProperties.line.dashStyle = 'dash'
        ch_.set_categories(xref)
        ch_.x_axis.number_format = 'dd mmm'
        ch_.x_axis.delete = False
        ch_.y_axis.delete = False
        _tidy_chart(ch_, xref.max_row - xref.min_row + 1, legend)
        ws.add_chart(ch_, anchor)

    xr = Reference(y, min_col=1, min_row=y1, max_row=y2)
    line('Applications, cumulative (dashed line: target)', y, xr, [(Reference(y, min_col=3, min_row=ytop, max_row=y2), S1, False),
                                              (Reference(y, min_col=4, min_row=ytop, max_row=y2), MUTED, True)], 'A7', GBP_M1.split(';')[0], w=15.5, h=7.6, ymin=0)
    line('Weighted RoE, year to date', y, xr, [(Reference(y, min_col=10, min_row=ytop, max_row=y2), S2, False)], 'A23', '0.0%', w=15.5, h=7.6)
    line('Completions, cumulative (dashed line: target)', y, xr, [(Reference(y, min_col=6, min_row=ytop, max_row=y2), S1, False),
                                             (Reference(y, min_col=7, min_row=ytop, max_row=y2), MUTED, True)], 'G7', GBP_M1.split(';')[0], w=15.5, h=7.6, ymin=0)
    line('Weighted fee-adjusted margin over swap', y, xr, [(Reference(y, min_col=13, min_row=ytop, max_row=y2), S2, False),
                                                            (Reference(y, min_col=14, min_row=ytop, max_row=y2), S4, False)], 'G23', '0.00%', w=15.5, h=7.6, legend='t')
    # no freeze panes: the charts sit above the table
    y['A3'] = '=TEXT(%s,"dd mmm yyyy")' % S['rd']
    y['A3'].font = st['h2']
    _print_page(y, 'A1:N38', '&8Fleet daily dashboard: year to date')

    # ======================================================================= Dashboard (Page 1)
    d = wb.create_sheet('Dashboard', 0)
    _title(d, st, 'Fleet Daily Dashboard', 'Applications against the previous working day. Every figure is a formula over the input tabs.', 13)
    d['A3'] = '="Applications on "&TEXT(%s,"dddd dd mmmm yyyy")&" against "&TEXT(%s,"dd mmm")' % (S['rd'], S['prev'])
    d['A3'].font = st['h2']
    for col, w in zip('ABCDEFGHIJKLMN', [32, 12, 12, 11, 11, 11, 11, 3, 30, 12, 13, 14, 3, 3]):
        d.column_dimensions[col].width = w
    vol = lambda day, ps: ('SUMIFS(%s,%s,%s,%s,"%s")' % (A('F'), A('B'), day, A('D'), ps)) if ps else 'SUMIFS(%s,%s,%s)' % (A('F'), A('B'), day)
    cnt = lambda day, ps: ('SUMIFS(%s,%s,%s,%s,"%s")' % (A('G'), A('B'), day, A('D'), ps)) if ps else 'SUMIFS(%s,%s,%s)' % (A('G'), A('B'), day)
    d['A5'] = 'Volume'
    d['A5'].font = st['h2']
    for j, v in enumerate(['', 'Volume', 'On day', 'Cases', 'On day']):
        d.cell(6, 1 + j, v).font = st['hdr']
        d.cell(6, 1 + j).alignment = st['right']
    vrows = [('Applications (excl. product switches)', vol(S['rd'], 'N'), vol(S['prev'], 'N'), cnt(S['rd'], 'N'), cnt(S['prev'], 'N')),
             ('Applications incl. product switches', vol(S['rd'], None), vol(S['prev'], None), cnt(S['rd'], None), cnt(S['prev'], None)),
             ('Completions', 'SUMIFS(%s,%s,%s)' % (CM('B'), CM('A'), S['rd']), 'SUMIFS(%s,%s,%s)' % (CM('B'), CM('A'), S['prev']),
              'SUMIFS(%s,%s,%s)' % (CM('C'), CM('A'), S['rd']), 'SUMIFS(%s,%s,%s)' % (CM('C'), CM('A'), S['prev']))]
    for i, (k, v_, vp, n_, np_) in enumerate(vrows, 7):
        d.cell(i, 1, k).font = st['body']
        for col, f_, fm in (('B', '=' + v_, GBP_M), ('C', '=%s-(%s)' % (v_, vp), GBP_M), ('D', '=' + n_, '0'), ('E', '=%s-(%s)' % (n_, np_), '+0;-0;0')):
            c_ = d['%s%d' % (col, i)]
            c_.value, c_.number_format, c_.font = f_, fm, st['bold'] if col == 'B' else st['body']
    d['I5'] = 'Daily returns (excluding product switches)'
    d['I5'].font = st['h2']
    for j, v in enumerate(['Return', 'vs pricing RoEs', 'vs proposed']):
        d.cell(6, 10 + j, v).font = st['hdr']
        d.cell(6, 10 + j).alignment = st['right']
    d['I7'], d['I8'] = 'Return on equity today', 'With 10bp off the margin'
    d['J7'] = '=%s' % PX('F', av1)
    d['K7'] = '=%s-%s' % (PX('F', av1), PX('C', av1))
    d['L7'] = '=%s-%s' % (PX('F', av1), S['prop'])
    d['J8'] = '=%s' % PX('G', av1)
    d['K8'] = '=%s-%s' % (PX('G', av1), PX('C', av1))
    d['L8'] = '=%s-%s' % (PX('G', av1), S['prop'])
    d['I9'], d['J9'] = 'Change from 10bp off', '=J8-J7'
    d['I10'], d['J10'] = 'Pricing RoE at today\'s mix', '=%s' % PX('C', av1)
    d['I11'], d['J11'] = 'Proposed-mix RoE', '=%s' % S['prop']
    for r_ in range(7, 12):
        d['I%d' % r_].font = st['body']
        for col in 'JKL':
            c_ = d['%s%d' % (col, r_)]
            c_.number_format = PCT1 if col == 'J' and r_ != 9 else PTS
            c_.font = st['bold'] if col == 'J' else st['body']
    d['A13'] = 'Fee-adjusted margin over swap'
    d['A13'].font = st['h2']
    for j, v in enumerate(['', 'Today', 'On day']):
        d.cell(14, 1 + j, v).font = st['hdr']
        d.cell(14, 1 + j).alignment = st['right']
    frows = [('2-year (trackers excluded)', fap(S['rd'], 'B', 'C')[1:],
              # the Excel dashboard compares today's figure with today's 2-year figure including trackers
              fap(S['rd'], 'D', 'E')[1:] if D.MATCH_EXCEL else fap(S['prev'], 'B', 'C')[1:]),
             ('5-year', fap(S['rd'], 'F', 'G')[1:], fap(S['prev'], 'F', 'G')[1:]),
             ('2- and 5-year, trackers included', fap(S['rd'], 'H', 'I')[1:], fap(S['prev'], 'H', 'I')[1:])]
    for i, (k, t_, p_) in enumerate(frows, 15):
        d.cell(i, 1, k).font = st['body']
        d['B%d' % i], d['C%d' % i] = '=' + t_, '=IFERROR(%s-%s,"")' % (t_, p_)
        d['B%d' % i].number_format, d['C%d' % i].number_format = PCT2, PTS2
        d['B%d' % i].font = st['bold']
    d['I13'] = 'Swaps'
    d['I13'].font = st['h2']
    for j, v in enumerate(['1-year', '2-year', '5-year']):
        d.cell(14, 10 + j, v).font = st['hdr']
        d.cell(14, 10 + j).alignment = st['right']
    for i, (k, keys) in enumerate((('Today', ('s1', 's2', 's5')), ('Previous day', ('s1p', 's2p', 's5p'))), 15):
        d.cell(i, 9, k).font = st['body']
        for j, kk in enumerate(keys):
            c_ = d.cell(i, 10 + j, '=IF(%s="","",%s)' % (S[kk], S[kk]))
            c_.number_format = PCT2
    d.cell(17, 9, 'Movement').font = st['body']
    for j in range(3):
        c_ = d.cell(17, 10 + j, '=IFERROR((%s15-%s16)*10000,"")' % (L(10 + j), L(10 + j)))
        c_.number_format = BP
    d['A19'] = 'Year to date'
    d.row_dimensions[20].height = 26
    d['A19'].font = st['h2']
    for j, (k, ref, fm) in enumerate([('Applications YTD', YB(0), GBP_M1), ('Weighted RoE YTD', YB(2), PCT1), ('Still needed', YB(3), GBP_M1),
                                      ('RoE needed on them', YB(4), PCT2), ('Completions YTD', YB(5), GBP_M1)]):
        d.cell(20, 1 + j, k).font = st['hdr']
        d.cell(20, 1 + j).alignment = st['wrap_bottom'] if j == 0 else st['wrap_right']
        c_ = d.cell(21, 1 + j, '=' + ref)
        c_.number_format, c_.font = fm, st['bold']
    # applications by scheme, last 10 working days
    d['A23'] = 'Applications by scheme, last 10 working days'
    d['A23'].font = st['h2']
    for j, v in enumerate(['Date', 'BTL', 'LTD', 'HMO', 'Loyalty', 'Other', 'Total']):
        d.cell(24, 1 + j, v).font = st['hdr']
        d.cell(24, 1 + j).alignment = st['left'] if j == 0 else st['right']
    last10 = list(D.bdays(rd - pd.Timedelta(days=20), rd))[-10:][::-1]
    for i, day in enumerate(last10, 25):
        c_ = d.cell(i, 1, '=WORKDAY(%s,-%d,%s)' % (S['rd'], i - 25, HOL) if i > 25 else '=%s' % S['rd'])
        c_.number_format, c_.alignment = DATE, st['left']
        for j, sc in enumerate(['BTL', 'LTD', 'HMO', 'LOYALTY', 'Other']):
            c_ = d.cell(i, 2 + j, '=SUMIFS(%s,%s,$A%d,%s,"%s")' % (A('F'), A('B'), i, A('E'), sc))
            c_.number_format = GBP_M
        c_ = d.cell(i, 7, '=SUM(B%d:F%d)' % (i, i))
        c_.number_format, c_.font = GBP_M, st['bold']
    # trend data and charts (last 60 reported days)
    d['P5'] = 'Trend data (last 60 reported days, from Daily History)'
    d['P5'].font = st['h2']
    for j, v in enumerate(['Date', 'Applications', 'Target', 'RoE', 'Target RoE']):
        d.cell(6, 16 + j, v).font = st['hdr']
    n_tr = 60
    for k in range(n_tr):
        r_ = 7 + k
        idx = 'ROWS(%s)-%d' % (DH('A'), n_tr - 1 - k)
        d.cell(r_, 16, '=IFERROR(TEXT(INDEX(%s,%s),"dd mmm"),"")' % (DH('A'), idx)).alignment = st['right']   # text, so the bar chart has no weekend gaps
        d.cell(r_, 17, '=IFERROR(INDEX(%s,%s),"")' % (DH('C'), idx)).number_format = GBP_M
        d.cell(r_, 18, '=%s' % S['day_t']).number_format = GBP_M
        d.cell(r_, 19, '=IFERROR(INDEX(%s,%s),"")' % (DH('B'), idx)).number_format = PCT1
        d.cell(r_, 20, '=%s' % S['roe_t']).number_format = PCT1
    for col, w in zip('PQRST', [11, 13, 12, 9, 10]):
        d.column_dimensions[col].width = w
    xr = Reference(d, min_col=16, min_row=7, max_row=6 + n_tr)
    bc = BarChart()
    bc.title, bc.height, bc.width = _ctitle('Daily applications excl. switches (dashed line: target)'), 7.6, 13.6
    bc.add_data(Reference(d, min_col=17, min_row=6, max_row=6 + n_tr), titles_from_data=True)
    bc.series[0].graphicalProperties.solidFill = S1
    bc.series[0].graphicalProperties.line.noFill = True
    bc.set_categories(xr)
    bc.y_axis.number_format = '£#,##0,,"m"'
    bc.y_axis.majorGridlines = None
    bc.gapWidth = 40
    lc = LineChart()
    lc.add_data(Reference(d, min_col=18, min_row=6, max_row=6 + n_tr), titles_from_data=True)
    lc.series[0].graphicalProperties.line.solidFill = MUTED
    lc.series[0].graphicalProperties.line.dashStyle = 'dash'
    bc += lc
    bc.x_axis.number_format = 'dd mmm'
    bc.x_axis.delete = False
    bc.y_axis.delete = False
    _tidy_chart(bc, n_tr)
    d.add_chart(bc, 'I20')
    line('Daily RoE excl. switches (dashed line: target)', d, xr,
         [(Reference(d, min_col=19, min_row=6, max_row=6 + n_tr), S2, False), (Reference(d, min_col=20, min_row=6, max_row=6 + n_tr), MUTED, True)],
         'I36', '0%', w=13.6, h=7.6)
    d['A37'] = 'Commentary draft'
    d['A37'].font = st['h2']
    d['A38'] = commentary
    d['A38'].alignment = st['wrap']
    d['A38'].font = st['body']
    d.merge_cells('A38:G46')
    d.freeze_panes = 'A4'
    _print_page(d, 'A1:M51', '&8Fleet daily dashboard')

    # ======================================================================= Checks
    ck = wb.create_sheet('Checks', 3)
    _title(ck, st, 'Checks', 'From the notebook run.', 4)
    _table(ck, st, 4, ['Check', 'Result', 'Detail', 'OK'],
           [[r['check'], str(r['value']), str(r['detail']), 'yes' if r['ok'] else 'CHECK'] for r in checks_df.to_dict('records')],
           [None] * 4, font='body', widths=[60, 14, 90, 8])
    for r_ in range(5, 5 + len(checks_df)):
        ck.cell(r_, 3).alignment = st['wrap']

    # tab colours: pages purple, inputs grey
    for ws in wb.worksheets:
        ws.sheet_properties.tabColor = PURPLE if ws.title in ('Dashboard', 'Products', 'Year to date', 'Checks') else 'B0B0B0'
    for ws in wb.worksheets:
        if ws.title not in ('Dashboard', 'Products', 'Year to date'):
            _print_wide(ws)
        # the three page tabs open selected together, so File > Save as / Export > PDF gives a three-page dashboard
        ws.sheet_view.tabSelected = ws.title in ('Dashboard', 'Products', 'Year to date')
    wb.active = wb.sheetnames.index('Dashboard')
    wb.save(path)
    return path

In [ ]:
# @title
%%writefile fleet_daily_png.py
"""fleet_daily_png: a one-image summary of the D01 dashboard for the Slack daily update.

Slack shows images inline (it cannot preview an HTML file), so this draws the headline figures, the last 40 days of
applications and returns, and today's top products as a PNG. Totals only, no loan rows. Post it with the dashboard link.
"""
import numpy as np
import pandas as pd

PURPLE, INK, INK2, MUTED, GRID, PAPER = '#321E37', '#24172A', '#5E5062', '#8E8590', '#E8E4E9', '#FFFFFF'
S1, S2, S3, S4 = '#5A6FE0', '#1E9E8F', '#D9731E', '#8E5AA8'
GOOD, BAD = '#1E7F4E', '#B4441F'


def _font():
    import matplotlib
    from matplotlib import font_manager
    names = {f.name for f in font_manager.fontManager.ttflist}
    for n in ('Inter', 'Arial', 'Liberation Sans', 'Helvetica', 'DejaVu Sans'):
        if n in names:
            matplotlib.rcParams['font.family'] = n
            return n


def _m(x, dp=2, sign=False):
    if x is None or not np.isfinite(x):
        return '-'
    s = format(x / 1e6, ('+' if sign else '') + ',.' + str(dp) + 'f')
    s = s.replace('-', '−')
    if sign:
        return s[0] + '£' + s[1:] + 'm'
    return '£' + s + 'm'


def _p(x, dp=1):
    return '-' if x is None or not np.isfinite(x) else ('%.' + str(dp) + 'f%%') % (x * 100)


def _pts(x, dp=1, unit=' pts'):
    return '-' if x is None or not np.isfinite(x) else (('%+.' + str(dp) + 'f') % (x * 100)).replace('-', '−') + unit


def _tone(x, good_if_positive=True):
    if x is None or not np.isfinite(x) or abs(x) < 1e-12:
        return INK2
    return GOOD if (x > 0) == good_if_positive else BAD


def build(path, p1, p2, avg, p5, hist_daily, targets, link=None, days=40, dpi=150):
    """Write the Slack summary PNG to path and return the path."""
    import matplotlib
    matplotlib.use('Agg')
    import matplotlib.pyplot as plt
    from matplotlib.ticker import FuncFormatter
    _font()
    rd = pd.Timestamp(p1['date'])
    prev = pd.Timestamp(p1['prev'])
    fig = plt.figure(figsize=(12, 7.6), dpi=dpi, facecolor=PAPER)

    # --- header band
    hb = fig.add_axes([0, 0.915, 1, 0.085])
    hb.set_axis_off()
    hb.add_patch(plt.Rectangle((0, 0), 1, 1, color=PURPLE, transform=hb.transAxes))
    hb.text(0.025, 0.56, 'Fleet daily dashboard', color='white', fontsize=17, fontweight='bold', va='center', transform=hb.transAxes)
    hb.text(0.025, 0.2, 'Applications on %s, against %s' % (rd.strftime('%A %d %B %Y'), prev.strftime('%A %d %B')),
            color='#E9E2EC', fontsize=10, va='center', transform=hb.transAxes)

    # --- headline tiles
    sw = p1['swaps']
    sw_bits = []
    for k in ('1Y', '2Y', '5Y'):
        if pd.notna(sw.get(k)):
            sw_bits.append((k, sw[k], sw[k] - sw[k + '_prev'] if pd.notna(sw.get(k + '_prev')) else np.nan))
    ytd_target = p5['apps_target'][-1] if len(p5.get('apps_target', [])) else np.nan
    tiles = [
        ('Applications excl. switches', _m(p1['apps']['value']),
         [('%s on day' % _m(p1['apps']['dod'], sign=True), _tone(p1['apps']['dod'])), ('%d cases' % p1['apps']['count'], INK2)]),
        ('RoE excl. switches', _p(p1['returns']['roe']),
         [('%s vs pricing' % _pts(p1['returns']['vs_pricing']), _tone(p1['returns']['vs_pricing'])),
          ('%s vs proposed mix' % _pts(p1['returns']['vs_proposed']), _tone(p1['returns']['vs_proposed']))]),
        ('Margin over swap, 2-year', _p(p1['fapsm']['2Y'], 2),
         [('5-year %s' % _p(p1['fapsm']['5Y'], 2), INK2), ('all, trackers incl. %s' % _p(p1['fapsm']['all'], 2), INK2)]),
        ('Completions', _m(p1['comps']['value']),
         [('%s on day' % _m(p1['comps']['dod'], sign=True), _tone(p1['comps']['dod'])), ('%d cases' % p1['comps']['count'], INK2)]),
        ('Applications year to date', _m(p5['app_ytd'], 1),
         [('target to date %s' % _m(ytd_target, 1), INK2), ('W.A. RoE %s' % _p(p5['app_wa_roe']), INK2)]),
        ('Swaps and move on the day', None, sw_bits),
    ]
    n = len(tiles)
    w, gap, x0, y0, h = 0.152, 0.008, 0.025, 0.735, 0.155
    for i, (label, value, subs) in enumerate(tiles):
        ax = fig.add_axes([x0 + i * (w + gap), y0, w, h])
        ax.set_axis_off()
        ax.add_patch(plt.Rectangle((0, 0), 1, 1, facecolor='#F8F5F0', edgecolor='none', transform=ax.transAxes))
        ax.text(0.07, 0.83, label, fontsize=8.2, color=INK2, transform=ax.transAxes, va='center')
        if value is None:                              # the swaps tile: one row per tenor, with the move on the day
            if not subs:
                ax.text(0.07, 0.56, '-', fontsize=15, color=INK, fontweight='bold', transform=ax.transAxes, va='center')
            for j, (k, v, d) in enumerate(subs):
                y_ = 0.6 - j * 0.2 if len(subs) == 3 else 0.56 - j * 0.26
                ax.text(0.07, y_, k, fontsize=9, color=INK2, transform=ax.transAxes, va='center')
                ax.text(0.24, y_, _p(v, 2), fontsize=12.5, color=INK, fontweight='bold', transform=ax.transAxes, va='center')
                mv = ('%+.0f bp' % (d * 1e4)).replace('-', '−') if np.isfinite(d) else 'new'
                ax.text(0.93, y_, mv, fontsize=8.5, color=INK2, transform=ax.transAxes, va='center', ha='right')
            continue
        big = 15 if len(value) < 12 else 11.5
        ax.text(0.07, 0.56, value, fontsize=big, color=INK, fontweight='bold', transform=ax.transAxes, va='center')
        for j, (t, c) in enumerate(subs):
            ax.text(0.07, 0.29 - j * 0.17, t, fontsize=8, color=c, transform=ax.transAxes, va='center')

    # --- trend charts
    dh = hist_daily.copy()
    dh['Date'] = pd.to_datetime(dh['Date'])
    dh = dh[dh['Date'] <= rd].sort_values('Date').tail(days)
    lab = [d.strftime('%d %b') for d in dh['Date']]
    xs = np.arange(len(dh))
    step = max(1, len(xs) // 6)

    def style(ax, title):
        ax.set_title(title, loc='left', fontsize=11, color=INK, fontweight='bold', pad=20)
        for s in ('top', 'right', 'left'):
            ax.spines[s].set_visible(False)
        ax.spines['bottom'].set_color(GRID)
        ax.tick_params(colors=INK2, labelsize=8, length=0)
        ax.grid(axis='y', color=GRID, linewidth=0.8)
        ax.set_axisbelow(True)
        ax.set_xticks(xs[::-1][::step][::-1])
        ax.set_xticklabels([lab[i] for i in xs[::-1][::step][::-1]])
        ax.set_xlim(-0.8, len(xs) - 0.2)

    a1 = fig.add_axes([0.045, 0.40, 0.43, 0.24])
    vol = dh['Volume'].astype(float).to_numpy()
    cols = [S1] * len(vol)
    a1.bar(xs, vol, width=0.72, color=cols, linewidth=0)
    tgt = targets.get('daily_volume')
    if tgt:
        a1.axhline(tgt, color=MUTED, linestyle=(0, (4, 3)), linewidth=1.3)
    a1.yaxis.set_major_formatter(FuncFormatter(lambda v, _: '£%.0fm' % (v / 1e6)))
    style(a1, 'Daily applications excl. switches, last %d reported days' % len(xs))
    if tgt:
        a1.text(0, 1.0, 'dashed line: daily target %s' % _m(tgt, 0), fontsize=8, color=INK2, transform=a1.transAxes, va='bottom')
    if len(vol):
        a1.annotate(_m(vol[-1]), (xs[-1], vol[-1]), xytext=(0, 4), textcoords='offset points', ha='right', fontsize=8, color=INK,
                    bbox=dict(boxstyle='round,pad=0.2', facecolor='white', edgecolor='none', alpha=0.9))

    a2 = fig.add_axes([0.545, 0.40, 0.43, 0.24])
    roe = dh['RoE'].astype(float).to_numpy()
    a2.plot(xs, roe, color=S2, linewidth=2)
    rt = targets.get('roe')
    if rt:
        a2.axhline(rt, color=MUTED, linestyle=(0, (4, 3)), linewidth=1.3)
    if len(roe) and np.isfinite(roe[-1]):
        a2.plot([xs[-1]], [roe[-1]], 'o', color=S2, markersize=5, markeredgecolor='white', markeredgewidth=1.5)
        a2.annotate(_p(roe[-1]), (xs[-1], roe[-1]), xytext=(-6, 9), textcoords='offset points', ha='right', fontsize=8, color=INK,
                    bbox=dict(boxstyle='round,pad=0.2', facecolor='white', edgecolor='none', alpha=0.9))
    lo = np.nanmin(np.append(roe, rt or np.nan)) if len(roe) else 0
    hi = np.nanmax(np.append(roe, rt or np.nan)) if len(roe) else 0.3
    a2.set_ylim(max(0, lo - 0.03), hi + 0.03)
    from matplotlib.ticker import MultipleLocator
    a2.yaxis.set_major_locator(MultipleLocator(0.05 if hi - lo > 0.12 else 0.02))
    a2.yaxis.set_major_formatter(FuncFormatter(lambda v, _: '%.0f%%' % (v * 100)))
    style(a2, 'Daily return on equity excl. switches')
    if rt:
        a2.text(0, 1.0, 'dashed line: target %s' % _p(rt), fontsize=8, color=INK2, transform=a2.transAxes, va='bottom')

    # --- top products today
    t = p2[p2['volume'] > 0].sort_values('volume', ascending=False).head(6)
    tb = fig.add_axes([0.025, 0.06, 0.95, 0.27])
    tb.set_axis_off()
    tb.text(0, 0.97, "Today's largest products by application volume", fontsize=11, color=INK, fontweight='bold', va='top', transform=tb.transAxes)
    heads = [('Pricing code', 0.0, 'left'), ('Volume today', 0.30, 'right'), ('Return today', 0.43, 'right'), ('Pricing RoE', 0.56, 'right'),
             ('Return vs pricing', 0.71, 'right'), ('Volume last 30 days', 0.86, 'right'), ('Return last 30 days', 1.0, 'right')]
    ry = 0.78
    for h_, x, ha in heads:
        tb.text(x, ry, h_, fontsize=8, color=INK2, ha=ha, va='center', transform=tb.transAxes)
    tb.plot([0, 1], [ry - 0.07, ry - 0.07], color=GRID, linewidth=1, transform=tb.transAxes)
    rh = 0.112
    for k, r in enumerate(t.itertuples(index=False)):
        y = ry - 0.15 - k * rh
        diff = r.roe - r.pricing_roe if np.isfinite(r.roe) and np.isfinite(r.pricing_roe) else np.nan
        vals = [(r.code + ('  (switch)' if r.loyalty else ''), INK), (_m(r.volume), INK), (_p(r.roe), INK), (_p(r.pricing_roe), INK2),
                (_pts(diff), _tone(diff)), (_m(r.mth_volume, 1), INK2), (_p(r.mth_roe), INK2)]
        for (h_, x, ha), (v, c) in zip(heads, vals):
            tb.text(x, y, v, fontsize=9, color=c, ha=ha, va='center', transform=tb.transAxes, fontweight='bold' if h_ == 'Pricing code' else 'normal')
    if not len(t):
        tb.text(0, ry - 0.15, 'No applications with a return today.', fontsize=9, color=INK2, transform=tb.transAxes)

    foot = 'Totals only. Returns are application-weighted from the loan level pricing model run for %s.' % rd.strftime('%d %b %Y')
    fig.text(0.025, 0.022, foot, fontsize=7.5, color=MUTED)
    if link:
        fig.text(0.975, 0.022, 'Full dashboard: ' + link, fontsize=7.5, color=INK2, ha='right')
    fig.savefig(path, dpi=dpi, facecolor=PAPER)
    plt.close(fig)
    return path

In [ ]:
# @title
import fleet_sql as fs
import fleet_daily as D
import fleet_daily_html as H
import fleet_daily_xlsx as X
import fleet_daily_png as G
for m_ in (fs, D, H, X, G):
    importlib.reload(m_)
if BILLING_PROJECT: fs.use_project(BILLING_PROJECT)
fs.INPUT_DIR, fs.UPLOAD = INPUT_DIR, UPLOAD
D.MATCH_EXCEL = MATCH_EXCEL
D.TARGETS.update(TARGETS)
def offer_download(paths, title):
    # download buttons (default), Colab's own download, or nothing; the files are also in the file browser on the left
    if DOWNLOAD == 'buttons':
        display(HTML(D.download_links(paths, title)))
    elif DOWNLOAD == 'colab':
        try:
            from google.colab import files
            for p_ in paths:
                files.download(p_)
        except Exception as e:
            print('Download from the file browser on the left (%s).' % str(e)[:60])
    print('Also in the file browser on the left (folder icon), right-click -> Download:', ', '.join(paths))

SHARE = str(SHARE_FOLDER).strip().rstrip('/')
KEEP = ['Fleet_Current_Pricing.xlsx', 'Fleet_Product_Codes.csv', 'Fleet_Daily_Dashboard_History.xlsx']   # kept in SHARE_FOLDER between runs
_local = lambda n: os.path.join(INPUT_DIR or '.', n)

def share_get(names):
    # copy the kept files from the shared folder into INPUT_DIR (unless a copy is already here); returns the names found
    got = []
    for n in names:
        try:
            if os.path.exists(_local(n)) or D.gcs_get(SHARE, n, _local(n), BILLING_PROJECT or None):
                got.append(n)
        except Exception as e:
            print('Shared folder not readable (%s): %s' % (n, str(e)[:150]))
            break
    return got

def share_put(path, name=None):
    try:
        return D.gcs_put(path, SHARE, name, BILLING_PROJECT or None)
    except Exception as e:
        print('Could not save %s to %s: %s' % (os.path.basename(path), SHARE, str(e)[:200]))
        return None

def product_codes():
    p_ = fs.find_files(('Fleet_Product_Codes*.csv',))
    if not p_:
        print('No Fleet_Product_Codes.csv: pricing codes come from the product descriptions.')
        return None
    m_ = D.read_product_codes(p_[-1])
    print('Product codes: %s (%d products)' % (os.path.basename(p_[-1]), len(m_)))
    return m_

if SHARE:
    print('Shared folder %s: %s' % (SHARE, ', '.join(share_get(KEEP)) or 'none of the kept files there yet'))

_today = pd.Timestamp(datetime.date.today())

def choose_day():
    # the application date to report, from RUN_FOR
    mode = str(RUN_FOR).strip().lower()
    if mode.startswith('date'):
        if not str(REPORTING_DATE).strip():
            raise ValueError("RUN_FOR = 'Date below' needs REPORTING_DATE, e.g. '2026-10-28'.")
        return pd.Timestamp(REPORTING_DATE).normalize()
    if mode.startswith('previous'):
        return D.prev_bday(_today)
    if mode.startswith('2'):
        return D.prev_bday(D.prev_bday(_today))
    av = fs.run(D.availability_sql(fs), {'start': str((_today - pd.Timedelta(days=21)).date()), 'end': str(_today.date())}, label='data available')
    ch = D.day_choices(av, _today)
    show_ = ch.drop(columns=['Date']).copy()
    show_['Ready'] = np.where(ch['Ready'], 'yes', 'not yet')
    display(show_.set_index('#'))
    ready = ch[ch['Ready']]
    dflt = int(ready['#'].iloc[0]) if len(ready) else 1
    while True:
        ans = input('Run the dashboard for which day? Type 1-%d or a date YYYY-MM-DD (Enter = %d, %s): '
                    % (len(ch), dflt, ch.loc[ch['#'] == dflt, 'Day'].iloc[0])).strip()
        if not ans:
            return ch.loc[ch['#'] == dflt, 'Date'].iloc[0]
        if ans.isdigit() and 1 <= int(ans) <= len(ch):
            return ch.loc[ch['#'] == int(ans), 'Date'].iloc[0]
        try:
            return pd.Timestamp(ans).normalize()
        except ValueError:
            print('Please type a number from the list or a date like 2026-10-28.')

print('shared SQL', fs.VERSION, '| dashboard', D.VERSION, '| MATCH_EXCEL', MATCH_EXCEL)

## Day and swaps

With `RUN_FOR = 'Ask'` this lists the last five working days: applications, £, and how many have no balance yet ("Ready" = balances filled in). Type the number of the day to run, or a date. Yesterday's balances usually arrive around midday.

In [ ]:
# @title
RD = choose_day()
PREV = D.prev_bday(RD)
START = min(D.fy_start(RD), RD - pd.Timedelta(days=120))
PARAMS = {'start': str(START.date()), 'end': str(_today.date())}
BBR_ = float(BBR) / 100

def _swap(v, label, needed):
    v = str(v).strip()
    while needed and not v:
        v = input('%s swap rate on %s, in %% (e.g. 4.74): ' % (label, RD.date())).strip()
    try:
        return float(v) if v else None
    except ValueError:
        return _swap(input('Please type the %s swap as a number: ' % label), label, needed)

SWAPS_TODAY = {12: _swap(SWAP_1Y, '1-year', False), 24: _swap(SWAP_2Y, '2-year', True), 60: _swap(SWAP_5Y, '5-year', True)}

def pull_tapes():
    # DIPs & apps and completions from BigQuery (loan-level, kept in memory only)
    raw_a = fs.run(D.apps_sql(fs), PARAMS, label='DIPs and apps')
    raw_c = fs.run(D.comps_sql(fs), PARAMS, label='completions')
    return raw_a, raw_c

print('Reporting date %s (%s) | previous working day %s | tapes from %s to %s' % (RD.date(), RD.strftime('%A'), PREV.date(), START.date(), _today.date()))
print('Swaps today: ' + ', '.join('%s %s' % (l_, ('%.3f%%' % v_) if v_ is not None else 'not given') for l_, v_ in zip(('1-year', '2-year', '5-year'), SWAPS_TODAY.values()))
      + ' | Bank Rate %.2f%%' % BBR)

---
# Part 1 · Inputs for the pricing model

## 1.1 Current pricing and the tapes

In [ ]:
# @title
CP_PATH = fs.need_file('current_pricing')
CP = D.read_current_pricing(CP_PATH)
CODE_MAP = product_codes()
print('Current pricing: %s | %d product codes, %d loyalty codes | proposed-mix RoE %.2f%%'
      % (CP['source'], len(CP['products']), len(CP['loyalty']), (CP['proposed_roe'] or np.nan) * 100))
zero = CP['products'].loc[CP['products']['rate'] == 0, 'code'].tolist()
if zero:
    print('NOTE: new rate is 0 in Current Pricing for', ', '.join(zero), '- the model will find no RoE for these.')

RAW_APPS, RAW_COMPS = pull_tapes()
APPS = D.enrich_apps(RAW_APPS, pd.DataFrame({'2Y': [SWAPS_TODAY[24]], '5Y': [SWAPS_TODAY[60]]}, index=[RD]), BBR_, CODE_MAP)
t_ = APPS[APPS['ApplicationDate'] == RD]
if CODE_MAP:
    new_ = t_[(t_['CodeSource'] == 'rule') & (t_['ProductCode'] != 'ND')].groupby(['ProductCode', 'ProductDescription'])['PricingCode'].first()
    if len(new_):
        print('Product codes not in Fleet_Product_Codes.csv (coded from the description; add them to the file if this is wrong):')
        for (pc_, d_), c_ in new_.items():
            print('   %-10s %-45s -> %s' % (pc_, d_[:45], c_))
print('Applications on %s: %d cases, £%.2fm (%d product switches) | DIPs on that day: %d'
      % (RD.date(), len(t_), t_['OriginalBalance'].sum() / 1e6, (t_['Product Switch'] == 'Y').sum(), int((APPS['DIPDate'] == RD).sum())))
nb_ = int((t_['OriginalBalance'].fillna(0) == 0).sum())
if nb_:
    print('WARNING: %d of the %d applications on %s have no balance yet (data not complete for that day); '
          'consider running for an earlier day.' % (nb_, len(t_), RD.date()))
if APPS['ApplicationDate'].max() < RD:
    print('WARNING: the latest application in the warehouse is %s, before the reporting date.' % APPS['ApplicationDate'].max().date())
u_ = t_[t_['PricingCode'] == 'UNMAPPED']
off_ = t_[~t_['PricingCode'].isin(D.code_list(CP) + ['ND', 'UNMAPPED'])]
print('Pricing codes today: %d unreadable descriptions (£%.2fm); %d on codes not in Current Pricing (%s)'
      % (len(u_), u_['OriginalBalance'].sum() / 1e6, len(off_), ', '.join(sorted(off_['PricingCode'].unique())) or 'none'))

## 1.2 The input file

One row per pricing code, built as the Prodinput and ProdInput - PricingSwaps tabs did:
- **App** rows: today's average balance for the code, today's shares of flats, portfolio landlords and freehold, the new rate and fee from Current Pricing.
- **-10** rows: the same with the rate 10bp lower.
- **PricingSwaps** rows: the code's default loan from Current Pricing (AS:AY), so its RoE is the pricing RoE at today's swaps.

Columns A:AT are the LLPM input layout; the last two (`Pricing_Code`, `Run`) are the workbook's helper columns AU:AV.

In [ ]:
# @title
GRID = D.build_grid(CP, APPS, RD, BBR_)
INPUT_CSV = 'LLPM_Inputs_%s.csv' % RD.strftime('%Y-%m-%d')
D.write_llpm_input_csv(GRID, INPUT_CSV)
show = GRID[GRID['Run'] == 'App'].loc[~GRID['Identifier'].str.endswith('-10'),
            ['Identifier', 'Original_Balance', 'Current_Rate', 'Capitalised_Fees', 'Initial_Index', 'ERC_Year1', 'ERC_Year2', 'Property_Type', 'Perc_Portfolio', 'Tenure', 'Cashback']]
print('%s: %d rows (%d codes x App, -10, PricingSwaps). App rows, averages per code:' % (INPUT_CSV, len(GRID), len(GRID) // 3))
display(show.set_index('Identifier').round(4))
print('\nIn the LLPM:')
print('  1. Paste the CSV rows (not the header) into the input data, over the old rows, and clear any rows left below them.')
print('  2. 3. Rates manual swaps (Overwrite swaps? = Yes): ' +
      ', '.join('%s %.2f%%' % (c_, v_) for c_, v_ in zip(('N35 12m', 'N36 24m', 'N39 60m'), SWAPS_TODAY.values()) if v_ is not None)
      + ('' if SWAPS_TODAY[12] is not None else '  (1-year not given: leave N35 as it is)'))
print('  3. Run the model, save it, then run Part 2 below.')
offer_download([INPUT_CSV], 'Today\'s LLPM input file')
if SHARE:
    print('Saved to the shared folder:', share_put(INPUT_CSV))

---
# Part 2 · Dashboard from the model's results

Run this once the LLPM has been run on today's input file. Upload the saved LLPM workbook, or its **Outputs** tab saved as CSV or xlsx. The notebook checks the results are for today's inputs (same codes and order, today's date, same balances) and says so if not.

## 2.1 The model's results and the history

In [ ]:
# @title
if 'GRID' not in globals() or 'RAW_APPS' not in globals():
    print('Part 1 is not in memory (new session?): pulling the tapes and rebuilding the input grid to check the results against.')
    CP_PATH = fs.need_file('current_pricing')
    CP = D.read_current_pricing(CP_PATH)
    CODE_MAP = product_codes()
    RAW_APPS, RAW_COMPS = pull_tapes()
    GRID = D.build_grid(CP, D.enrich_apps(RAW_APPS, pd.DataFrame({'2Y': [SWAPS_TODAY[24]], '5Y': [SWAPS_TODAY[60]]}, index=[RD]), BBR_, CODE_MAP), RD, BBR_)

OUT_PATH = fs.need_file('llpm_outputs', upload='ask' if UPLOAD == 'ask' else None)
OUT, OUT_WARN = D.read_llpm_outputs(OUT_PATH, expected=GRID, rd=RD)
print('Model results: %s, %d rows' % (os.path.basename(OUT_PATH), len(OUT)))
for w_ in OUT_WARN:
    print('WARNING:', w_)
if not OUT_WARN:
    print('Checked: same %d codes in the same order as today\'s input file, dated %s, same balances.' % (len(GRID), RD.date()))
piv = OUT.pivot_table(index='Code', columns='Run', values='RoE', aggfunc='first').reindex(D.code_list(CP))
display((piv * 100).round(2).rename(columns=lambda c: c + ' RoE %'))

# the history: from the shared folder if it is there, otherwise upload it (or a Daily Dashboard workbook to start one)
try:
    hp_ = fs.need_file('dd_history', upload='auto' if (SHARE and os.path.exists(_local('Fleet_Daily_Dashboard_History.xlsx'))) else None)
    if hp_.lower().endswith('.xlsm'):
        print('Starting the history from the Daily Dashboard workbook', os.path.basename(hp_))
        HIST = D.seed_history_from_workbook(hp_)
    else:
        HIST = D.read_history(hp_)
except FileNotFoundError:
    seed = fs.find_files(('Daily_Dashboard*.xlsm',))
    HIST = D.seed_history_from_workbook(seed[-1]) if seed else D.empty_history()
    print('No history file:', ('started one from ' + os.path.basename(seed[-1])) if seed else
          'starting an empty one, so trends and year-to-date returns are missing. Upload Fleet_Daily_Dashboard_History.xlsx or a Daily Dashboard workbook.')
HIST = D.add_swaps(HIST, RD, SWAPS_TODAY)
SWAPS = D.swaps_frame(HIST)
print('History: %d days of headline returns, %d days of code returns, swaps to %s'
      % (HIST['daily']['Date'].nunique(), HIST['codes']['Date'].nunique(), SWAPS.index.max().date()))
APPS = D.enrich_apps(RAW_APPS, SWAPS, BBR_, CODE_MAP)
COMPS = D.enrich_comps(RAW_COMPS, CODE_MAP)

## 2.2 Dashboard figures

In [ ]:
# @title
P2, AVG = D.page2(CP, APPS, OUT, RD, HIST['codes'])
P1 = D.page1(CP, APPS, COMPS, RD, AVG, SWAPS)
HIST = D.update_history(HIST, RD, P1, P2, AVG, OUT, APPS, COMPS)
P5 = D.page5(APPS, COMPS, RD, HIST['daily'])
CHK = D.checks(APPS, COMPS, RD, CP, OUT, SWAPS, OUT_WARN)

ex = AVG['excl_ps']
sw_ = P1['swaps']
head = pd.DataFrame([
    ['Applications excl. PS', '£%.2fm' % (P1['apps']['value'] / 1e6), '£%+.2fm on day, %d cases' % (P1['apps']['dod'] / 1e6, P1['apps']['count'])],
    ['Applications incl. PS', '£%.2fm' % (P1['apps_ps']['value'] / 1e6), '£%+.2fm on day, %d cases' % (P1['apps_ps']['dod'] / 1e6, P1['apps_ps']['count'])],
    ['Completions', '£%.2fm' % (P1['comps']['value'] / 1e6), '£%+.2fm on day, %d cases' % (P1['comps']['dod'] / 1e6, P1['comps']['count'])],
    ['Return excl. PS', '%.2f%%' % (P1['returns']['roe'] * 100), 'vs pricing %+.2f pts, vs proposed %+.2f pts' % (P1['returns']['vs_pricing'] * 100, P1['returns']['vs_proposed'] * 100)],
    ['Return with 10bp off', '%.2f%%' % (ex['roe_10'] * 100), 'change %+.2f pts' % (P1['returns_10']['delta'] * 100)],
    ['FAPSM 2Y / 5Y / all', '%.2f%% / %.2f%% / %.2f%%' % (P1['fapsm']['2Y'] * 100, P1['fapsm']['5Y'] * 100, P1['fapsm']['all'] * 100),
     'on day %+.2f / %+.2f / %+.2f pts' % (P1['fapsm']['2Y_dod'] * 100, P1['fapsm']['5Y_dod'] * 100, P1['fapsm']['all_dod'] * 100)],
    ['Swaps 1Y / 2Y / 5Y', ' / '.join('%.2f%%' % (sw_[k_] * 100) if pd.notna(sw_[k_]) else '-' for k_ in ('1Y', '2Y', '5Y')),
     'on day ' + ' / '.join('%+.0f bp' % ((sw_[k_] - sw_[k_ + '_prev']) * 1e4) if pd.notna(sw_[k_]) and pd.notna(sw_[k_ + '_prev']) else '-' for k_ in ('1Y', '2Y', '5Y'))],
    ['Applications YTD', '£%.1fm' % (P5['app_ytd'] / 1e6), 'W.A. RoE %.2f%%; £%.1fm still needed at %.2f%%' % (P5['app_wa_roe'] * 100, P5['todo'] / 1e6, P5['need_roe'] * 100)],
    ['Completions YTD', '£%.1fm' % (P5['comps_ytd_total'] / 1e6), 'target to date £%.1fm' % (P5['comps_target'][-1] / 1e6)],
], columns=['', 'Value', 'Detail'])
display(head.set_index(''))
t2 = P2[['code', 'pricing_roe', 'pricing_roe_today_swaps', 'volume', 'roe', 'roe_10', 'contribution', 'mth_volume', 'mth_roe']].copy()
for c_ in ('pricing_roe', 'pricing_roe_today_swaps', 'roe', 'roe_10', 'mth_roe'):
    t2[c_] = (t2[c_] * 100).round(2)
display(t2.round(2).set_index('code'))
display(CHK)
print(D.commentary(P1, P2))

## 2.3 Save the dashboard and the history

Writes four files:

- `Fleet_Daily_Dashboard_<date>.html`: the dashboard page (opens in any browser, no internet needed).
- `Fleet_Daily_Dashboard_<date>.png`: a one-image summary for the Slack daily update. Slack shows images inline but can't preview HTML, so post this with the dashboard link.
- `Fleet_Daily_Dashboard_<date>.xlsx`: the dashboard as an Excel workbook. Input tabs (settings, model output, current pricing, applications and completions by day and code, margins, history) and Dashboard / Products / Year to date / Checks tabs built from them with formulas and charts. Totals only, no loan rows. Excel recalculates on opening, so a Slack or Teams preview may show blanks until it is opened in Excel. Each page tab is set to print on one landscape A4 page, and the three open selected together: File > Save As (or Export) > PDF gives a three-page dashboard PDF. If you click onto another tab first, click Dashboard then Shift-click Year to date to select the three again.
- `Fleet_Daily_Dashboard_History.xlsx`: the updated history (picked up from the shared folder tomorrow, or upload it).

In [ ]:
# @title
meta = {'generated': datetime.datetime.now().strftime('%d %b %Y %H:%M'), 'engine': 'pricing model results from ' + os.path.basename(OUT_PATH),
        'match_excel': MATCH_EXCEL, 'excel_notes': D.EXCEL_NOTES,
        'inputs': {'Reporting date': RD.strftime('%d %b %Y'), 'Previous working day': PREV.strftime('%d %b %Y'),
                   'Tapes': 'BigQuery, applications and completions %s to %s' % (START.date(), _today.date()),
                   'Pricing model input': INPUT_CSV if 'INPUT_CSV' in globals() else 'rebuilt in Part 2',
                   'Pricing model results': os.path.basename(OUT_PATH) + (' (with warnings)' if OUT_WARN else ' (checked against the input file)'),
                   'Current pricing': CP['source'],
                   'Swaps today (%)': ', '.join('%dm %.3f' % (t_, v_) for t_, v_ in SWAPS_TODAY.items() if v_ is not None),
                   'Bank Rate': '%.2f%%' % BBR,
                   'History': '%d days' % HIST['daily']['Date'].nunique(),
                   'Notebook': 'D01 | fleet_daily %s | fleet_sql %s' % (D.VERSION, fs.VERSION)}}
DATA = D.dashboard_data(CP, APPS, COMPS, RD, OUT, HIST, SWAPS, P5, CHK, meta, back_days=BACK_DAYS)
tag = RD.strftime('%Y-%m-%d')
html_path = 'Fleet_Daily_Dashboard_%s.html' % tag
with open(html_path, 'w', encoding='utf-8') as f_:
    f_.write(H.render_html(DATA))
hist_path = D.save_history(HIST, 'Fleet_Daily_Dashboard_History.xlsx')
xlsx_path = 'Fleet_Daily_Dashboard_%s.xlsx' % tag
X.build(xlsx_path, RD, PREV, CP, APPS, COMPS, OUT, HIST, SWAPS, CHK, meta, D.TARGETS, D.commentary(P1, P2))
png_path = 'Fleet_Daily_Dashboard_%s.png' % tag
link_ = dated_ = None
if SHARE:
    link_ = share_put(html_path, 'Fleet_Daily_Dashboard_latest.html')
    dated_ = share_put(html_path)
G.build(png_path, P1, P2, AVG, P5, HIST['daily'], D.TARGETS, link=link_)
print('Saved', html_path, '|', png_path, '|', xlsx_path, '|', hist_path)
if SHARE:
    png_link_ = share_put(png_path)
    xlsx_link_ = share_put(xlsx_path)
    share_put(hist_path)
    if CP_PATH.lower().endswith('.xlsm'):            # pricing read from a Daily Dashboard workbook: keep just its Current Pricing tab
        CP_PATH = D.write_current_pricing_template(CP_PATH, _local('Fleet_Current_Pricing.xlsx'))
    share_put(CP_PATH, 'Fleet_Current_Pricing.xlsx')
    pc_ = fs.find_files(('Fleet_Product_Codes*.csv',))
    if pc_:
        share_put(pc_[-1], 'Fleet_Product_Codes.csv')
    if link_:
        display(HTML('<div style="font:14px sans-serif;margin:8px 0"><b>Dashboard link (share this):</b> <a href="%s" target="_blank">%s</a><br>'
                     'This date only: <a href="%s" target="_blank">%s</a><br>Slack image: <a href="%s" target="_blank">%s</a><br>'
                     'Excel version: <a href="%s" target="_blank">%s</a><br>History, pricing and product codes saved in %s for the next run.</div>'
                     % (link_, link_, dated_, dated_, png_link_, png_link_, xlsx_link_, xlsx_link_, SHARE)))
else:
    print('No SHARE_FOLDER: to share the dashboard and keep the history between runs, set SHARE_FOLDER to a Cloud Storage folder (see the top of the notebook).')
print('For Slack: attach the .png (it shows inline) and paste the dashboard link underneath.')
print('For a PDF: open the .xlsx in Excel, then File > Save As / Export > PDF. The three page tabs are already selected and each fits one landscape page.')
offer_download([png_path, html_path, xlsx_path, hist_path], 'Slack image, dashboard, Excel version and history')

In [ ]:
# @title
import base64, warnings
warnings.filterwarnings('ignore', message='Consider using IPython.display.IFrame')
from IPython.display import Image
display(Image(png_path, width=900))
src = open(html_path, encoding='utf-8').read()
display(HTML('<iframe src="data:text/html;base64,%s" style="width:100%%;height:900px;border:0"></iframe>' % base64.b64encode(src.encode('utf-8')).decode()))